In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

In [ ]:
import time
NOTEBOOK_STARTED = time.perf_counter()
from pathlib import Path
from types import SimpleNamespace as _ITDA_Namespace
_ITDA_BUNDLE_ROOT = Path.cwd().resolve()
_ITDA_NS = {}


In [ ]:
from __future__ import annotations
# Embedded from src/crop_batching.py; generated, do not edit separately.
def _itda_define_crop_batching():
    """Keep pixels/aspect ratio intact; batch similar normalized widths together."""
    import math
    
    def width_batches(crops, batch_size=8, height=48, minimum_width=320, bucket_step=160):
        if batch_size < 1 or height < 1 or bucket_step < 1:
            raise ValueError('Positive crop batch dimensions required')
        buckets = {}
        for (index, crop) in enumerate(crops):
            if crop.ndim != 3 or crop.shape[2] != 3 or min(crop.shape[:2]) < 1:
                raise ValueError('Nonempty three-channel crop required')
            width = max(minimum_width, math.ceil(height * crop.shape[1] / crop.shape[0]))
            bucket = math.ceil(min(width, 3200) / bucket_step)
            buckets.setdefault(bucket, []).append((width, index))
        for bucket in sorted(buckets):
            ordered = [i for (_, i) in sorted(buckets[bucket])]
            for start in range(0, len(ordered), batch_size):
                yield ordered[start:start + batch_size]
    return _ITDA_Namespace(**locals())
_ITDA_NS['crop_batching'] = _itda_define_crop_batching()
del _itda_define_crop_batching


In [ ]:
from __future__ import annotations
# Embedded from src/ctc_candidates.py; generated, do not edit separately.
def _itda_define_ctc_candidates():
    """Bounded CTC prefix search for diagnostics, never automatic digit correction."""
    import math
    import numpy as np
    
    def candidates(probabilities, characters, blank=0, beam_width=4, token_top_k=3, max_steps=160):
        p = np.asarray(probabilities)
        if p.ndim != 2 or not 0 < len(p) <= max_steps or p.shape[1] != len(characters) or (not np.isfinite(p).all()) or (p < 0).any() or (p > 1).any() or (not np.allclose(p.sum(axis=1), 1.0, atol=0.02)):
            return []
        if not 1 <= beam_width <= 8 or not 1 <= token_top_k <= 5:
            raise ValueError('Bounded CTC search configuration required')
        neg = -math.inf
        beams = {(): (0.0, neg)}
    
        def add(table, prefix, slot, score):
            pair = list(table.get(prefix, (neg, neg)))
            pair[slot] = float(np.logaddexp(pair[slot], score))
            table[prefix] = tuple(pair)
        for row in p:
            top = set(np.argsort(row)[-token_top_k:].tolist()) | {blank}
            next_beams = {}
            for (prefix, (pb, pn)) in beams.items():
                total = float(np.logaddexp(pb, pn))
                for token in top | ({prefix[-1]} if prefix else set()):
                    score = math.log(max(float(row[token]), 1e-30))
                    if token == blank:
                        add(next_beams, prefix, 0, total + score)
                    elif prefix and token == prefix[-1]:
                        add(next_beams, prefix, 1, pn + score)
                        add(next_beams, prefix + (token,), 1, pb + score)
                    else:
                        add(next_beams, prefix + (token,), 1, total + score)
            beams = dict(sorted(next_beams.items(), key=lambda pair: float(np.logaddexp(*pair[1])), reverse=True)[:beam_width])
        return [dict(text=''.join((characters[t] for t in prefix)), log_probability=float(np.logaddexp(*scores)), calibrated=False) for (prefix, scores) in sorted(beams.items(), key=lambda pair: float(np.logaddexp(*pair[1])), reverse=True)]
    return _ITDA_Namespace(**locals())
_ITDA_NS['ctc_candidates'] = _itda_define_ctc_candidates()
del _itda_define_ctc_candidates


In [ ]:
from __future__ import annotations
# Embedded from src/date_fields.py; generated, do not edit separately.
def _itda_define_date_fields():
    """date-fields-v1: independent Y/M/D values, fixed three-field denominator."""
    import re
    from datetime import date
    FIELDS = ('year', 'month', 'day')
    POLICY = 'date-fields-v1'
    
    def field_values(row):
        values = {}
        for (name, width, maximum) in (('year', 4, 2099), ('month', 2, 12), ('day', 2, 31)):
            value = row.get(name)
            if value != 'NONE' and (not isinstance(value, str) or not re.fullmatch('[0-9]{%d}' % width, value) or (not 1 <= int(value) <= maximum)):
                raise ValueError('Invalid ' + name + ' field: ' + repr(value))
            values[name] = value
        return values
    
    def serialize_fields(row):
        values = field_values(row)
        (y, m, d) = (values[k] for k in FIELDS)
        if m != 'NONE' and d != 'NONE':
            date(int(y) if y != 'NONE' else 2000, int(m), int(d))
        result = '-'.join((values[k] for k in FIELDS))
        return dict(values, final_date='NONE' if result == 'NONE-NONE-NONE' else result)
    
    def fields_from_date(value):
        if value is None or value in ('NONE', 'NONE-NONE-NONE'):
            return serialize_fields(dict.fromkeys(FIELDS, 'NONE'))
        if not isinstance(value, str) or len(value.split('-')) != 3:
            raise ValueError('Invalid date: ' + repr(value))
        return serialize_fields(dict(zip(FIELDS, value.split('-'))))
    
    def compliant_row(row):
        try:
            expected = serialize_fields(row)
        except ValueError:
            return False
        partial = any((expected[k] == 'NONE' for k in FIELDS))
        return list(row) == ['image_id', *FIELDS, 'final_date'] and (row.get('final_date') == expected['final_date'] or (partial and row.get('final_date') == 'NONE'))
    return _ITDA_Namespace(**locals())
_ITDA_NS['date_fields'] = _itda_define_date_fields()
del _itda_define_date_fields


In [ ]:
from __future__ import annotations
# Embedded from src/date_region_recovery.py; generated, do not edit separately.
def _itda_define_date_region_recovery():
    """Bounded pixel-based proposals for missed dot-matrix rows; no labels/IDs."""
    import math
    import re
    import time
    from dataclasses import replace
    import cv2
    import numpy as np
    
    def rectified_crop(image, polygon, padding=0.0):
        points = np.asarray(polygon, dtype=np.float32)
        if points.shape != (4, 2) or not np.isfinite(points).all():
            return None
        if not cv2.isContourConvex(points) or abs(cv2.contourArea(points)) < 64:
            return None
        width = max(np.linalg.norm(points[1] - points[0]), np.linalg.norm(points[2] - points[3]))
        height = max(np.linalg.norm(points[3] - points[0]), np.linalg.norm(points[2] - points[1]))
        if width < 2 * height or height < 8:
            return None
        if padding:
            center = points.mean(axis=0)
            points = center + (points - center) * np.array([1 + padding * height / width, 1 + padding], dtype=np.float32)
            width += padding * height
            height *= 1 + padding
        (width, height) = (int(math.ceil(width)), int(math.ceil(height)))
        if width > image.shape[1] * 2 or height > image.shape[0] * 2:
            return None
        target = np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])
        return cv2.warpPerspective(image, cv2.getPerspectiveTransform(points, target), (width, height), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    
    def recognition_views(image, polygon):
        """Two genuinely different views; normalize wide rows, never duplicate one.
    
        Fixed-width recognition squeezes long date/time rows. Bound their aspect
        ratio without rewriting pixels into digits. Short rows keep distinct margins.
        """
        padded = rectified_crop(image, polygon, 0.16)
        if padded is None:
            return []
        (height, width) = padded.shape[:2]
        if width / height > 5.2:
            return [(f'aspect-{ratio}', cv2.resize(padded, (round(height * ratio), height), interpolation=cv2.INTER_AREA)) for ratio in (4.8, 5.2)]
        plain = rectified_crop(image, polygon)
        return [('padding-0.0', plain), ('padding-0.16', padded)] if plain is not None else []
    
    def split_row_polygon(image, polygon):
        """Separate two printed rows using an ink-free horizontal gap after deskew."""
        crop = rectified_crop(image, polygon)
        if crop is None:
            return []
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        ink = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 15)
        (height, width) = gray.shape
        profile = np.count_nonzero(ink, axis=1) / width
        smooth = max(3, round(height * 0.04))
        occupied = np.flatnonzero(np.convolve(profile, np.ones(smooth) / smooth, mode='same') >= 0.035)
        if not len(occupied):
            return [polygon]
        gaps = np.flatnonzero(np.diff(occupied) > max(3, round(height * 0.05)))
        runs = np.split(occupied, gaps + 1)
        runs = [run for run in runs if run[-1] - run[0] >= max(8, height * 0.2)]
        if len(runs) != 2:
            return [polygon]
        points = np.asarray(polygon, dtype=np.float32)
        result = []
        for run in runs:
            a = max(0, (run[0] - height * 0.025) / height)
            b = min(1, (run[-1] + 1 + height * 0.025) / height)
            left = points[3] - points[0]
            right = points[2] - points[1]
            result.append(tuple((tuple((float(v) for v in p)) for p in [points[0] + a * left, points[1] + a * right, points[1] + b * right, points[0] + b * left])))
        return result
    
    def _slanted_row_split(ink, polygon):
        """Find a nearly ink-free straight seam between two skewed printed rows."""
        (height, width) = ink.shape
        if height < 32 or width < 3 * height:
            return [polygon]
        scale = min(1.0, 600 / width)
        small = cv2.resize(ink, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else ink
        mask = small > 100
        (h, w) = mask.shape
        xs = np.arange(w)
        mids = np.arange(round(0.3 * h), round(0.7 * h))[:, None]
        best = None
        for drift in np.linspace(-0.3 * h, 0.3 * h, 31):
            ys = np.rint(mids + drift * (xs / max(1, w - 1) - 0.5)).astype(int)
            costs = mask[ys, xs].mean(axis=1)
            for index in np.flatnonzero(costs <= 0.01):
                mid = int(mids[index, 0])
                rank = (float(costs[index]), abs(mid - h / 2) / h + abs(drift) / h * 0.1)
                if best is None or rank < best[0]:
                    best = (rank, mid, drift)
        if best is None:
            return [polygon]
        (_, mid, drift) = best
        seam = np.rint(mid + drift * (xs / max(1, w - 1) - 0.5)).astype(int)
        yy = np.arange(h)[:, None]
        for side in (yy < seam - 2, yy > seam + 2):
            if np.count_nonzero(np.any(mask & side, axis=0)) / w < 0.45:
                return [polygon]
        points = np.asarray(polygon, dtype=np.float32)
        left = points[0] + (mid - drift / 2) / h * (points[3] - points[0])
        right = points[1] + (mid + drift / 2) / h * (points[2] - points[1])
        return [tuple((tuple((float(v) for v in p)) for p in row)) for row in ([points[0], points[1], right, left], [left, right, points[2], points[3]])]
    
    def numeric_region_proposals(image, lines, limit=4):
        """Bounded original detector proposals, including weak numeric fragments."""
        candidates = []
        for line in lines:
            text = line.text.strip()
            if not line.geometry_valid or len(line.polygon) != 4 or len(text) > 40 or (not any((c.isdigit() for c in text))) or re.search('%|kcal|\\b(?:mg|ml|TEL|LOT)\\b|\\d\\s*g\\b|원|가격|영양|전화|상담|수신|품목|보고번호|(?<!\\d)0\\d{1,2}[- )]\\d{2,4}-\\d{3,4}', text, re.I) or re.fullmatch('\\d{1,2}:\\d{2}(?::\\d{2})?', text):
                continue
            if re.search('\\d{12,}', text):
                continue
            numeric = sum((c.isdigit() for c in text))
            if numeric < 3 and (not (line.score < 0.85 and line.width * line.height >= image.shape[0] * image.shape[1] * 0.002)):
                continue
            candidates.append((numeric >= 6, line.width * line.height, line.polygon))
        rows = []
        for (_, __, polygon) in sorted(candidates, key=lambda x: (x[0], x[1]), reverse=True)[:limit]:
            rows.extend(split_row_polygon(image, polygon))
        return rows[:limit]
    
    def dot_row_proposals(image, limit=4, *, thin=False, contrast=15):
        """Small, disconnected dark components grouped into long shallow rows.
    
        Geometry is only a proposal, never a statement that the row is a date.
        Work at a bounded resolution and return at most four original-space quads.
        """
        scale = min(1.0, 1800 / max(image.shape[:2]))
        small = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_LINEAR if thin else cv2.INTER_AREA) if scale < 1 else image
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        ink = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, contrast)
        (count, labels, stats, centers) = cv2.connectedComponentsWithStats(ink, 8)
        max_dot = max(4, round(max(gray.shape) * 0.006))
        valid = np.flatnonzero((stats[:, cv2.CC_STAT_AREA] >= 2) & (stats[:, cv2.CC_STAT_AREA] <= max_dot ** 2 * 0.7) & (stats[:, cv2.CC_STAT_WIDTH] <= max_dot) & (stats[:, cv2.CC_STAT_HEIGHT] <= max_dot))
        valid = valid[valid != 0]
        if len(valid) < 25:
            return []
        lookup = np.zeros(count, dtype=np.uint8)
        lookup[valid] = 255
        dots = lookup[labels]
        vertical = max(2, round(max_dot * 0.4)) if thin else max_dot
        merged = cv2.morphologyEx(dots, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (max_dot * 6, vertical)))
        (contours, _) = cv2.findContours(merged, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        proposals = []
        for contour in contours:
            (x, y, w, h) = cv2.boundingRect(contour)
            if w < 80 or h < 10 or (not (1.5 if thin else 3.5) <= w / h <= 35) or (w > gray.shape[1] * 0.85):
                continue
            selected = valid[(centers[valid, 0] >= x) & (centers[valid, 0] < x + w) & (centers[valid, 1] >= y) & (centers[valid, 1] < y + h)]
            if len(selected) < 25:
                continue
            rectangle = cv2.minAreaRect(contour)
            points = cv2.boxPoints(rectangle)
            ordered = points[np.argsort(np.arctan2(points[:, 1] - points[:, 1].mean(), points[:, 0] - points[:, 0].mean()))]
            ordered = np.roll(ordered, -np.argmin(ordered.sum(axis=1)), axis=0)
            if np.linalg.norm(ordered[1] - ordered[0]) < np.linalg.norm(ordered[3] - ordered[0]):
                continue
            polygon = tuple((tuple((float(v / scale) for v in p)) for p in ordered))
            proposals.append((len(selected), polygon))
        result = []
        for (_, polygon) in sorted(proposals, key=lambda p: p[0], reverse=True)[:limit]:
            result.extend([polygon] if thin else split_row_polygon(image, polygon))
        return result[:limit]
    
    def recover_geometric_rows(image, lines, recognize_crops, fallback_recognize=None):
        """Try up to eight automatic rows, two deskewed views per recognizer.
    
        Used only after normal selection abstains. No annotation/answer lookup;
        one row remains one selector observation, not multiple votes. Both views
        must pass the existing strict character-evidence gate without digit repair.
        """
        OCRLine = _ITDA_NS['date_extraction'].OCRLine
        _apply_views = _ITDA_NS['line_recovery']._apply_views
        _digit_evidence = _ITDA_NS['line_recovery']._digit_evidence
        _signature = _ITDA_NS['line_recovery']._signature
        started = time.perf_counter()
        polygons = numeric_region_proposals(image, lines) + dot_row_proposals(image)
        (anchors, jobs, crops) = ([], [], [])
        for polygon in polygons:
            views = recognition_views(image, polygon)
            if len(views) != 2:
                continue
            (xs, ys) = zip(*polygon)
            bounds = (min(xs), min(ys), max(xs), max(ys))
            index = len(anchors)
            anchors.append(OCRLine('', 0.0, bounds, source='paddle-geometric', variant='geometric-rows', polygon=polygon, original_box=bounds))
            for (padding, crop) in views:
                crops.append(crop)
                jobs.append((index, padding, bounds))
        if not crops:
            return ([], [], [], time.perf_counter() - started)
        (observations, decisions, choices) = ([], [], {})
        for (name, recognizer) in [('primary', recognize_crops), ('english', fallback_recognize)]:
            if recognizer is None:
                continue
            try:
                results = recognizer(crops)
                if len(results) != len(jobs):
                    raise ValueError('Geometric recognition count does not match requested crops')
                extra = []
                for ((index, padding, bounds), result) in zip(jobs, results):
                    chars = tuple(result[2]) if len(result) > 2 else ()
                    (mean, minimum) = _digit_evidence(result[0], chars)
                    extra.append(replace(anchors[index], text=result[0], score=result[1], variant=f'geometry-{index}-{padding}-{name}', character_scores=chars, date_digit_score=mean, date_digit_min_score=minimum))
                active = list(anchors)
                audit = _apply_views(active, anchors, anchors, jobs, extra, strict=True)
                for decision in audit:
                    index = decision['line_index']
                    decision.update(recognizer=name, stage='automatic-geometric-rows', line_index=len(lines) + index)
                    if decision['accepted_text']:
                        choices.setdefault(index, []).append(active[index])
                decisions.extend(audit)
                observations.extend(extra)
                if name == 'primary':
                    pending = [d['line_index'] - len(lines) for d in audit if not d['accepted_text'] and d['decision_basis'] != 'conflicting-dates' and any((e['signature'] and e['score'] >= 0.65 for e in d['evidence']))][:2]
                    (retry_jobs, retry_crops) = ([], [])
                    for index in pending:
                        crop = rectified_crop(image, anchors[index].polygon)
                        if crop is None:
                            continue
                        h = crop.shape[0]
                        plain = cv2.resize(crop, (round(h * 4.8), h), interpolation=cv2.INTER_AREA)
                        gray = cv2.cvtColor(plain, cv2.COLOR_BGR2GRAY)
                        contrast = cv2.cvtColor(cv2.createCLAHE(2.0, (4, 4)).apply(gray), cv2.COLOR_GRAY2BGR)
                        if np.array_equal(plain, contrast):
                            continue
                        for (variant, view) in [('unmargined', plain), ('unmargined-contrast', contrast)]:
                            retry_jobs.append((index, variant, anchors[index].box))
                            retry_crops.append(view)
                    if retry_crops:
                        retry_results = recognizer(retry_crops)
                        if len(retry_results) != len(retry_jobs):
                            raise ValueError('Unmargined recognition count mismatch')
                        retry_obs = []
                        for ((index, variant, bounds), result) in zip(retry_jobs, retry_results):
                            chars = tuple(result[2]) if len(result) > 2 else ()
                            (mean, minimum) = _digit_evidence(result[0], chars)
                            retry_obs.append(replace(anchors[index], text=result[0], score=result[1], variant=variant, character_scores=chars, date_digit_score=mean, date_digit_min_score=minimum))
                        active = list(anchors)
                        retry_audit = _apply_views(active, anchors, anchors, retry_jobs, retry_obs, strict=True)
                        for decision in retry_audit:
                            index = decision['line_index']
                            if decision['accepted_text']:
                                choices.setdefault(index, []).append(active[index])
                            decision.update(recognizer=name, stage='unmargined-geometric-row', line_index=len(lines) + index)
                        decisions.extend(retry_audit)
                        observations.extend(retry_obs)
            except Exception as exc:
                decisions.append(dict(line_index=-1, original_text='', original_box=None, accepted_text=None, reason='geometric-recognition-error', recognizer=name, error=f'{type(exc).__name__}: {exc}'))
        (seam_anchors, seam_jobs, seam_crops) = ([], [], [])
        for (index, anchor) in enumerate(anchors):
            if index in choices:
                continue
            crop = rectified_crop(image, anchor.polygon)
            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            ink = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 15)
            split = _slanted_row_split(ink, anchor.polygon)
            if len(split) != 2:
                continue
            for polygon in split:
                crop = rectified_crop(image, polygon)
                if crop is None:
                    continue
                h = crop.shape[0]
                plain = cv2.resize(crop, (round(h * 4.8), h), interpolation=cv2.INTER_AREA)
                gray = cv2.cvtColor(plain, cv2.COLOR_BGR2GRAY)
                contrast = cv2.cvtColor(cv2.createCLAHE(2.0, (4, 4)).apply(gray), cv2.COLOR_GRAY2BGR)
                if np.array_equal(plain, contrast):
                    continue
                (xs, ys) = zip(*polygon)
                bounds = (min(xs), min(ys), max(xs), max(ys))
                child = len(seam_anchors)
                seam_anchors.append(replace(anchor, box=bounds, original_box=bounds, polygon=polygon))
                for (variant, view) in [('seam-plain', plain), ('seam-contrast', contrast)]:
                    seam_jobs.append((child, variant, bounds))
                    seam_crops.append(view)
            if len(seam_anchors) >= 4:
                break
        if seam_crops:
            try:
                results = recognize_crops(seam_crops)
                if len(results) != len(seam_jobs):
                    raise ValueError('Seam recognition count mismatch')
                extra = []
                for ((index, variant, bounds), result) in zip(seam_jobs, results):
                    chars = tuple(result[2]) if len(result) > 2 else ()
                    (mean, minimum) = _digit_evidence(result[0], chars)
                    extra.append(replace(seam_anchors[index], text=result[0], score=result[1], variant=variant, character_scores=chars, date_digit_score=mean, date_digit_min_score=minimum))
                active = list(seam_anchors)
                audit = _apply_views(active, seam_anchors, seam_anchors, seam_jobs, extra, strict=True)
                offset = len(anchors)
                for decision in audit:
                    index = decision['line_index']
                    if decision['accepted_text']:
                        choices[offset + index] = [active[index]]
                    decision.update(stage='slanted-seam-row', recognizer='primary', line_index=len(lines) + offset + index)
                decisions.extend(audit)
                observations.extend(extra)
            except Exception as exc:
                decisions.append(dict(line_index=-1, original_text='', original_box=None, accepted_text=None, reason='seam-recognition-error', error=f'{type(exc).__name__}: {exc}'))
        accepted = []
        for (index, values) in choices.items():
            if len({_signature(value.text) for value in values}) > 1:
                for decision in decisions:
                    if decision['line_index'] == len(lines) + index:
                        decision.update(accepted_text=None, decision_basis='cross-recognizer-conflict')
                continue
            accepted.append(max(values, key=lambda value: value.score))
        discarded = set()
        for (i, first) in enumerate(accepted):
            for (j, second) in enumerate(accepted[:i]):
                (a, b) = (first.box, second.box)
                intersection = max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
                if intersection / min(first.width * first.height, second.width * second.height) < 0.7:
                    continue
                if _signature(first.text) != _signature(second.text):
                    discarded.update((i, j))
                else:
                    discarded.add(i if first.score <= second.score else j)
        kept = [value for (i, value) in enumerate(accepted) if i not in discarded]
        for decision in decisions:
            if decision.get('accepted_text') and (not any((value.box == decision['original_box'] and value.text == decision['accepted_text'] for value in kept))):
                decision.update(accepted_text=None, decision_basis='overlapping-row-conflict-or-duplicate')
        return (kept, observations, decisions, time.perf_counter() - started)
    return _ITDA_Namespace(**locals())
_ITDA_NS['date_region_recovery'] = _itda_define_date_region_recovery()
del _itda_define_date_region_recovery


In [ ]:
from __future__ import annotations
# Embedded from src/performance_profile.py; generated, do not edit separately.
def _itda_define_performance_profile():
    """Opt-in predictor timings; generator consumer time is excluded."""
    import json
    import time
    from contextlib import contextmanager
    from pathlib import Path
    
    class Profile:
    
        def __init__(self, path):
            self.path = Path(path) if path else None
            self.image_id = None
            if self.path:
                self.path.parent.mkdir(parents=True, exist_ok=True)
                self.path.open('x', encoding='utf-8').close()
    
        def emit(self, **event):
            if self.path:
                with self.path.open('a', encoding='utf-8') as stream:
                    stream.write(json.dumps(dict(image_id=self.image_id, **event)) + '\n')
    
        @contextmanager
        def measure(self, stage):
            (wall, cpu) = (time.perf_counter(), time.process_time())
            error = None
            try:
                yield
            except BaseException as exc:
                error = repr(exc)
                raise
            finally:
                self.emit(stage=stage, wall_seconds=time.perf_counter() - wall, cpu_seconds=time.process_time() - cpu, error=error)
    
        def wrap(self, model, stage):
            return TimedPredictor(model, self, stage) if self.path else model
    
    class TimedPredictor:
    
        def __init__(self, model, profile, stage):
            object.__setattr__(self, '_model', model)
            object.__setattr__(self, '_profile', profile)
            object.__setattr__(self, '_stage', stage)
    
        def __getattr__(self, name):
            return getattr(self._model, name)
    
        def __setattr__(self, name, value):
            setattr(self._model, name, value)
    
        def __call__(self, *args, **kwargs):
            (wall, cpu, count) = (0.0, 0.0, 0)
            error = None
            values = args[0] if args and isinstance(args[0], list) else list(args[:1])
            shapes = [list(v.shape) for v in values if hasattr(v, 'shape')]
            (started, processor) = (time.perf_counter(), time.process_time())
            try:
                iterator = iter(self._model(*args, **kwargs))
            except BaseException as exc:
                self._profile.emit(stage=self._stage, wall_seconds=time.perf_counter() - started, cpu_seconds=time.process_time() - processor, results=0, shapes=shapes, error=repr(exc))
                raise
            wall += time.perf_counter() - started
            cpu += time.process_time() - processor
            try:
                while True:
                    (started, processor) = (time.perf_counter(), time.process_time())
                    try:
                        value = next(iterator)
                    except StopIteration:
                        break
                    finally:
                        wall += time.perf_counter() - started
                        cpu += time.process_time() - processor
                    count += 1
                    yield value
            except BaseException as exc:
                error = repr(exc)
                raise
            finally:
                close = getattr(iterator, 'close', None)
                if close:
                    close()
                self._profile.emit(stage=self._stage, wall_seconds=wall, cpu_seconds=cpu, results=count, shapes=shapes, error=error)
    return _ITDA_Namespace(**locals())
_ITDA_NS['performance_profile'] = _itda_define_performance_profile()
del _itda_define_performance_profile


In [ ]:
from __future__ import annotations
# Embedded from src/recognizer_padding.py; generated, do not edit separately.
def _itda_define_recognizer_padding():
    """Exclude CTC frames beyond real resized pixels, not date-specific characters."""
    import math
    import numpy as np
    
    def mask_padding_frames(probabilities, valid_ratios):
        values = np.asarray(probabilities)
        if values.ndim != 3 or len(valid_ratios) != values.shape[0]:
            raise ValueError('Padding guard CTC batch/geometry mismatch')
        ratios = np.asarray(valid_ratios, dtype=float)
        if not np.isfinite(ratios).all() or np.any(ratios <= 0) or np.any(ratios > 1):
            raise ValueError('Invalid real-image width ratios')
        if np.all(ratios == 1):
            return probabilities
        result = values.copy()
        for (row, ratio) in enumerate(ratios):
            end = int(math.ceil(values.shape[1] * ratio))
            result[row, end:, :] = 0
            result[row, end:, 0] = 1
        return result
    
    class GeometryResize:
    
        def __init__(self, resize):
            self.resize = resize
            self.valid_ratios = []
    
        def __getattr__(self, name):
            return getattr(self.resize, name)
    
        @property
        def rec_image_shape(self):
            return self.resize.rec_image_shape
    
        @rec_image_shape.setter
        def rec_image_shape(self, value):
            self.resize.rec_image_shape = value
    
        def __call__(self, imgs):
            normalized = self.resize(imgs)
            if len(normalized) != len(imgs):
                raise ValueError('Resize changed image count')
            ratios = []
            for (image, tensor) in zip(imgs, normalized):
                (height, width) = tensor.shape[-2:]
                if self.resize.input_shape is not None:
                    valid_width = width
                else:
                    valid_width = min(width, int(math.ceil(height * image.shape[1] / image.shape[0])))
                ratios.append(valid_width / width)
            self.valid_ratios = ratios
            return normalized
    
    class GeometryRunner:
    
        def __init__(self, runner, resize):
            (self.runner, self.resize) = (runner, resize)
    
        def __getattr__(self, name):
            return getattr(self.runner, name)
    
        def __call__(self, *args, **kwargs):
            output = self.runner(*args, **kwargs)
            if not isinstance(output, (list, tuple)) or len(output) != 1:
                raise ValueError('Unexpected recognizer output contract')
            masked = mask_padding_frames(output[0], self.resize.valid_ratios)
            return (masked,) if isinstance(output, tuple) else [masked]
    
    def install_padding_guard(model):
        resize = model.pre_tfs['ReisizeNorm']
        if isinstance(resize, GeometryResize):
            return
        if list(resize.rec_image_shape)[:2] != [3, 48] or not hasattr(model, 'runner'):
            raise ValueError('Unsupported recognizer padding contract')
        wrapped = GeometryResize(resize)
        model.pre_tfs['ReisizeNorm'] = wrapped
        model.runner = GeometryRunner(model.runner, wrapped)
    return _ITDA_Namespace(**locals())
_ITDA_NS['recognizer_padding'] = _itda_define_recognizer_padding()
del _itda_define_recognizer_padding


In [ ]:
from __future__ import annotations
# Embedded from src/shared_detector.py; generated, do not edit separately.
def _itda_define_shared_detector():
    """Reuse the mobile pipeline's Korean recognizer in serial secondary-detector passes."""
    
    def inner_ocr_pipeline(model):
        """Resolve actual storage, not AutoParallel's forwarding __getattr__."""
        pipeline = model.paddlex_pipeline
        seen = set()
        while id(pipeline) not in seen:
            seen.add(id(pipeline))
            attributes = vars(pipeline)
            if 'text_rec_model' in attributes and 'text_det_model' in attributes:
                return pipeline
            pipeline = attributes.get('_pipeline')
            if pipeline is None:
                break
        raise TypeError('Unsupported PaddleX OCR pipeline structure; refusing a shadow model assignment')
    
    class SharedDetectorView:
    
        def __init__(self, mobile, detector, side_limit):
            (self.mobile, self.detector, self.side_limit) = (mobile, detector, side_limit)
    
        def predict(self, image):
            pipeline = inner_ocr_pipeline(self.mobile)
            original = pipeline.text_det_model
            try:
                pipeline.text_det_model = self.detector
                return list(self.mobile.predict(image, text_det_limit_side_len=self.side_limit))
            finally:
                pipeline.text_det_model = original
    return _ITDA_Namespace(**locals())
_ITDA_NS['shared_detector'] = _itda_define_shared_detector()
del _itda_define_shared_detector


In [ ]:
from __future__ import annotations
# Embedded from src/date_extraction.py; generated, do not edit separately.
def _itda_define_date_extraction():
    import math
    import re
    import unicodedata
    from collections.abc import Iterable, Sequence
    from dataclasses import dataclass, replace
    from datetime import date
    from functools import lru_cache
    FIELDS = _ITDA_NS['date_fields'].FIELDS
    fields_from_date = _ITDA_NS['date_fields'].fields_from_date
    serialize_fields = _ITDA_NS['date_fields'].serialize_fields
    MIN_YEAR = 1
    MAX_YEAR = 2099
    POSITIVE_CONTEXT = re.compile('소비\\s*기한|유통\\s*기한|사용\\s*기한|품질\\s*유지\\s*기한|까지|EXP(?:IRY|IRES|DATE)?|USE\\s*BY|BEST\\s*(?:BEFORE|BY)|SON\\s*KUL\\.?\\s*TA\\.?|BB[DE]?\\b|TETT\\b|賞味|有效期?|(?<![A-Z])ED(?=\\s*[:.]?\\s*\\d)', re.IGNORECASE)
    NEGATIVE_CONTEXT = re.compile('제조(?:일자|일)?|생산(?:일자|일)?|포장(?:일자|일)?|부터|MFG|MFD|(?<![A-Z])PROD(?:UCTION)?(?![A-Z])|PACK(?:ED)?\\s*ON|(?<![A-Z])PD(?=\\s*[:.]?\\s*\\d)', re.IGNORECASE)
    UNTIL_CONTEXT = re.compile('까지|EXP(?:IRY|IRES|DATE)?|USE\\s*BY|BEST\\s*(?:BEFORE|BY)', re.IGNORECASE)
    FROM_CONTEXT = re.compile('부터|제조(?:일자|일)?|생산(?:일자|일)?|MFG|MFD|(?<![A-Z])PROD(?:UCTION)?(?![A-Z])', re.IGNORECASE)
    MONTHS = {'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6, 'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12}
    
    @dataclass(frozen=True)
    class OCRLine:
        text: str
        score: float
        box: tuple[float, float, float, float]
        source: str = 'paddle-mobile'
        variant: str = 'original'
        members: tuple[int, ...] = ()
        polygon: tuple[tuple[float, float], ...] = ()
        geometry_valid: bool = True
        geometry_source: str = 'box'
        original_box: tuple[float, float, float, float] | None = None
        role: str | None = None
        role_basis: str | None = None
        character_scores: tuple[float, ...] = ()
        date_digit_score: float | None = None
        date_digit_min_score: float | None = None
    
        @property
        def width(self) -> float:
            return max(1.0, self.box[2] - self.box[0])
    
        @property
        def height(self) -> float:
            return max(1.0, self.box[3] - self.box[1])
    
        @property
        def center(self) -> tuple[float, float]:
            return ((self.box[0] + self.box[2]) / 2, (self.box[1] + self.box[3]) / 2)
    
    @dataclass(frozen=True)
    class ParsedDate:
        value: date
        raw: str
        start: int
        end: int
        year_digits: int
        separator: str
        repaired: bool = False
        order: str = 'ymd'
        order_reason: str = 'calendar_unique'
    
    @dataclass(frozen=True)
    class DateCandidate:
        value: date
        raw: str
        score: float
        ocr_score: float
        year_digits: int
        repaired: bool
        separator: str
        source: str
        variant: str
        positive_hits: tuple[str, ...] = ()
        negative_hits: tuple[str, ...] = ()
        box: tuple[float, float, float, float] = (0, 0, 0, 0)
        members: tuple[int, ...] = ()
        explicit_positive: bool = False
        explicit_negative: bool = False
        span: tuple[int, int] = (0, 0)
        order: str = 'ymd'
        order_reason: str = 'calendar_unique'
        repair_kind: str | None = None
    
        @property
        def iso(self) -> str:
            return self.value.isoformat()
    
    @dataclass(frozen=True)
    class DateSelection:
        final_date: str | None
        score: float
        margin: float
        confident: bool
        reason: str
        candidates: tuple[DateCandidate, ...]
        digits_confident: bool = False
        order_resolved: bool = False
        policy_details: dict | None = None
        role_resolved: bool = False
        output_fields: dict | None = None
    
        @property
        def stop_ocr(self) -> bool:
            """Execution decision, not a claim that date order is proven.
    
            `confident` remains the compatible legacy spelling. A clear-digit DMY
            fallback can stop OCR with order_resolved=False; forced final output
            can stop with digits_confident=False.
            """
            return self.confident
    
        @property
        def is_partial(self) -> bool:
            return self.final_date is not None and 'NONE' in self.final_date
    
    @dataclass(frozen=True)
    class SubmissionDecision:
        candidate_date: str | None
        output_date: str | None
        status: str
        reason: str
        recovery_reason: str | None
    
    def submission_decision(selection: DateSelection, *, base_partial=False) -> SubmissionDecision:
        """Evidence acceptance is independent of the scheduler's stop_ocr flag."""
        candidate = selection.final_date
        reason = selection.reason
        accepted = False
        recovery = None
        if candidate is None:
            recovery = None if reason in {'negative-context', 'expiry-not-printed'} else 'order' if reason.startswith('review_order:') else 'digits' if selection.candidates else 'detection'
        elif selection.is_partial:
            accepted = base_partial or reason == 'explicit-partial-date'
            recovery = None if reason == 'explicit-partial-date' else 'digits'
        else:
            best = next((c for c in selection.candidates if c.iso == candidate), None)
            if reason.startswith('review_'):
                recovery = 'order' if reason.startswith('review_order:') else 'digits'
            elif reason == 'accepted' and best is not None:
                accepted = True
            elif best is None or best.ocr_score < 0.65 or (best.repaired and best.repair_kind != 'format-only'):
                recovery = 'digits'
            elif not selection.order_resolved:
                accepted = reason == 'fallback_dmy' and selection.stop_ocr
                recovery = None if accepted else 'order'
            else:
                rivals = [c for c in selection.candidates if c.iso != candidate and (not (c.explicit_negative and (not c.explicit_positive))) and (not (c.order_reason.startswith('review_') and (not c.explicit_positive))) and (not (c.year_digits == 2 and (not c.explicit_positive) and (c.order_reason == 'calendar_unique')))]
                role_supported = best.explicit_positive and (not best.explicit_negative) and (not any((c.explicit_positive for c in rivals)))
                unique = not rivals and (not best.explicit_negative) and (not best.repaired or best.repair_kind == 'format-only') and (best.score >= 1.05)
                accepted = role_supported or unique or (selection.role_resolved and (not best.explicit_negative)) or (selection.stop_ocr and reason in {'accepted', 'local-date-pair'} and (not (best.explicit_negative and (not best.explicit_positive))))
                recovery = None if accepted else 'role'
        status = 'SELECTED' if accepted else 'NOT_FOUND' if recovery is None else 'REVIEW_REQUIRED'
        return SubmissionDecision(candidate, candidate if accepted else None, status, reason, recovery)
    
    @dataclass(frozen=True)
    class ProductDateRule:
        """Locally verified package rule, never an image-ID or country shortcut."""
        name: str
        required_text: tuple[str, ...]
        order: str
        evidence: str
    
        def __post_init__(self):
            if self.order not in {'ymd', 'dmy', 'mdy'}:
                raise ValueError('Invalid product date order')
            if not self.name.strip() or not self.evidence.strip() or (not self.required_text) or any((not token.strip() for token in self.required_text)):
                raise ValueError('Product rules require package identifiers and verification evidence')
    
    @dataclass(frozen=True)
    class DateContext:
        """Caller-supplied product context; None market is inferred, never image-ID based.
    
        Passing this activates the 2026-09-11 market/review policy. Omitting it from
        the low-level parser retains the old DMY API for reproducible comparisons.
        """
        market: str | None = None
        product_type: str = 'food'
        language: str | None = None
    
        def __post_init__(self):
            if self.market not in {None, 'KR', 'JP', 'US', 'GB', 'EU'}:
                raise ValueError('Unsupported market; use None for unknown')
    _KOREAN_EXPIRY = re.compile('소비\\s*기한|유통\\s*기한|품질\\s*유지\\s*기한|까지')
    _IMPORT_CONTEXT = re.compile('수입|(?:원산지|제조국)\\s*[:：]?\\s*(?:중국|미국|일본|영국|독일|벨기에|터키|튀르키예)|MADE\\s+IN\\s+(?!KOREA)\\w+', re.I)
    _CLOCK = re.compile('(?<![\\d:])(?:[01]?\\d|2[0-3]):[0-5]\\d(?::[0-5]\\d)?(?![\\d:])')
    _OTHER_DATE = re.compile('할인\\s*판매\\s*시작일|판매\\s*시작일|허가\\s*일|등록\\s*일|Ruhsat\\s*numaras', re.I)
    
    def _market_order(line, originals, context):
        if context is None or context.product_type != 'food':
            return (None, 'unknown_context')
        nearby = [other for other in originals if other.geometry_valid and other.score >= 0.8 and ((other.source, other.variant) == (line.source, line.variant)) and (math.hypot(*_box_gap(line, other)) <= 8 * max(line.height, other.height))]
        korean = line.role == 'end' and line.role_basis == 'printed_reference' or bool(_KOREAN_EXPIRY.search(line.text)) or (line.role in {'start', 'end'} and any((_KOREAN_EXPIRY.search(other.text) for other in nearby)))
        if not korean and line.role == 'end' and line.original_box:
            korean = any((other.original_box and other.score >= 0.8 and _KOREAN_EXPIRY.search(other.text) and (_same_region(line, other) or (not parse_dates(other.text) and len(other.text) <= 32 and (abs((line.original_box[1] + line.original_box[3]) / 2 - (other.original_box[1] + other.original_box[3]) / 2) <= 0.5 * max(line.original_box[3] - line.original_box[1], other.original_box[3] - other.original_box[1])) and (math.hypot(*_box_gap(replace(line, box=line.original_box), replace(other, box=other.original_box))) <= 6 * max(line.original_box[3] - line.original_box[1], other.original_box[3] - other.original_box[1])))) for other in originals))
        imported = any((_IMPORT_CONTEXT.search(other.text) for other in nearby))
        if not imported and line.original_box:
            anchor = replace(line, box=line.original_box)
            imported = any((other.original_box and other.geometry_valid and (other.score >= 0.8) and _IMPORT_CONTEXT.search(other.text) and (math.hypot(*_box_gap(anchor, replace(other, box=other.original_box))) <= 8 * max(anchor.height, other.original_box[3] - other.original_box[1])) for other in originals))
        if context.market is not None:
            if context.market == 'KR' and imported:
                return (None, 'conflicting_import_context')
            if context.market == 'KR' and (not (korean or context.language == 'ko')):
                return (None, 'missing_korean_expiry_context')
            return ({'KR': 'ymd', 'JP': 'ymd', 'US': 'mdy', 'GB': 'dmy', 'EU': 'dmy'}[context.market], 'metadata:' + context.market)
        if imported:
            return (None, 'import_context')
        if korean and line.role != 'conflict':
            return ('ymd', 'korean_food_context')
        return (None, 'unknown_context')
    
    def _mask_auxiliary(text):
        """Mask clocks/barcode candidates without shifting date or character offsets."""
        chars = list(text)
        spans = [m.span() for m in _CLOCK.finditer(text)]
        spans.extend((m.span() for m in re.finditer('(?<![\\d./-])\\d+(?:[.,]\\d+)?\\s*(?:mg|kg|g|kcal|%)(?![A-Za-z])', text, re.I)))
        for match in re.finditer('(?<![\\d.\\-/])\\d(?:[ \\t]*\\d){7,13}(?![\\d.\\-/])', text):
            if re.match('(?:\\d{4}\\s+\\d{1,2}\\s+\\d{1,2}|\\d{1,2}\\s+\\d{1,2}\\s+\\d{4})(?!\\d)', match[0]):
                continue
            if re.fullmatch('20\\d{6}', match[0]) and parse_dates(match[0]):
                continue
            spans.append(match.span())
        protected = [(p.start, p.end) for p in parse_dates(text) if not p.repaired and p.year_digits == 4 and re.fullmatch('20\\d{2}\\s*[./-]\\s*\\d{1,2}\\s*[./:\\-]\\s*\\d{1,2}', p.raw)]
        for (start, end) in spans:
            if any((a <= start and end <= b for (a, b) in protected)):
                continue
            chars[start:end] = ' ' * (end - start)
        return ''.join(chars)
    
    def _policy_parses(text):
        masked = _mask_auxiliary(text)
        return [p for p in parse_dates(text) if not any((masked[i] == ' ' and text[i] != ' ' for i in range(p.start, min(p.end, len(text)))))]
    
    def _valid_date(year: int, month: int, day: int) -> date | None:
        if year < 100:
            year += 2000
        if not MIN_YEAR <= year <= MAX_YEAR:
            return None
        try:
            return date(year, month, day)
        except ValueError:
            return None
    
    def _normalise_text(text: str) -> str:
        value = unicodedata.normalize('NFKC', text or '')
        value = re.sub('[‐‑‒–—―−]', '-', value)
        return re.sub('\\s+', ' ', value).strip()
    
    def _date_text(text: str) -> str:
        value = _normalise_text(text)
        month_key = list(re.finditer('\\b(?:' + '|'.join(MONTHS) + ')\\s*-\\s*\\d{1,2}\\s*월', value, re.I))
        if len(month_key) >= 3:
            for item in month_key:
                value = value[:item.start()] + ' ' * (item.end() - item.start()) + value[item.end():]
        for (start, end, _) in _format_hints(value):
            value = value[:start] + ' ' * (end - start) + value[end:]
        return value.replace('년', '.').replace('월', '.').replace('일', ' ').replace('·', '.')
    
    def _format_hints(text: str) -> list[tuple[int, int, str]]:
        gap = '\\s*[/.,\\-]?\\s*'
        hints = []
        for (order, letters, korean) in (('ymd', ('Y{2,4}', 'MM', 'DD'), ('[년연]{1,2}', '월{1,2}', '일{1,2}')), ('dmy', ('DD', 'MM', 'Y{2,4}'), ('일{1,2}', '월{1,2}', '[년연]{1,2}')), ('mdy', ('MM', 'DD', 'Y{2,4}'), ('월{1,2}', '일{1,2}', '[년연]{1,2}'))):
            patterns = ('(?<![A-Z])' + gap.join(letters) + '(?![A-Z])', gap.join(('(?:00)?' + part for part in korean)))
            for pattern in patterns:
                hints.extend(((m.start(), m.end(), order) for m in re.finditer(pattern, text, re.I)))
        hints.extend(((m.start(), m.end(), 'ymd') for m in re.finditer('[년연]{2}\\s*[./,\\-]?\\s*월{2}(?!\\s*[./,\\-]?\\s*일)', text)))
        return hints
    
    def _span_gap(first: tuple[int, int], second: tuple[int, int]) -> int:
        return max(0, first[0] - second[1], second[0] - first[1])
    
    def _repaired_text(text: str) -> str:
        text = re.sub('(?<!\\d)2[UD](?=\\d{2}\\s*[./-])', '20', text, flags=re.I)
        text = re.sub('(?<=\\d)\\$(?=\\s*(?:까지|$))', '5', text)
        return text.translate(str.maketrans({'O': '0', 'o': '0', 'Q': '0', 'I': '1', 'l': '1', '|': '1', 'Z': '2', 'S': '5', 'B': '8'}))
    
    def _emit_match(match: re.Match[str], order: str, year_digits: int, separator: str, repaired: bool) -> ParsedDate | None:
        groups = match.groups()
        if order == 'ymd':
            (year, month, day) = (int(groups[0]), int(groups[1]), int(groups[2]))
        elif order == 'dmy':
            (day, month, year) = (int(groups[0]), int(groups[1]), int(groups[2]))
        elif order == 'mdy':
            (month, day, year) = (int(groups[0]), int(groups[1]), int(groups[2]))
        else:
            raise ValueError(f'Unknown date order: {order}')
        parsed = _valid_date(year, month, day)
        if parsed is None:
            return None
        return ParsedDate(parsed, match.group(0), match.start(), match.end(), year_digits, separator, repaired, order)
    
    def _iter_numeric_dates(text: str, repaired: bool) -> Iterable[ParsedDate]:
        for joined in list(re.finditer('(?<!\\d)(\\d{2}[./-]\\d{2}[./-]\\d{2})([0-2]\\d:[0-5]\\d)(?!\\d)', text)):
            if int(joined[2][:2]) <= 23:
                (start, end) = joined.span(2)
                text = text[:start] + ' ' * (end - start) + text[end:]
        separator = '[./,:\\-]'
        delimiter = '(?:\\s*[./,:\\-]{1,2}\\s*|\\s+)'
        occupied = []
        numeric = re.compile(f'(?<!\\d)(20\\d{{2}}|\\d{{1,2}}){delimiter}(\\d{{1,2}}){delimiter}((?:19|20)\\d{{2}}|\\d{{1,2}})')
        for start in re.finditer('(?=\\d)', text):
            match = numeric.match(text, start.start())
            if match is None:
                continue
            occupied.append((match.start(), match.end()))
            if max(len(match[1]), len(match[3])) < 4 and re.search('[./,:\\-]{2}', match[0]):
                continue
            if re.fullmatch('\\d{1,2}\\s+\\d{1,2}:\\d{1,2}', match[0]):
                continue
            if re.fullmatch('\\d{1,2}:\\d{1,2}\\s+\\d{1,2}', match[0]):
                continue
            tail = text[match.end():]
            prefix = text[:match.start()]
            if re.fullmatch('20\\d{2}\\s*[./-]\\s*\\d\\s+\\d', match[0]) and re.match('\\s*[./-]\\s*\\d', tail):
                continue
            if len(match[3]) == 1 and tail.startswith('$'):
                continue
            if re.fullmatch('\\d\\s+\\d{1,2}[./-]\\d{1,2}', match[0]) and re.match('\\s+\\d{1,2}:\\d{2}', tail):
                continue
            long_year = len(match[1]) == 4
            attached_lot_digit = len(match[1]) == 1 and prefix and prefix[-1].isalpha() and (not re.search('(?:EXP|BBE?)$', prefix, re.I))
            if not long_year and (re.search('\\d[./,:\\-]+\\s*$', prefix) or attached_lot_digit):
                continue
            if re.match('\\s*:', tail) and len(match[3]) != 4 or (not long_year and tail[:1].isdigit() and (not re.match('(?:\\d{1,2}[^\\W\\d_]|\\d{2}:\\d{2})', tail))):
                continue
            for order in ('ymd', 'dmy', 'mdy'):
                year_index = 1 if order == 'ymd' else 3
                day_index = 3 if order == 'ymd' else 1 if order == 'dmy' else 2
                month_index = 1 if order == 'mdy' else 2
                if len(match[year_index]) not in (2, 4) or len(match[day_index]) > 2 or len(match[month_index]) > 2:
                    continue
                parsed = _emit_match(match, order, len(match[year_index]), 'separated', repaired)
                if parsed:
                    yield parsed
        patterns: list[tuple[str, str, int, str]] = [(f'(?<!\\d)(20\\d{{2}})\\s*{separator}\\s*(\\d{{2}})(\\d{{2}})(?!\\d)', 'ymd', 4, 'joined'), (f'(?<!\\d)(20\\d{{2}})(\\d{{2}})\\s*{separator}\\s*(\\d{{1,2}})(?!\\d)', 'ymd', 4, 'joined'), ('(?<!\\d)(20\\d{2})(\\d{2})(\\d{2})(?!\\d)', 'ymd', 4, 'compact'), *(('(?<!\\d)(\\d{2})(\\d{2})(\\d{2})(?!\\d)', order, 2, 'compact') for order in ('ymd', 'dmy', 'mdy')), *(('(?<!\\d)(\\d{2})(\\d{2})(20\\d{2})(?!\\d)', order, 4, 'compact') for order in ('dmy', 'mdy')), *(('(?<!\\d)(\\d{1,2})\\s+(\\d{2})(20\\d{2})(?!\\d)', order, 4, 'joined') for order in ('dmy', 'mdy')), *(('(?<!\\d)(\\d{2})(\\d{2})\\s*[./-]\\s*(20\\d{2})(?!\\d)', order, 4, 'joined') for order in ('dmy', 'mdy')), *(('(?<!\\d)(\\d{2})(\\d{2})\\s*/\\s*(\\d{2})(?!\\d)', order, 2, 'joined') for order in ('ymd', 'dmy', 'mdy'))]
        for (pattern, order, year_digits, sep_name) in patterns:
            for match in re.finditer(pattern, text):
                if any((max(start, match.start()) < min(end, match.end()) for (start, end) in occupied)):
                    continue
                parsed = _emit_match(match, order, year_digits, sep_name, repaired)
                if parsed:
                    yield parsed
        for match in re.finditer('(?<!\\d)(20\\d{2})\\s*[./-]\\s*(\\d\\s+\\d)\\s*[./-]\\s*(\\d{1,2})(?!\\d)', text):
            parsed = _valid_date(int(match[1]), int(re.sub('\\s', '', match[2])), int(match[3]))
            if parsed:
                yield ParsedDate(parsed, match[0], match.start(), match.end(), 4, 'repaired-spacing', True)
        for match in re.finditer(f'(?<!\\d)0(2\\d){delimiter}(\\d{{1,2}}){delimiter}(\\d{{1,2}})', text):
            parsed = _valid_date(2000 + int(match.group(1)), int(match.group(2)), int(match.group(3)))
            if parsed:
                yield ParsedDate(parsed, match.group(0), match.start(), match.end(), 4, 'repaired-year', True)
        for match in re.finditer(f'(?<!\\d)[01](1[5-9]|2\\d|3[0-5]){delimiter}(\\d{{1,2}}){delimiter}(\\d{{1,2}})', text):
            parsed = _valid_date(2000 + int(match.group(1)), int(match.group(2)), int(match.group(3)))
            if parsed:
                yield ParsedDate(parsed, match.group(0), match.start(), match.end(), 4, 'repaired-year', True)
        for match in re.finditer(f'(?<!\\d)(20\\d{{2}})[A-Z](\\d{{1,2}}){delimiter}(\\d{{1,2}})', text, re.IGNORECASE):
            parsed = _valid_date(int(match.group(1)), int(match.group(2)), int(match.group(3)))
            if parsed:
                yield ParsedDate(parsed, match.group(0), match.start(), match.end(), 4, 'repaired-separator', True)
    
    def _iter_month_name_dates(text: str, repaired: bool) -> Iterable[ParsedDate]:
        month = 'JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC'
        original = text
        text = re.sub('(?<=\\d)([ /.,-]*)0CT(?=[ /.,-]*\\d)', '\\1OCT', text, flags=re.I)
        text = re.sub('(?<=EXP)[OQ](?=\\d(?:' + month + '))', '0', text, flags=re.I)
        for match in re.finditer(f'(?<!\\d)(20\\d{{2}})\\s*[./-]?\\s*({month})\\s*[./-]?\\s*(\\d{{1,2}})(?!\\d)', text, re.IGNORECASE):
            parsed = _valid_date(int(match[1]), MONTHS[match[2].upper()], int(match[3]))
            if parsed:
                yield ParsedDate(parsed, match[0], match.start(), match.end(), 4, 'month-name', repaired)
        for match in re.finditer(f'(?:(?<!\\w)|(?<=EXP))(\\d{{1,2}})\\s*[./,\\-]?\\s*({month})\\s*[./,\\-]?\\s*(\\d{{2}}|20\\d{{2}})(?!\\d)', text, re.IGNORECASE):
            parsed = _valid_date(int(match.group(3)), MONTHS[match.group(2).upper()], int(match.group(1)))
            if parsed:
                yield ParsedDate(parsed, match.group(0), match.start(), match.end(), len(match.group(3)), 'month-name', repaired or match[0] != original[match.start():match.end()], 'dmy')
        for match in re.finditer(f'(?<!\\w)({month})\\s*[./,\\-]?\\s*(\\d{{1,2}})(?:\\s*[./,\\-]\\s*|\\s+)(\\d{{2}}|20\\d{{2}})(?!\\d)', text, re.IGNORECASE):
            parsed = _valid_date(int(match.group(3)), MONTHS[match.group(1).upper()], int(match.group(2)))
            if parsed:
                yield ParsedDate(parsed, match.group(0), match.start(), match.end(), len(match.group(3)), 'month-name', repaired, 'mdy')
    
    def parse_dates(text: str) -> list[ParsedDate]:
        return list(_cached_date_parses(text))
    
    @lru_cache(maxsize=4096)
    def _cached_date_parses(text: str) -> tuple[ParsedDate, ...]:
        return tuple(_parse_dates_uncached(text))
    
    def _parse_dates_uncached(text: str) -> list[ParsedDate]:
        text = _normalise_text(text)
        original = _date_text(text)
        variants = [(original, False)]
        repaired = _repaired_text(original)
        if repaired != original:
            variants.append((repaired, True))
        best: dict[tuple[date, int, int, str], ParsedDate] = {}
        for (value, was_repaired) in variants:
            matches = list(_iter_numeric_dates(value, was_repaired))
            matches.extend(_iter_month_name_dates(value, was_repaired))
            for item in matches:
                key = (item.value, item.start, item.end, item.order)
                current = best.get(key)
                if current is None or (current.repaired and (not item.repaired)):
                    best[key] = item
        groups: dict[tuple[int, int], list[ParsedDate]] = {}
        for item in best.values():
            groups.setdefault((item.start, item.end), []).append(item)
        groups = {span: values for (span, values) in groups.items() if not any((other != span and other[0] <= span[0] and (span[1] <= other[1]) for other in groups))}
        complete = [span for (span, values) in groups.items() if any((v.year_digits == 4 for v in values))]
        groups = {span: values for (span, values) in groups.items() if span in complete or not any((max(span[0], full[0]) < min(span[1], full[1]) for full in complete))}
        hints = _format_hints(text)
        hints.extend(((m.start(), m.end(), 'ymd') for m in re.finditer('\\d{2,4}\\s*년\\s*\\d{1,2}\\s*월\\s*\\d{1,2}\\s*일', text)))
        selected = []
        for (span, values) in groups.items():
            linked = {order for (start, end, order) in hints if _span_gap(span, (start, end)) <= 64 and _span_gap(span, (start, end)) == min((_span_gap(s, (start, end)) for s in groups))}
            if len(linked) > 1:
                continue
            if linked:
                selected.extend((replace(v, order_reason='explicit_order') for v in values if v.order in linked))
            else:
                selected.extend(values)
        return sorted(selected, key=lambda item: (item.start, item.end, item.value, item.order))
    
    def _union_box(lines: Sequence[OCRLine]) -> tuple[float, float, float, float]:
        return (min((line.box[0] for line in lines)), min((line.box[1] for line in lines)), max((line.box[2] for line in lines)), max((line.box[3] for line in lines)))
    
    def merge_horizontal_lines(lines: Sequence[OCRLine]) -> list[OCRLine]:
        """Create short same-row windows so split strings such as ``2026.`` + ``05.29`` are parsed."""
        indexed = [replace(line, members=(index,)) for (index, line) in enumerate(lines)]
        ordered = sorted(indexed, key=lambda line: (line.center[1], line.box[0]))
        merged: list[OCRLine] = _split_year_month_day_windows(indexed)
        for (start, first) in enumerate(ordered):
            group = [first]
            for other in ordered[start + 1:]:
                if (first.source, first.variant) != (other.source, other.variant):
                    continue
                previous = group[-1]
                height = max(previous.height, other.height)
                if abs(other.center[1] - first.center[1]) > 0.8 * max(first.height, other.height):
                    if other.box[1] > first.box[3] + height:
                        break
                    continue
                gap = other.box[0] - previous.box[2]
                if gap < -0.35 * height or gap > 6.0 * height:
                    continue
                group.append(other)
                text = ' '.join((item.text for item in group))
                if len(text) <= 96 and len(group) <= 4:
                    digit_count = sum((character.isdigit() for character in text))
                    if digit_count >= 4:
                        merged.append(OCRLine(text=text, score=min((item.score for item in group)), box=_union_box(group), source=first.source, variant=first.variant, members=tuple((member for item in group for member in item.members))))
                if len(group) == 4:
                    break
        return merged
    
    def _split_year_month_day_windows(lines: Sequence[OCRLine]) -> list[OCRLine]:
        """Join a printed YYYY.MM and DD only in one unambiguous local row.
    
        Sorting all OCR boxes by centre can interleave a nearby logo or another
        row and hide this pair from the generic window builder. No digits are
        supplied or corrected here; retain the weakest component's score.
        """
        result = []
        for first in lines:
            if not first.geometry_valid or first.score < 0.65 or (not re.fullmatch('\\s*20\\d{2}\\s*[./-]\\s*\\d{1,2}\\s*', first.text)):
                continue
            tails = []
            for other in lines:
                if other is first or not other.geometry_valid or other.score < 0.65 or ((first.source, first.variant) != (other.source, other.variant)) or (not re.fullmatch('\\s*\\d{1,2}\\s*(?:일|까지|일\\s*까지)?\\s*', other.text)):
                    continue
                h = max(first.height, other.height)
                gap = other.box[0] - first.box[2]
                if -0.15 * h <= gap <= 1.5 * h and other.center[0] > first.center[0] and (abs(other.center[1] - first.center[1]) <= 0.8 * h):
                    tails.append(other)
            if len(tails) != 1:
                continue
            last = tails[0]
            text = first.text.strip() + ' ' + last.text.strip()
            if not parse_dates(text):
                continue
            result.append(OCRLine(text, min(first.score, last.score), _union_box((first, last)), first.source, first.variant, members=first.members + last.members))
        return result
    
    def _box_gap(left: OCRLine, right: OCRLine) -> tuple[float, float]:
        horizontal = max(0.0, max(left.box[0], right.box[0]) - min(left.box[2], right.box[2]))
        vertical = max(0.0, max(left.box[1], right.box[1]) - min(left.box[3], right.box[3]))
        return (horizontal, vertical)
    
    def _role_text(text: str) -> str:
        """Mask facility/origin prose without moving token offsets."""
        return re.sub('포장\\s*재질|포장재|[내외]\\s*포장|포장\\s*[-:]\\s*폴리|(?:제조|생산|포장)\\s*(?:시설|공장|업소|업자|회사|사|의뢰자|자|번호|원|국가?|방법|방식)|(?:제조|생산|포장)\\s*(?:하고|합니다|한\\s*제품|되는|하였)', lambda match: ' ' * len(match.group()), text)
    
    def _date_fragment(text: str) -> bool:
        return len(text) <= 64 and sum((c.isdigit() for c in text)) >= 2 and any((separator in text for separator in './-')) and (not re.search('전화|고객|영양|%|\\b(?:TEL|LOT|kcal|mg|ml)\\b|\\b(?:AT|ART)[.-]?\\s*NR\\b', text, re.I))
    
    def _inline_role(text: str) -> str | None:
        text = _role_text(_normalise_text(text))
        if re.fullmatch('ED\\s*[:.]?', text, re.I):
            return 'end'
        if re.fullmatch('PD\\s*[:.]?', text, re.I):
            return 'start'
        positive = bool(POSITIVE_CONTEXT.search(text))
        negative = bool(NEGATIVE_CONTEXT.search(text))
        if positive != negative:
            return 'end' if positive else 'start'
        prefix = re.match('^([가-힣]{3,4})\\s*[:：]\\s*\\S', text)
        if not positive and (not negative) and prefix:
            matches = {role for (target, role) in (('제조일자', 'start'), ('소비기한', 'end'), ('유통기한', 'end')) if len(prefix[1]) == len(target) and sum((a != b for (a, b) in zip(prefix[1], target))) == 1}
            if len(matches) == 1:
                return matches.pop()
        return None
    
    def _same_region(first: OCRLine, other: OCRLine) -> bool:
        if first.original_box is None or other.original_box is None:
            return False
        (a, b) = (first.original_box, other.original_box)
        (area_a, area_b) = ((a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1]))
        intersection = max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
        union = area_a + area_b - intersection
        overlap = union > 0 and intersection / union >= 0.85
        clipped = area_a > 0 and area_b > 0 and (0.25 <= area_a / area_b <= 1.0) and (intersection / area_a >= 0.95) and (0.5 <= (a[3] - a[1]) / max(1.0, b[3] - b[1]) <= 1.5)
        return overlap or clipped
    
    def _link_roles(lines: Sequence[OCRLine]) -> list[OCRLine]:
        """Retain grounded roles even when a date has not parsed; never rewrite text."""
        normalized = [replace(line, text=_normalise_text(line.text)) for line in lines]
        parsed = [parse_dates(line.text) for line in normalized]
        paired = _printed_pair_roles(normalized, parsed)
        direct = {i: _inline_role(line.text) or (line.role if line.role_basis == 'recovery_anchor' else None) for (i, line) in enumerate(normalized)}
        roles = {i: role for (i, role) in direct.items() if role}
        roles.update(paired)
        references = set()
        manufacturing_reference = False
        for label in normalized:
            if not re.fullmatch('제조\\s*(?:연월일|일자|일)', label.text) or label.score < 0.9:
                continue
            for hint in normalized:
                if not re.fullmatch('별도\\s*표[기시]', hint.text) or hint.score < 0.9 or (hint.source, hint.variant) != (label.source, label.variant) or (not label.geometry_valid) or (not hint.geometry_valid):
                    continue
                (dx, dy) = _box_gap(label, hint)
                scale = max(label.height, hint.height)
                if dx <= 2 * scale and abs(label.center[1] - hint.center[1]) <= 0.5 * scale:
                    manufacturing_reference = True
        values = {p.value for tokens in parsed for p in tokens}
        if manufacturing_reference and len(values) == 1 and (not any((POSITIVE_CONTEXT.search(l.text) for l in normalized))):
            roles.update({i: 'start' for (i, tokens) in enumerate(parsed) if tokens})
        for label in normalized:
            reference = re.fullmatch('(?:소비\\s*기한|유통\\s*기한)\\s*[:：]?\\s*(상단|하단)\\s*표[기시]일?까지', label.text)
            if not reference or label.score < 0.9 or (not label.geometry_valid):
                continue
            targets = [i for (i, line) in enumerate(normalized) if line.geometry_valid and line.score >= 0.85 and parsed[i] and (len({(p.start, p.end) for p in parsed[i]}) == 1) and ((line.source, line.variant) == (label.source, label.variant)) and (i not in roles) and (line.box[3] < label.box[1] if reference[1] == '상단' else line.box[1] > label.box[3])]
            if len(targets) == 1:
                roles[targets[0]] = 'end'
                references.add(targets[0])
        for (i, line) in enumerate(normalized):
            if i in roles or not (parsed[i] or _date_fragment(line.text)):
                continue
            proposals = set()
            for (j, label) in enumerate(normalized):
                if not direct[j] or parsed[j] or list(_partial_dates(label.text)) or _date_fragment(label.text) or (len(label.text) > 32) or (label.score < 0.8):
                    continue
                same_frame = (line.source, line.variant) == (label.source, label.variant)
                if not same_frame and (not (line.original_box and label.original_box)):
                    continue
                if not same_frame and len({(p.start, p.end) for p in parsed[i]}) > 1:
                    continue
                anchor = line if same_frame else replace(line, box=line.original_box)
                hint = label if same_frame else replace(label, box=label.original_box)
                if label.text.strip() == '부터' and (anchor.height > 1.2 * anchor.width) != (hint.height > 1.2 * hint.width):
                    continue
                scale = max(anchor.height, hint.height)
                (dx, dy) = _box_gap(anchor, hint)
                same_row = dx <= 6 * scale and dy <= scale and (abs(anchor.center[1] - hint.center[1]) <= 0.5 * scale)
                overlap_x = max(0.0, min(anchor.box[2], hint.box[2]) - max(anchor.box[0], hint.box[0]))
                same_column = dy <= 1.5 * scale and overlap_x >= 0.7 * min(anchor.width, hint.width) and (hint.width <= 1.25 * anchor.width) and (abs(anchor.center[0] - hint.center[0]) <= 0.3 * anchor.width) and bool(parsed[i] or list(_partial_dates(line.text)))
                if not (same_row or same_column):
                    continue
    
                def rival_distance(other):
                    other = other if same_frame else replace(other, box=other.original_box)
                    (ox, oy) = _box_gap(other, hint)
                    other_scale = max(other.height, hint.height)
                    inline = ox <= 6 * other_scale and oy <= other_scale and (abs(other.center[1] - hint.center[1]) <= 0.5 * other_scale)
                    return (0 if inline else 1, math.hypot(ox, oy))
                rivals = [other for (k, other) in enumerate(normalized) if k != i and (parsed[k] or _date_fragment(other.text)) and ((other.source, other.variant) == (line.source, line.variant)) and (not (_same_region(line, other) or _same_region(other, line))) and (same_frame or other.original_box) and (rival_distance(other) <= (0 if same_row else 1, math.hypot(dx, dy)))]
                if not rivals:
                    proposals.add(direct[j])
            if len(proposals) == 1:
                roles[i] = proposals.pop()
        result = []
        for (i, line) in enumerate(normalized):
            inherited = set()
            for (j, other) in enumerate(normalized):
                if j not in roles or other.score < 0.7 or (line.source, line.variant) == (other.source, other.variant):
                    continue
                if not _same_region(line, other):
                    continue
                rivals = [candidate for (k, candidate) in enumerate(normalized) if k != i and (candidate.source, candidate.variant) == (line.source, line.variant) and _same_region(candidate, other)]
                if not rivals:
                    inherited.add(roles[j])
            if i in roles:
                inherited.add(roles[i])
            role = next(iter(inherited)) if len(inherited) == 1 else 'conflict' if inherited else None
            result.append(replace(line, role=role, role_basis='printed_reference' if i in references else 'local' if i in roles else 'same_region' if inherited else None))
        return result
    
    def _nearby_context(line: OCRLine, originals: Sequence[OCRLine]) -> tuple[list[str], list[str], float]:
        positives: list[str] = []
        negatives: list[str] = []
        positive_adjustment = 0.0
        negative_adjustment = 0.0
        member_set = set(line.members)
        for (index, other) in enumerate(originals):
            if index in member_set or (line.source, line.variant) != (other.source, other.variant):
                continue
            if other.score < 0.7:
                continue
            positive = POSITIVE_CONTEXT.search(other.text)
            negative = NEGATIVE_CONTEXT.search(_role_text(other.text))
            if not positive and (not negative):
                continue
            (horizontal, vertical) = _box_gap(line, other)
            scale = max(12.0, line.height, other.height)
            same_row = abs(line.center[1] - other.center[1]) <= 0.9 * scale and horizontal <= 9.0 * scale
            close = math.hypot(horizontal, vertical) <= 5.0 * scale
            if not (same_row or close):
                continue
            if positive:
                token = positive.group(0)
                positives.append(token)
                if same_row:
                    positive_adjustment = max(positive_adjustment, 1.35 if UNTIL_CONTEXT.search(token) else 0.95)
                elif close:
                    positive_adjustment = max(positive_adjustment, 0.3)
            if negative:
                token = negative.group(0)
                negatives.append(token)
                if same_row:
                    negative_adjustment = max(negative_adjustment, 0.75 if FROM_CONTEXT.search(token) else 0.55)
                elif close:
                    negative_adjustment = max(negative_adjustment, 0.12)
        return (positives, negatives, positive_adjustment - negative_adjustment)
    
    def _candidate_from_match(match: ParsedDate, line: OCRLine, originals: Sequence[OCRLine], role: str | None=None) -> DateCandidate:
        before = line.text[max(0, match.start - 24):match.start]
        after = line.text[match.end:match.end + 24]
        (before, after) = (_role_text(before), _role_text(after))
        local = _role_text(f'{before} {match.raw} {after}')
        role = role or line.role
        if role is not None:
            local = f"{match.raw} {('까지' if role == 'end' else '부터')}"
            (before, after) = ('', '까지' if role == 'end' else '부터')
        positive_hits = [item.group(0) for item in POSITIVE_CONTEXT.finditer(local)]
        negative_hits = [item.group(0) for item in NEGATIVE_CONTEXT.finditer(local)]
        score = 1.45 * max(0.0, min(1.0, line.score))
        score += 0.45 if match.year_digits == 4 else 0.15
        if match.separator in {'compact', 'joined'}:
            score -= 0.18
        elif match.separator == 'month-name':
            score += 0.15
        if match.repaired:
            score -= 0.55
        if positive_hits:
            score += 1.45
        if negative_hits:
            score -= 1.9
        if UNTIL_CONTEXT.search(after[:12]):
            score += 1.1
        if FROM_CONTEXT.search(after[:12]) or FROM_CONTEXT.search(before[-12:]):
            score -= 1.2
        (nearby_positive, nearby_negative, nearby_adjustment) = _nearby_context(line, originals) if role is None else ([], [], 0.0)
        positive_hits.extend(nearby_positive)
        negative_hits.extend(nearby_negative)
        score += nearby_adjustment
        digits = re.sub('\\D', '', line.text)
        if len(digits) >= 11 and match.separator == 'compact':
            score -= 1.25
        if re.search('(?:TEL|전화|고객\\s*센터|LOT\\s*NO)', line.text, re.IGNORECASE):
            score -= 0.45
        return DateCandidate(value=match.value, raw=match.raw, score=score, ocr_score=line.date_digit_score if line.date_digit_score is not None else line.score, year_digits=match.year_digits, repaired=match.repaired, separator=match.separator, source=line.source, variant=line.variant, positive_hits=tuple(dict.fromkeys(positive_hits)), negative_hits=tuple(dict.fromkeys(negative_hits)), box=line.box, members=line.members, explicit_positive=bool(POSITIVE_CONTEXT.search(local)), explicit_negative=bool(NEGATIVE_CONTEXT.search(local)), span=(match.start, match.end), order=match.order, order_reason=match.order_reason, repair_kind='format-only' if match.repaired and match.separator in {'repaired-spacing', 'month-name'} and (re.sub('\\D', '', re.sub('\\b0CT\\b', 'OCT', line.text[match.start:match.end], flags=re.I)) == re.sub('\\D', '', match.raw)) else 'inferred-digits' if match.repaired else None)
    
    def _printed_pair_roles(originals: Sequence[OCRLine], parsed: Sequence[list[ParsedDate]]) -> dict[int, str]:
        """Align two standalone role labels to two date rows as a block.
    
        Matching vertical order tolerates perspective skew without assigning both
        labels to the same nearest date. Competing dates/labels invalidate the block.
        """
        date_rows = [(i, line) for (i, (line, dates)) in enumerate(zip(originals, parsed)) if dates and len({(d.start, d.end) for d in dates}) == 1 or (not dates and _date_fragment(line.text))]
        proposals: dict[int, set[str]] = {}
        for (pos, (i, first)) in enumerate(date_rows):
            for (j, second) in date_rows[pos + 1:]:
                if (first.source, first.variant) != (second.source, second.variant):
                    continue
                scale = max(first.height, second.height)
                (dx, dy) = _box_gap(first, second)
                if dx > scale or dy > 2 * scale or abs(first.center[1] - second.center[1]) < 0.3 * scale:
                    continue
                block = OCRLine('', 1.0, _union_box((first, second)), first.source, first.variant)
                if any((k not in (i, j) and (other.source, other.variant) == (first.source, first.variant) and (_box_gap(block, other)[0] <= scale) and (_box_gap(block, other)[1] < scale) for (k, other) in date_rows)):
                    continue
                labels = []
                for label in originals:
                    if (label.source, label.variant) != (first.source, first.variant) or label.score < 0.85 or math.hypot(*_box_gap(block, label)) > 9 * scale:
                        continue
                    if re.fullmatch('부터|제조(?:일자|일)?|생산(?:일자|일)?|MFG|MFD', label.text, re.I):
                        labels.append(('start', label))
                    elif re.fullmatch('까지|EXP(?:IRY|IRES|DATE)?|USE\\s*BY|BEST\\s*(?:BEFORE|BY)', label.text, re.I):
                        labels.append(('end', label))
                if len(labels) != 2 or {kind for (kind, _) in labels} != {'start', 'end'}:
                    continue
                rows = sorted(((i, first), (j, second)), key=lambda item: item[1].center[1])
                labels.sort(key=lambda item: item[1].center[1])
                row_step = rows[1][1].center[1] - rows[0][1].center[1]
                label_step = labels[1][1].center[1] - labels[0][1].center[1]
                if not 0.35 * row_step <= label_step <= 3 * row_step:
                    continue
                if any((abs(row.center[1] - label.center[1]) > 2 * scale for ((_, row), (_, label)) in zip(rows, labels))):
                    continue
                if any((kind == 'start' and UNTIL_CONTEXT.search(row.text) or (kind == 'end' and FROM_CONTEXT.search(row.text)) for ((_, row), (kind, _)) in zip(rows, labels))):
                    continue
                for ((index, _), (kind, _)) in zip(rows, labels):
                    proposals.setdefault(index, set()).add(kind)
        return {i: next(iter(kinds)) for (i, kinds) in proposals.items() if len(kinds) == 1}
    
    def _paired_orders(originals: Sequence[OCRLine], parsed: Sequence[list[ParsedDate]], roles: dict[int, str]) -> dict[int, str]:
        """Use chronology only for a close, explicitly labelled, same-format pair.
    
        Identical field widths/separators on one printed block are the bounded
        same-order assumption. Across OCR frames, both explicit roles and original
        image coordinates are required. Unlabelled pairs, repaired digits,
        conflicting pairs and multiple possible orders supply no order evidence.
        """
        groups = []
        for (index, (line, dates)) in enumerate(zip(originals, parsed)):
            if line.role == 'conflict' or (line.date_digit_score if line.date_digit_score is not None else line.score) < 0.85 or (not dates) or any((d.repaired for d in dates)):
                continue
            if len({(d.start, d.end) for d in dates}) != 1:
                continue
            raw = dates[0].raw
            if not re.fullmatch('\\d{2,4}([./-])\\d{2}\\1\\d{2,4}', raw):
                continue
            signature = re.sub('\\d', '#', raw)
            candidate = _candidate_from_match(dates[0], line, originals, roles.get(index))
            groups.append((index, line, dates, signature, candidate))
        proposed: dict[int, set[str]] = {}
        for (index, line, dates, signature, candidate) in groups:
            for (other_index, other, other_dates, other_signature, other_candidate) in groups:
                if index >= other_index or signature != other_signature:
                    continue
                (first_geometry, second_geometry) = (line, other)
                if (line.source, line.variant) != (other.source, other.variant):
                    if not line.geometry_valid or not other.geometry_valid or line.original_box is None or (other.original_box is None) or ({line.role, other.role} != {'start', 'end'}):
                        continue
                    first_geometry = replace(line, box=line.original_box)
                    second_geometry = replace(other, box=other.original_box)
                    if _same_region(line, other) or _same_region(other, line):
                        continue
                (horizontal, vertical) = _box_gap(first_geometry, second_geometry)
                scale = max(first_geometry.height, second_geometry.height)
                if horizontal > scale or vertical > 3 * scale:
                    continue
                start = candidate.explicit_negative
                end = other_candidate.explicit_positive
                reverse_start = other_candidate.explicit_negative
                reverse_end = candidate.explicit_positive
                if start and end and (not reverse_start) and (not reverse_end):
                    (first, last) = (dates, other_dates)
                elif reverse_start and reverse_end and (not start) and (not end):
                    (first, last) = (other_dates, dates)
                else:
                    continue
                orders = {a.order for a in first for b in last if a.order == b.order and a.value < b.value}
                if len(orders) == 1:
                    proposed.setdefault(index, set()).update(orders)
                    proposed.setdefault(other_index, set()).update(orders)
        return {index: next(iter(orders)) for (index, orders) in proposed.items() if len(orders) == 1}
    
    def _resolve_orders(line: OCRLine, matches: list[ParsedDate], originals: Sequence[OCRLine], parsed_originals: Sequence[list[ParsedDate]], product_rules: Sequence[ProductDateRule], paired_orders: dict[int, str], context: DateContext | None=None) -> list[ParsedDate]:
        """Resolve a printed token before OCR scores rank different printed dates."""
        if not matches:
            return []
        groups: dict[tuple[int, int], list[ParsedDate]] = {}
        for match in matches:
            groups.setdefault((match.start, match.end), []).append(match)
        nearby = set()
        for (hint_line, hint_dates) in zip(originals, parsed_originals):
            if hint_dates:
                continue
            hints = _format_hints(hint_line.text)
            if not hints or hint_line.score < 0.65:
                continue
            same_frame = (hint_line.source, hint_line.variant) == (line.source, line.variant)
            if not same_frame:
                if line.geometry_valid and hint_line.geometry_valid and line.original_box and hint_line.original_box:
                    target = replace(line, box=line.original_box)
                    legend = replace(hint_line, box=hint_line.original_box)
                    gap = math.hypot(*_box_gap(target, legend))
                    rivals = [replace(other, box=other.original_box) for (other, dates) in zip(originals, parsed_originals) if dates and other.geometry_valid and other.original_box and (not (_same_region(line, other) or _same_region(other, line)))]
                    if gap <= 3 * max(target.height, legend.height) and (not any((math.hypot(*_box_gap(other, legend)) <= gap for other in rivals))):
                        nearby.update((order for (_, _, order) in hints))
                        continue
                reference = POSITIVE_CONTEXT.search(hint_line.text) and re.search('별도\\s*표[기시]|(?:후면|뒷면|상단|뚜껑(?:\\s*옆면)?)\\s*표[기시]', hint_line.text)
                if not reference or not line.original_box or (not hint_line.original_box):
                    continue
                (x1, y1, x2, y2) = line.original_box
    
                def same_region(other):
                    if not other.original_box:
                        return False
                    (a, b, c, d) = other.original_box
                    intersection = max(0, min(x2, c) - max(x1, a)) * max(0, min(y2, d) - max(y1, b))
                    return intersection / max(1, min((x2 - x1) * (y2 - y1), (c - a) * (d - b))) >= 0.7
                if any((dates and (not same_region(other)) for (other, dates) in zip(originals, parsed_originals))):
                    continue
                nearby.update((order for (_, _, order) in hints))
                continue
            gap = math.hypot(*_box_gap(line, hint_line))
            rivals = [other for (other, dates) in zip(originals, parsed_originals) if dates and (other.source, other.variant) == (line.source, line.variant) and (not set(other.members) & set(line.members))]
            explicit_reference = not rivals and len(groups) == 1 and POSITIVE_CONTEXT.search(hint_line.text) and re.search('별도\\s*표[기시]|(?:후면|뒷면|상단)\\s*표[기시]|뚜껑(?:\\s*옆면)?\\s*표[기시]', hint_line.text)
            if gap > 3 * max(line.height, hint_line.height) and (not explicit_reference):
                continue
            if any((math.hypot(*_box_gap(other, hint_line)) <= gap for other in rivals)):
                continue
            nearby.update((order for (_, _, order) in hints))
        frame_text = ' '.join((other.text.casefold() for other in originals if other.geometry_valid and other.score >= 0.8 and ((other.source, other.variant) == (line.source, line.variant) and math.hypot(*_box_gap(line, other)) <= 8 * max(line.height, other.height) or (line.original_box and other.original_box and (math.hypot(*_box_gap(replace(line, box=line.original_box), replace(other, box=other.original_box))) <= 8 * max(line.original_box[3] - line.original_box[1], other.original_box[3] - other.original_box[1]))))))
        matching_rules = [rule for rule in product_rules if all((token.casefold() in frame_text for token in rule.required_text))]
        result = []
        for values in groups.values():
            explicit = {v.order for v in values if v.order_reason == 'explicit_order'}
            if explicit:
                result.extend(values)
                continue
            if nearby:
                if len(nearby) == 1:
                    result.extend((replace(v, order_reason='explicit_order') for v in values if v.order in nearby))
                continue
            if len({v.value for v in values}) == 1 and (context is None or values[0].year_digits == 4):
                result.append(values[0])
                continue
            counterparts = {v.value for (other, dates) in zip(originals, parsed_originals) if (other.source, other.variant) == (line.source, line.variant) and (not set(other.members) & set(line.members)) and (math.hypot(*_box_gap(line, other)) <= 5 * max(line.height, other.height)) and POSITIVE_CONTEXT.search(other.text) and (not NEGATIVE_CONTEXT.search(other.text)) for v in dates if v.order_reason == 'explicit_order' or v.separator == 'month-name'}
            corresponding = [v for v in values if v.value in counterparts]
            if len({v.value for v in corresponding}) == 1:
                result.append(replace(corresponding[0], order_reason='corresponding_date'))
                continue
            pair = {paired_orders[index] for index in line.members if index in paired_orders}
            if len(pair) == 1:
                result.extend((replace(v, order_reason='labelled_date_pair') for v in values if v.order in pair))
                continue
            orders = {rule.order for rule in matching_rules}
            if orders:
                if len(orders) == 1:
                    result.extend((replace(v, order_reason='product_rule:' + matching_rules[0].name) for v in values if v.order in orders))
                continue
            if context is not None:
                (order, basis) = _market_order(line, originals, context)
                short = bool(re.fullmatch('\\d{2}(?:\\s*[./,:·-]\\s*|\\s+)\\d{2}(?:\\s*[./,:·-]\\s*|\\s+)\\d{2}', values[0].raw))
                if order and (short or (values[0].year_digits == 4 and order in {'dmy', 'mdy'})):
                    chosen = [v for v in values if v.order == order]
                    if chosen:
                        result.extend((replace(v, order_reason='market_policy:' + basis) for v in chosen))
                    else:
                        result.extend((replace(v, order_reason='review_invalid_market_date') for v in values))
                elif len({v.value for v in values}) == 1:
                    result.append(values[0])
                else:
                    result.extend((replace(v, order_reason='review_order:' + basis) for v in values))
                continue
            dmy = next((v for v in values if v.order == 'dmy'), None)
            if dmy:
                result.append(replace(dmy, order_reason='fallback_dmy'))
        return result
    
    def extract_candidates(lines: Sequence[OCRLine], *, product_rules: Sequence[ProductDateRule]=(), context: DateContext | None=None) -> list[DateCandidate]:
        originals = [replace(line, text=_normalise_text(line.text), members=line.members or (index,)) for (index, line) in enumerate(lines)]
        search_lines = originals + merge_horizontal_lines(originals)
        parser = _policy_parses if context is not None else parse_dates
        parsed_originals = [parser(line.text) for line in originals]
        roles = _printed_pair_roles(originals, parsed_originals)
        paired_orders = _paired_orders(originals, parsed_originals, roles)
        candidates: list[DateCandidate] = []
        for (index, line) in enumerate(search_lines):
            if index >= len(originals):
                borrowed = False
                for m in line.members:
                    label = originals[m]
                    role = _inline_role(label.text)
                    if not role or parsed_originals[m] or list(_partial_dates(label.text)):
                        continue
                    inside = [originals[k] for k in line.members if parsed_originals[k]]
                    if inside and any((k not in line.members and other.role == role and (other.role_basis == 'local') and ((other.source, other.variant) == (label.source, label.variant)) and parsed_originals[k] and (math.hypot(*_box_gap(other, label)) < min((math.hypot(*_box_gap(target, label)) for target in inside))) for (k, other) in enumerate(originals))):
                        borrowed = True
                        break
                if borrowed:
                    continue
            matches = parsed_originals[index] if index < len(originals) else parser(line.text)
            member_roles = {roles[member] for member in line.members if member in roles}
            role = next(iter(member_roles)) if len(member_roles) == 1 else None
            if line.role == 'conflict' or any((originals[m].role == 'conflict' for m in line.members if m < len(originals))):
                continue
            for match in _resolve_orders(line, matches, originals, parsed_originals, product_rules, paired_orders, context):
                candidate = _candidate_from_match(match, line, originals, role)
                if context is not None and match.separator == 'compact' and (match.year_digits == 4):
                    if not candidate.explicit_positive and (not candidate.explicit_negative):
                        continue
                if _OTHER_DATE.search(line.text):
                    continue
                other_labels = [hint for hint in originals if _OTHER_DATE.search(hint.text) and hint.score >= 0.8 and (not parse_dates(hint.text))]
    
                def local_other_date(hint):
                    target = line
                    if (hint.source, hint.variant) != (line.source, line.variant):
                        if not (hint.original_box and line.original_box and hint.geometry_valid and line.geometry_valid):
                            return False
                        target = replace(line, box=line.original_box)
                        hint = replace(hint, box=hint.original_box)
                    return math.hypot(*_box_gap(target, hint)) <= 1.5 * max(target.height, hint.height) and abs(target.center[0] - hint.center[0]) <= max(target.width, hint.width)
                if any((local_other_date(hint) for hint in other_labels)):
                    continue
                candidates.append(candidate)
        return candidates
    
    def _nearer_role(candidate: DateCandidate, other: DateCandidate, lines: Sequence[OCRLine], pattern: re.Pattern[str]) -> bool:
        """A split role token must be near this date and nearer than to its partner."""
        for line in lines:
            if (line.source, line.variant) != (candidate.source, candidate.variant) or not pattern.search(line.text):
                continue
            scale = max(12.0, candidate.box[3] - candidate.box[1], line.height)
            horizontal = max(0.0, max(candidate.box[0], line.box[0]) - min(candidate.box[2], line.box[2]))
            distance = abs((candidate.box[1] + candidate.box[3]) / 2 - line.center[1])
            other_distance = abs((other.box[1] + other.box[3]) / 2 - line.center[1])
            if horizontal <= 9 * scale and distance <= 1.5 * scale and (distance < other_distance):
                return True
        return False
    
    def _interval_endpoint(candidates: Sequence[DateCandidate], ranked: Sequence[DateCandidate], lines: Sequence[OCRLine]) -> DateCandidate | None:
        """Prefer a local, supported endpoint; unrelated dates never form an interval."""
        endpoints = set()
        ranked_scores = {item.iso: item.score for item in ranked}
        for first in candidates:
            for last in candidates:
                if first.order_reason.startswith('review_') or last.order_reason.startswith('review_'):
                    continue
                if (first.source, first.variant) != (last.source, last.variant):
                    continue
                if not 1 <= (last.value - first.value).days <= 550:
                    continue
                height = max(12.0, first.box[3] - first.box[1], last.box[3] - last.box[1])
                horizontal = max(0.0, max(first.box[0], last.box[0]) - min(first.box[2], last.box[2]))
                vertical = max(0.0, max(first.box[1], last.box[1]) - min(first.box[3], last.box[3]))
                if horizontal > 9 * height or vertical > 5 * height:
                    continue
                if set(first.members) & set(last.members) and first.members != last.members:
                    continue
                if first.members == last.members and max(first.span[0], last.span[0]) < min(first.span[1], last.span[1]):
                    continue
                paired_roles = (first.explicit_negative or _nearer_role(first, last, lines, FROM_CONTEXT)) and (not last.explicit_negative)
                explicit_end = (last.explicit_positive or _nearer_role(last, first, lines, UNTIL_CONTEXT)) and (not last.explicit_negative)
                equal_unlabelled = not first.positive_hits and (not first.negative_hits) and (not last.positive_hits) and (not last.negative_hits) and (abs(first.score - last.score) <= 0.15) and (min(first.ocr_score, last.ocr_score) >= 0.65)
                if (paired_roles or explicit_end or equal_unlabelled) and ranked_scores[last.iso] >= ranked[0].score - (2.6 if paired_roles or explicit_end else 0.15):
                    endpoints.add(last.iso)
        choices = [item for item in ranked if item.iso in endpoints]
        return max(choices, key=lambda item: (item.score, item.value), default=None)
    
    def _policy_pair_endpoint(candidates: Sequence[DateCandidate]) -> DateCandidate | None:
        """Organizer fallback on exactly two close printed tokens, after order resolution."""
        frames = {}
        if any((c.explicit_positive and (not c.explicit_negative) for c in candidates)):
            return None
        for candidate in candidates:
            if len(candidate.members) == 1:
                frames.setdefault((candidate.source, candidate.variant), {})[candidate.members, candidate.span] = candidate
        endpoints = []
        for frame in frames.values():
            values = list(frame.values())
            for (i, first) in enumerate(values):
                for last in values[i + 1:]:
                    if first.value > last.value:
                        (start, end) = (last, first)
                    else:
                        (start, end) = (first, last)
                    if start.value == end.value or min(start.ocr_score, end.ocr_score) < 0.85 or start.repaired or end.repaired or any((c.order_reason == 'fallback_dmy' and c.year_digits != 4 or c.order_reason.startswith('review_') or c.negative_hits for c in (start, end))) or any((not re.fullmatch('(?:\\d{2,4}([./-])\\d{2}\\1\\d{2,4}|\\d{2}\\s+\\d{2}\\s+\\d{4}|\\d{4}\\s+\\d{2}\\s+\\d{2})', c.raw) for c in (start, end))):
                        continue
                    if end.explicit_negative or (start.explicit_positive and (not start.explicit_negative)):
                        continue
                    if start.members == end.members and max(start.span[0], end.span[0]) < min(start.span[1], end.span[1]):
                        continue
                    a = OCRLine('', 1.0, start.box)
                    b = OCRLine('', 1.0, end.box)
                    (dx, dy) = _box_gap(a, b)
                    scale = max(a.height, b.height)
                    if dx > scale or dy > 3 * scale:
                        continue
                    block = OCRLine('', 1.0, _union_box((a, b)))
                    if any((other is not first and other is not last and (math.hypot(*_box_gap(block, OCRLine('', 1.0, other.box))) <= 2 * scale) for other in values)):
                        continue
                    endpoints.append(end)
        if len({candidate.iso for candidate in endpoints}) == 1:
            return max(endpoints, key=lambda candidate: candidate.score)
        return None
    
    def _select_full_date(lines: Sequence[OCRLine], *, final: bool=False, product_rules: Sequence[ProductDateRule]=(), context: DateContext | None=None) -> DateSelection:
        candidates = extract_candidates(lines, product_rules=product_rules, context=context)
        if _expiry_not_printed(lines, candidates):
            return DateSelection(None, float('-inf'), float('inf'), True, 'expiry-not-printed', tuple(candidates))
        readable_end = any((line.role == 'end' and (parse_dates(line.text) or list(_partial_dates(line.text))) for line in lines))
        pending_end = any((line.role == 'end' and line.score >= 0.7 and (_date_fragment(line.text) or (re.fullmatch('소비\\s*기한|유통\\s*기한|EXP|BBD|BEST\\s*BEFORE', line.text, re.I) and any((other.role == 'start' and parse_dates(other.text) and ((line.source, line.variant) == (other.source, other.variant)) and (math.hypot(*_box_gap(line, other)) <= 3 * max(line.height, other.height)) for other in lines)))) for line in lines))
        if pending_end and (not readable_end) and all((c.explicit_negative and (not c.explicit_positive) for c in candidates)):
            return DateSelection(None, float('-inf'), 0.0, False, 'unreadable-expiry', tuple(candidates))
        if not candidates:
            return DateSelection(None, float('-inf'), float('inf'), False, 'no-valid-date', ())
        grouped: dict[str, list[DateCandidate]] = {}
        for candidate in candidates:
            grouped.setdefault(candidate.iso, []).append(candidate)
        ranked: list[DateCandidate] = []
        for same_date in grouped.values():
            best = max(same_date, key=lambda item: item.score)
            if context is not None and best.ocr_score < 0.65:
                readable = [item for item in same_date if item.ocr_score >= 0.85 and (not item.repaired) and ((item.source, item.variant) == (best.source, best.variant)) and (set(item.members) <= set(best.members)) and (item.explicit_positive == best.explicit_positive) and (item.explicit_negative == best.explicit_negative)]
                if readable:
                    best = max(readable, key=lambda item: item.score)
            independent_passes = {(item.source, item.variant) for item in same_date}
            if context is not None:
                boxes = []
                for item in same_date:
                    members = [lines[i] for i in item.members if i < len(lines)]
                    if not members or any((line.original_box is None for line in members)):
                        boxes = []
                        break
                    box = (min((line.original_box[0] for line in members)), min((line.original_box[1] for line in members)), max((line.original_box[2] for line in members)), max((line.original_box[3] for line in members)))
                    boxes.append(box)
                if boxes:
                    regions = []
                    for a in sorted(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]), reverse=True):
                        if any((max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1])) >= 0.5 * min((a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1])) for b in regions)):
                            continue
                        regions.append(a)
                    independent_passes = set(range(len(regions)))
            agreement_bonus = min(0.6, 0.22 * (len(independent_passes) - 1))
            ranked.append(replace(best, score=best.score + agreement_bonus))
        ranked.sort(key=lambda item: (item.score, item.value), reverse=True)
        interval_preferred = False
        policy_endpoint = _policy_pair_endpoint(candidates)
        endpoint = next((item for item in ranked if policy_endpoint and item.iso == policy_endpoint.iso), None)
        if endpoint is None:
            endpoint = _interval_endpoint(candidates, ranked, lines)
        if endpoint is not None:
            best_score = ranked[0].score
            ranked.remove(endpoint)
            ranked.insert(0, replace(endpoint, score=max(endpoint.score, best_score + 0.05)))
            interval_preferred = True
        best = ranked[0]
        margin = best.score - ranked[1].score if len(ranked) > 1 else float('inf')
        has_positive = bool(best.positive_hits)
        has_negative = bool(best.negative_hits)
        if context is not None and best.ocr_score < 0.65:
            return DateSelection(None, best.score, margin, False, 'low-confidence-digits', tuple(ranked), digits_confident=False, order_resolved=False)
        if best.score < 1.05 and (not interval_preferred):
            explicit_manufacturing = best.explicit_negative and (not best.explicit_positive) and (best.ocr_score >= 0.7) and all((item.explicit_negative and (not item.explicit_positive) for item in ranked)) and (not any((POSITIVE_CONTEXT.search(line.text) and (not NEGATIVE_CONTEXT.search(line.text)) and (sum((char.isdigit() for char in line.text)) >= 2) for line in lines)))
            return DateSelection(None, best.score, margin, explicit_manufacturing, 'negative-context' if explicit_manufacturing else 'score-below-threshold', tuple(ranked), digits_confident=best.ocr_score >= 0.85 and (not best.repaired), order_resolved=best.order_reason != 'fallback_dmy')
        if best.order_reason.startswith('review_'):
            clear = best.ocr_score >= 0.85 and (not best.repaired) and (not has_negative)
            return DateSelection(None, best.score, margin, final, best.order_reason, tuple(ranked), digits_confident=clear, order_resolved=False)
        confident = has_positive and best.score >= 2.15 and (margin >= 0.35) and (not has_negative or interval_preferred or (len(ranked) == 1 and best.score >= 3.0)) or (len(ranked) == 1 and best.score >= 1.65 and (best.ocr_score >= 0.65) and (not best.repaired) and (not has_negative)) or (best.score >= 2.35 and margin >= 0.65 and (not has_negative or interval_preferred or (len(ranked) == 1 and best.score >= 3.0)))
        if final:
            confident = True
        if len(ranked) == 1 and best.ocr_score >= 0.85 and (not best.repaired) and (not has_negative) and (best.order_reason in {'explicit_order', 'corresponding_date', 'calendar_unique'} or best.order_reason.startswith(('product_rule:', 'market_policy:'))):
            confident = True
        reason = 'accepted' if confident else 'ambiguous'
        if best.order_reason == 'fallback_dmy':
            if not final:
                confident = (confident or len(ranked) == 1) and best.ocr_score >= 0.85 and (not best.repaired) and (not has_negative)
            reason = 'fallback_dmy' if confident else 'low-confidence-digits'
        if policy_endpoint is not None:
            reason = 'local-date-pair'
            singles = [c for c in candidates if len(c.members) == 1]
            covered = all((any((c.iso == s.iso and c.raw == s.raw and ((c.source, c.variant) == (s.source, s.variant)) and (set(s.members) <= set(c.members)) for s in singles)) for c in candidates))
            if best.ocr_score >= 0.95 and (not best.repaired) and singles and covered and all((c.ocr_score >= 0.95 and (not c.repaired) for c in singles)) and (not any((c.explicit_negative for c in candidates))) and (len({c.iso for c in candidates}) == 2):
                confident = True
        return DateSelection(best.iso, best.score, margin, confident, reason, tuple(ranked), digits_confident=best.ocr_score >= 0.85 and (not best.repaired), order_resolved=best.order_reason != 'fallback_dmy', role_resolved=policy_endpoint is not None or (interval_preferred and any((c.iso != best.iso and (c.explicit_negative or _nearer_role(c, best, lines, FROM_CONTEXT) or _nearer_role(best, c, lines, UNTIL_CONTEXT)) for c in candidates))))
    
    def _printed_month_day_token(text: str):
        """An explicit MM.DD-until token, optionally followed by a separate clock.
    
        Preserve the raw span for character evidence. A year/lot fragment or broken
        full date cannot be truncated into this shape.
        """
        match = re.fullmatch('\\s*(\\d{2})\\s*[./-]\\s*(\\d{2})\\s*까지\\s*(?:(?:[01]\\d|2[0-3]):[0-5]\\d)?\\s*', text)
        if match and _valid_date(2000, int(match[1]), int(match[2])):
            return match
        return None
    
    def _expiry_not_printed(lines, candidates):
        """A local explicit omission statement, never a missing-OCR assumption."""
        if not candidates or any((not c.explicit_negative or c.explicit_positive or c.ocr_score < 0.85 for c in candidates)) or any((line.role != 'start' and list(_partial_dates(line.text)) or (line.role == 'end' and _date_fragment(line.text)) for line in lines)):
            return False
        for note in lines:
            if note.score < 0.9 or not note.geometry_valid or (not re.search('(?:표기|표시)\\s*하지\\s*않', note.text)) or re.search('아니|않는\\s*경우|않으면|않을', note.text):
                continue
            frame = (note.source, note.variant)
            if not any(((c.source, c.variant) == frame for c in candidates)):
                continue
            labels = [line for line in lines if line.geometry_valid and line.score >= 0.9 and ((line.source, line.variant) == frame) and re.fullmatch('유통\\s*기한|소비\\s*기한|제조\\s*일자?', line.text)]
            nearby = []
            for label in labels:
                vertical = note.height > note.box[2] - note.box[0]
                if vertical != (label.height > label.box[2] - label.box[0]):
                    continue
                (dx, dy) = _box_gap(label, note)
                thickness = min(note.height, note.box[2] - note.box[0], label.height, label.box[2] - label.box[0])
                aligned = (abs(label.center[0] - note.center[0]) if vertical else abs(label.center[1] - note.center[1])) <= thickness
                if aligned and math.hypot(dx, dy) <= 3 * thickness:
                    nearby.append((math.hypot(dx, dy), label))
            nearby.sort(key=lambda item: item[0])
            if nearby and (len(nearby) == 1 or nearby[0][0] + 5 < nearby[1][0]) and re.fullmatch('유통\\s*기한|소비\\s*기한', nearby[0][1].text):
                return True
        return False
    
    def _partial_dates(text: str) -> Iterable[str]:
        """Parse missing fields without borrowing digits from a broken full date."""
        occupied = []
        for match in re.finditer('(?<![\\d./-])(20\\d{2})\\s*[년./-]\\s*(\\d{1,2})(?:월)?(?!월|\\d|\\s*[./-]\\s*\\d|\\s*\\d{1,2}\\s*일)', text):
            occupied.append((match.start(), match.end()))
            if MIN_YEAR <= int(match[1]) <= MAX_YEAR and 1 <= int(match[2]) <= 12:
                yield f'{int(match[1]):04d}-{int(match[2]):02d}-NONE'
        months = '|'.join(MONTHS)
        for match in re.finditer(f'(?<!\\w)({months})\\s*[./-]?\\s*(20\\d{{2}})(?!\\d)', text, re.IGNORECASE):
            if MIN_YEAR <= int(match[2]) <= MAX_YEAR:
                yield f'{int(match[2]):04d}-{MONTHS[match[1].upper()]:02d}-NONE'
        for match in re.finditer('(?<![\\d./-])(\\d{1,2})\\s*[./-]\\s*(20\\d{2})(?!\\d|\\s*[./-]\\s*\\d)', text):
            if re.search('\\d\\s*[./-]\\s*$', text[:match.start()]):
                continue
            if 1 <= int(match[1]) <= 12 and MIN_YEAR <= int(match[2]) <= MAX_YEAR:
                yield f'{int(match[2]):04d}-{int(match[1]):02d}-NONE'
        for match in re.finditer('(?<![\\d./-])(\\d{1,2})\\s*[월./-]\\s*(\\d{1,2})(?:일)?(?!\\d|\\s*[./-]\\s*\\d)', text):
            if re.search('\\d\\s*[년./-]\\s*$', text[:match.start()]) or any((start <= match.start() < end for (start, end) in occupied)):
                continue
            try:
                date(2000, int(match[1]), int(match[2]))
            except ValueError:
                continue
            yield f'NONE-{int(match[1]):02d}-{int(match[2]):02d}'
    
    def _printed_month_year(lines: Sequence[OCRLine]) -> DateSelection | None:
        """An explicit MM/YYYY legend constrains a printed token, not its digits."""
        legend = re.compile('(?<![A-Z])MM\\s*[/.-]\\s*YYYY(?![A-Z])|월\\s*[/.-]\\s*년\\s*순', re.I)
        values = set()
        for target in lines:
            if target.role == 'end' and target.score >= 0.9 and re.fullmatch('\\s*(?:EXP(?:IRY)?\\s*:?\\s*)?(?:\\d{1,2}[./-]20\\d{2}|20\\d{2}[./-]\\d{1,2})\\s*', target.text, re.I):
                if any((other.score >= 0.85 and other.role not in ('start', 'conflict') and (_same_region(target, other) or _same_region(other, target)) and any((not p.repaired and p.year_digits == 4 for p in _policy_parses(other.text))) for other in lines)):
                    continue
                values.update((v for v in _partial_dates(target.text) if v.endswith('-NONE')))
        for hint in lines:
            if hint.score < 0.75 or not legend.search(hint.text) or _format_hints(hint.text):
                continue
            targets = []
            for target in lines:
                if (target.source, target.variant) != (hint.source, hint.variant) or target.score < 0.85 or target.role in ('start', 'conflict'):
                    continue
                match = re.fullmatch('(\\d{2})\\s*[/.-]?\\s*(20\\d{2})', target.text)
                if not match or not 1 <= int(match[1]) <= 12 or int(match[2]) > MAX_YEAR:
                    continue
                targets.append((target, f'{match[2]}-{match[1]}-NONE'))
            if len(targets) != 1:
                continue
            (target, value) = targets[0]
            reference = re.search('별도\\s*표[기시]|표시|표기|SEE', hint.text, re.I)
            if not reference and math.hypot(*_box_gap(target, hint)) > 5 * max(target.height, hint.height):
                continue
            if any((other is not target and other.role == 'end' and parse_dates(other.text) and (not re.fullmatch('\\d{2}\\s*[/.-]?\\s*20\\d{2}', other.text)) for other in lines)):
                continue
            values.add(value)
        if len(values) == 1:
            return DateSelection(values.pop(), 3.0, float('inf'), True, 'printed-month-year', (), digits_confident=True, order_resolved=True)
        return None
    
    def _select_date_impl(lines: Sequence[OCRLine], *, final: bool=False, product_rules: Sequence[ProductDateRule]=(), context: DateContext | None=None) -> DateSelection:
        lines = _link_roles(lines)
        partial = _printed_month_year(lines)
        if partial is not None:
            return partial
        full = _select_full_date(lines, final=final, product_rules=product_rules, context=context)
        if full.reason.startswith('review_'):
            return full
        explicit_full = any((c.iso == full.final_date and c.explicit_positive and (not c.explicit_negative) for c in full.candidates))
        if full.final_date is not None and explicit_full:
            return full
        partials: dict[str, float] = {}
        clear_month_year_lines: dict[str, list[OCRLine]] = {}
        for (index, line) in enumerate(lines):
            text = _normalise_text(line.text)
            if context is not None:
                text = _mask_auxiliary(text)
            if parse_dates(text) or line.score < 0.65:
                continue
            positive = line.role == 'end' or bool(POSITIVE_CONTEXT.search(text))
            negative = line.role in ('start', 'conflict') or bool(NEGATIVE_CONTEXT.search(_role_text(text)))
            if negative or re.search('보관|온도|℃|kg|mg|g\\b|cm|%|TEL|전화|LOT\\s*NO|품목|인증|허가|등록|제\\s*\\d.*호', text, re.IGNORECASE):
                continue
            (_, near_negative, adjustment) = _nearby_context(replace(line, members=(index,)), lines)
            if near_negative and (not positive):
                continue
            for value in _partial_dates(text):
                score = 1.45 * line.score + (0.15 if value.startswith('NONE') else 0.3) + (1.45 if positive else 0) + adjustment
                if value.startswith('NONE') and re.fullmatch('\\s*\\d{1,2}[./-]\\d{1,2}\\s+\\d{1,2}:\\d{2}(?:\\s+[A-Z]{1,3})?\\s*', _normalise_text(line.text), re.I):
                    score += 0.25
                if score >= 1.35:
                    partials[value] = max(score, partials.get(value, -math.inf))
                    if value.endswith('-NONE') and line.score >= 0.9:
                        clear_month_year_lines.setdefault(value, []).append(line)
        if not partials:
            return full
        ordered = sorted(partials.items(), key=lambda item: item[1], reverse=True)
        (value, score) = ordered[0]
        if full.reason == 'unreadable-expiry' and (not any((line.role == 'end' and value in list(_partial_dates(line.text)) for line in lines))):
            return full
        if full.final_date is not None and (not any((line.role == 'end' and len(line.text) <= 32 and (line.score >= 0.85) and re.fullmatch('(?:소비\\s*기한|유통\\s*기한|EXP(?:IRY)?|BBD|BEST\\s*BEFORE(?:\\s*END)?)?\\s*[:：]?\\s*[\\d\\s./년월일-]+\\s*(?:까지)?', line.text, re.I) and (value in list(_partial_dates(line.text))) for line in lines))):
            return full
        margin = score - ordered[1][1] if len(ordered) > 1 else math.inf
        explicit_partial = False
        partial_legend = re.compile('월\\s*[.,/\\-]?\\s*년\\s*순|(?<!Y)MM\\s*[/.-]\\s*YYYY|BEST\\s*BEFORE\\s*END', re.I)
        if len(partials) == 1 and (not full.candidates):
            explicit_partial = any((_printed_month_day_token(line.text) and line.role == 'end' and (line.score >= 0.85) and (line.date_digit_score is not None) and (line.date_digit_score >= 0.9) and (line.date_digit_min_score is not None) and (line.date_digit_min_score >= 0.7) for line in lines))
            for target in clear_month_year_lines.get(value, []):
                for hint in lines:
                    if (target.source, target.variant) != (hint.source, hint.variant) or hint.score < 0.75 or (not partial_legend.search(hint.text)) or _format_hints(hint.text) or NEGATIVE_CONTEXT.search(hint.text):
                        continue
                    close = math.hypot(*_box_gap(target, hint)) <= 8 * max(target.height, hint.height)
                    reference = re.search('표시|표기|SEE', hint.text, re.I)
                    if close or reference:
                        explicit_partial = True
        return DateSelection(value, score, margin, final or explicit_partial, 'explicit-partial-date' if explicit_partial else 'partial-date', full.candidates, digits_confident=value in clear_month_year_lines, order_resolved=explicit_partial)
    
    def _join_role_headers(lines: Sequence[OCRLine]) -> list[OCRLine]:
        """Join only complete short printed role words in the same OCR frame."""
        joined = list(lines)
        endings = {'소비': ('기한',), '유통': ('기한',), '사용': ('기한',), '제조': ('일', '일자'), '생산': ('일', '일자'), '포장': ('일', '일자')}
        for (i, first) in enumerate(lines):
            suffixes = endings.get(first.text.strip())
            if not suffixes or first.score < 0.8 or (not first.geometry_valid):
                continue
            choices = []
            for (j, second) in enumerate(lines):
                if second.text.strip() not in suffixes or second.score < 0.8 or (not second.geometry_valid) or ((first.source, first.variant) != (second.source, second.variant)):
                    continue
                scale = max(first.height, second.height)
                (dx, dy) = _box_gap(first, second)
                horizontal = second.center[0] > first.center[0] and dx <= scale and (abs(first.center[1] - second.center[1]) <= 0.5 * scale)
                vertical = second.center[1] > first.center[1] and dy <= scale and (abs(first.center[0] - second.center[0]) <= 0.5 * max(first.width, second.width))
                if horizontal or vertical:
                    choices.append((j, second))
            if len(choices) != 1:
                continue
            (j, second) = choices[0]
            box = (min(first.box[0], second.box[0]), min(first.box[1], second.box[1]), max(first.box[2], second.box[2]), max(first.box[3], second.box[3]))
            original = None
            if first.original_box and second.original_box:
                (a, b) = (first.original_box, second.original_box)
                original = (min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3]))
            joined.append(replace(first, text=first.text.strip() + second.text.strip(), score=min(first.score, second.score), box=box, original_box=original, polygon=(), members=(i, j)))
        return joined
    
    def _separate_printed_lot_reference(lines):
        """Mask a lot prefix only when a local printed lot/date legend identifies it."""
        result = list(lines)
        for (i, line) in enumerate(lines):
            token = re.match('^\\s*\\d{4,}\\s*/\\s*(\\d{2,4}[.\\-/]\\d{1,2}[.\\-/]\\d{1,2})(?!\\d)', line.text)
            if not token or not line.geometry_valid or (not parse_dates(token[1])):
                continue
            for hint in lines:
                if hint.score < 0.8 or not hint.geometry_valid or (not _format_hints(hint.text)) or (not re.search('제조\\s*번호\\s*/\\s*(?:사용|소비|유통)\\s*기한', hint.text)):
                    continue
                (target, legend) = (line, hint)
                if (line.source, line.variant) != (hint.source, hint.variant):
                    if not line.original_box or not hint.original_box:
                        continue
                    (target, legend) = (replace(line, box=line.original_box), replace(hint, box=hint.original_box))
                if math.hypot(*_box_gap(target, legend)) > 3 * max(target.height, legend.height):
                    continue
                result[i] = replace(line, text=' ' * token.start(1) + line.text[token.start(1):])
                break
        return result
    
    def select_date(lines: Sequence[OCRLine], *, final: bool=False, product_rules: Sequence[ProductDateRule]=(), context: DateContext | None=None) -> DateSelection:
        if context is not None:
            lines = _separate_printed_lot_reference(lines)
        lines = _join_role_headers(lines)
        selected = _select_date_impl(lines, final=final, product_rules=product_rules, context=context)
        if context is None:
            return selected
        best = selected.candidates[0] if selected.candidates else None
        source = next((line for line in lines if best and (line.source, line.variant, line.box) == (best.source, best.variant, best.box)), None)
        raw = source.text if source else best.raw if best else None
        if selected.final_date and best:
            if best.separator == 'compact' and best.year_digits == 2 and (not best.explicit_positive) and (best.order_reason == 'calendar_unique'):
                selected = replace(selected, final_date=None, confident=False, reason='review_unanchored_compact', digits_confident=False, order_resolved=False)
            if source:
                text = _normalise_text(source.text)
                broken_year = any((m.start() <= best.span[0] and best.span[1] <= m.end() for m in re.finditer('(?<!\\d)20\\d\\s+\\d[./-]\\d{1,2}[./-]\\d{1,2}', text)))
                truncated_glyph = bool(re.match('[\\u1100-\\u11ff\\u3130-\\u318f?]', text[best.span[1]:]))
                if broken_year or truncated_glyph:
                    selected = replace(selected, final_date=None, confident=False, reason='review_incomplete_token', digits_confident=False, order_resolved=False)
        suffix = _normalise_text(raw)[best.span[1]:] if source and best else ''
        times = [m.group() for m in _CLOCK.finditer(suffix)]
        if not times and source:
            for other in lines:
                if other is source or not other.geometry_valid or other.score < 0.85 or ((other.source, other.variant) != (source.source, source.variant)) or (not _CLOCK.fullmatch(other.text.strip())):
                    continue
                (dx, dy) = _box_gap(source, other)
                if dx == 0 and dy <= 1.5 * max(source.height, other.height):
                    rivals = [candidate for candidate in lines if candidate is not source and candidate is not other and ((candidate.source, candidate.variant) == (source.source, source.variant)) and parse_dates(candidate.text) and (math.hypot(*_box_gap(candidate, other)) <= math.hypot(dx, dy))]
                    if not rivals:
                        times.append(other.text.strip())
        lots = re.findall('(?<![A-Za-z0-9])([A-Z][A-Z0-9]{0,2})(?![A-Za-z0-9])', _CLOCK.sub(' ', suffix))
        lots = [lot for lot in lots if lot not in {'EXP', 'MFG', 'MFD', 'BBD', 'BBE', 'LOT'}]
        basis = selected.reason if selected.reason.startswith('review_') else best.order_reason if best else selected.reason
        invalid = selected.final_date is None and (not selected.candidates) and any((_KOREAN_EXPIRY.search(line.text) and re.search('(?<!\\d)\\d{2,4}[./-]\\d{2}[./-]\\d{2}(?!\\d)', line.text) for line in lines))
        review = selected.reason.startswith('review_') or invalid
        if invalid:
            basis = 'invalid_calendar_date'
        market = context.market or ('KR' if basis == 'market_policy:korean_food_context' else None)
        manufactured = {c.iso for c in selected.candidates if c.explicit_negative and (not c.explicit_positive) and (not c.repaired) and (c.ocr_score >= 0.85) and (not c.order_reason.startswith(('review_', 'fallback_')))}
        details = dict(policy='market-context-20260911', status='REVIEW_REQUIRED' if review else 'SELECTED' if selected.final_date else 'NOT_FOUND', market=market, raw_text=raw, expiration_date=selected.final_date, manufactured_date=next(iter(manufactured)) if len(manufactured) == 1 else None, date_format=best.order.upper() if best and (not review) else None, auxiliary_time=times[0] if len(times) == 1 else None, lot_code=lots[0] if len(lots) == 1 else None, confidence_level='REVIEW' if review or not selected.digits_confident else 'MEDIUM' if basis.startswith('market_policy:') else 'HIGH', reason=[basis], confidence_is_calibrated=False)
        decision = submission_decision(selected, base_partial=selected.is_partial)
        details.update(candidate_date=decision.candidate_date, expiration_date=decision.output_date, status=decision.status, recovery_reason=decision.recovery_reason, confidence_level=details['confidence_level'] if decision.status == 'SELECTED' else 'REVIEW', scope='submission-evidence-v1')
        return replace(selected, policy_details=details)
    
    def submission_fields(final_date: str | None) -> dict[str, str]:
        return fields_from_date(final_date)
    
    def selection_fields(selection):
        return serialize_fields(selection.output_fields) if selection.output_fields is not None else submission_fields(selection.final_date)
    
    def field_evidence(selection):
        """Preserve individually supported fields without selecting an arbitrary date order."""
        decision = submission_decision(selection)
        if decision.output_date:
            return fields_from_date(decision.output_date)
        if selection.is_partial:
            return fields_from_date(selection.final_date)
        candidates = [c for c in selection.candidates if c.ocr_score >= 0.65 and (not c.explicit_negative) and (not c.repaired or c.repair_kind == 'format-only')]
        positives = [c for c in candidates if c.explicit_positive]
        if positives:
            candidates = positives
        origins = {(c.raw, tuple(c.box)) for c in candidates}
        if not candidates or len(origins) != 1 or selection.reason in {'negative-context', 'expiry-not-printed', 'review_unanchored_compact'}:
            return fields_from_date(None)
        if not selection.digits_confident and (not all((c.ocr_score >= 0.85 for c in candidates))):
            return fields_from_date(None)
        values = [fields_from_date(c.iso) for c in candidates]
        result = {name: values[0][name] if len({v[name] for v in values}) == 1 else 'NONE' for name in FIELDS}
        if result['year'] != 'NONE' and (not all((c.year_digits == 4 or c.explicit_positive for c in candidates))):
            result['year'] = 'NONE'
        return serialize_fields(result)
    return _ITDA_Namespace(**locals())
_ITDA_NS['date_extraction'] = _itda_define_date_extraction()
del _itda_define_date_extraction


In [ ]:
from __future__ import annotations
# Embedded from src/recognition_evidence.py; generated, do not edit separately.
def _itda_define_recognition_evidence():
    """Read-only CTC evidence adapter for the installed PaddleX recognizer.
    
    The original decoder remains authoritative. Unknown layouts/reordered strings
    fail closed to missing evidence, never guessed character alignment.
    """
    import numpy as np
    candidates = _ITDA_NS['ctc_candidates'].candidates
    
    class CTCEvidence:
    
        def __init__(self, decoder, collect_candidates=False):
            self.decoder = decoder
            self.rows = []
            self.collect_candidates = collect_candidates
            self.candidate_rows = []
    
        def __call__(self, pred, *args, **kwargs):
            output = self.decoder(pred, *args, **kwargs)
            texts = output[0]
            records = [() for _ in texts]
            alternatives = [[] for _ in texts]
            try:
                probabilities = np.asarray(pred[0])
                if probabilities.ndim != 3 or probabilities.shape[0] != len(texts):
                    raise ValueError('Unknown CTC shape')
                for (row, expected) in enumerate(texts):
                    indices = probabilities[row].argmax(axis=-1)
                    chosen = np.ones(len(indices), dtype=bool)
                    chosen[1:] = indices[1:] != indices[:-1]
                    for token in self.decoder.get_ignored_tokens():
                        chosen &= indices != token
                    positions = np.flatnonzero(chosen)
                    decoded = ''.join((self.decoder.character[indices[p]] for p in positions))
                    if decoded == expected and len(decoded) == len(positions):
                        records[row] = tuple((float(probabilities[row, p, indices[p]]) for p in positions))
                        if self.collect_candidates and records[row] and (min(records[row]) < 0.8) and (sum((c.isdigit() for c in decoded)) >= 4) and (len(decoded) <= 32) and (list(self.decoder.get_ignored_tokens()) == [0]):
                            alternatives[row] = candidates(probabilities[row], self.decoder.character)
            except (AttributeError, ValueError, IndexError, TypeError):
                pass
            self.rows.extend(zip(texts, records))
            self.candidate_rows.extend(alternatives)
            return output
    return _ITDA_Namespace(**locals())
_ITDA_NS['recognition_evidence'] = _itda_define_recognition_evidence()
del _itda_define_recognition_evidence


In [ ]:
from __future__ import annotations
# Embedded from src/line_recovery.py; generated, do not edit separately.
def _itda_define_line_recovery():
    """Bounded re-recognition of existing date boxes; no detector or digit rewriting.
    
    Agreement between views of one recognizer is a stability check, NOT independent
    evidence or calibrated confidence. Original and rejected observations are logged.
    """
    from collections import Counter
    from dataclasses import replace
    import math
    import re
    import time
    import cv2
    OCRLine = _ITDA_NS['date_extraction'].OCRLine
    _link_roles = _ITDA_NS['date_extraction']._link_roles
    _printed_month_day_token = _ITDA_NS['date_extraction']._printed_month_day_token
    parse_dates = _ITDA_NS['date_extraction'].parse_dates
    _inline_role = _ITDA_NS['date_extraction']._inline_role
    POSITIVE_CONTEXT = _ITDA_NS['date_extraction'].POSITIVE_CONTEXT
    NEGATIVE_CONTEXT = _ITDA_NS['date_extraction'].NEGATIVE_CONTEXT
    rectified_crop = _ITDA_NS['date_region_recovery'].rectified_crop
    recognition_views = _ITDA_NS['date_region_recovery'].recognition_views
    
    def recovery_targets(lines):
        targets = []
        for (index, line) in enumerate(lines):
            text = line.text.strip()
            digits = sum((c.isdigit() for c in text))
            if not line.geometry_valid or len(text) > 40 or digits < 3 or (digits / max(1, len(text)) < 0.25) or re.search('%|kcal|mg|ml|brix|영양|전화|고객|상담|수신|품목|인증|허가|등록|\\b(?:TEL|LOT)\\b|(?<!\\d)0\\d{1,2}[- )]\\d{2,4}-\\d{3,4}', text, re.I) or re.fullmatch('\\d{1,2}:\\d{2}(?::\\d{2})?', text):
                continue
            if not (parse_dates(text) or sum((text.count(s) for s in './-·')) >= 2 or (digits >= 4 and any((s in text for s in './-·')))):
                continue
            if line.score >= 0.98 and len(line.polygon) == 4 and _signature(text) and all((p.year_digits == 4 for p in parse_dates(text))):
                continue
            targets.append(index)
        return targets[:2]
    
    def _signature(text):
        partial = _printed_month_day_token(text)
        if partial:
            return ((f'NONE-{int(partial[1]):02d}-{int(partial[2]):02d}', 'printed-month-day'),)
        parsed = parse_dates(text)
        if not parsed or any((p.repaired for p in parsed)) or len({(p.start, p.end) for p in parsed}) != 1:
            return None
        return tuple(sorted(((p.value.isoformat(), p.order) for p in parsed)))
    
    def _digit_evidence(text, scores):
        partial = _printed_month_day_token(text)
        if partial and len(scores) == len(text):
            values = [scores[i] for group in (1, 2) for i in range(partial.start(group), partial.end(group))]
            return (sum(values) / len(values), min(values))
        parsed = parse_dates(text)
        if len(scores) != len(text) or not _signature(text):
            return (None, None)
        token = parsed[0]
        if text[token.start:token.end] != token.raw:
            return (None, None)
        values = [scores[i] for i in range(token.start, token.end) if text[i].isdigit()]
        return (sum(values) / len(values), min(values)) if values else (None, None)
    
    def _stable(obs):
        if obs.date_digit_min_score is not None and obs.date_digit_min_score < 0.5:
            return False
        return obs.score >= 0.85 or (obs.score >= 0.65 and obs.date_digit_score is not None and (obs.date_digit_score >= 0.9))
    
    def _supported_role(obs):
        """Require aligned evidence for the role glyphs, independently of digits."""
        role = _inline_role(obs.text)
        if role is None or len(obs.character_scores) != len(obs.text):
            return None
        pattern = POSITIVE_CONTEXT if role == 'end' else NEGATIVE_CONTEXT
        matches = list(pattern.finditer(obs.text))
        if matches and all((min(obs.character_scores[m.start():m.end()]) >= 0.8 for m in matches)):
            return role
        return None
    
    def safe_numeric_change(original, proposed):
        """Do not replace a readable complete token by a shortened crop reading."""
        old = parse_dates(original.text)
        new = parse_dates(proposed.text)
        if not _signature(original.text) or original.score < 0.85 or (not old) or (not new):
            return True
        old_digits = sum((c.isdigit() for c in old[0].raw))
        new_digits = sum((c.isdigit() for c in new[0].raw))
        return new_digits >= old_digits
    
    def _apply_views(active, lines, linked, jobs, observations, *, strict=False, defer_complete_conflicts=False):
        decisions = []
        for index in sorted({job[0] for job in jobs}):
            evidence = [obs for (obs, job) in zip(observations, jobs) if job[0] == index]
            repeated = []
            if strict:
                for obs in evidence:
                    signature = _signature(obs.text)
                    peers = [p for p in evidence if signature and _signature(p.text) == signature and (p.text == obs.text) and (p.date_digit_score is not None) and (p.date_digit_score >= 0.85) and (p.score >= 0.8) and (len(p.character_scores) == len(p.text))]
                    if len(peers) < 2 or max((p.score for p in peers)) < 0.85:
                        continue
                    parses = parse_dates(obs.text)
                    if not parses:
                        continue
                    token = parses[0]
                    positions = [i for i in range(token.start, token.end) if obs.text[i].isdigit()]
                    if positions and all((max((p.character_scores[i] for p in peers)) >= 0.8 for i in positions)):
                        repeated.extend(peers)
    
            def eligible(obs):
                return any((obs is p for p in repeated)) or (_stable(obs) and (not strict or (obs.date_digit_score is not None and obs.date_digit_score >= 0.95 and (obs.date_digit_min_score >= 0.8))))
            votes = Counter((_signature(obs.text) for obs in evidence if eligible(obs) and _signature(obs.text)))
            accepted = None
            change_kind = None
            decision_basis = 'no-eligible-date'
            if votes:
                (signature, count) = votes.most_common(1)[0]
                decision_basis = 'conflicting-dates' if len(votes) > 1 else 'insufficient-support' if count < 2 else 'same-date-no-context-gain'
                if count >= 2 and len(votes) == 1 and (signature != _signature(lines[index].text)):
                    supporters = [obs for obs in evidence if eligible(obs) and _signature(obs.text) == signature]
                    best = max(supporters, key=lambda obs: obs.score)
                    if defer_complete_conflicts and _signature(lines[index].text) and (lines[index].score >= 0.85) and parse_dates(lines[index].text):
                        decisions.append(dict(line_index=index, original_text=lines[index].text, original_box=lines[index].original_box, role=linked[index].role, role_basis=linked[index].role_basis, recognizer='primary', accepted_text=None, reason='keep-original', change_kind=None, decision_basis='complete-token-conflict-needs-cross-recognizer', evidence=[dict(variant=o.variant, text=o.text, score=o.score, signature=_signature(o.text), digit_min=o.date_digit_min_score, eligible=True, rejections=['correlated-views-only']) for o in supporters]))
                        continue
                    if not safe_numeric_change(lines[index], best):
                        decisions.append(dict(line_index=index, original_text=lines[index].text, original_box=lines[index].original_box, role=linked[index].role, role_basis=linked[index].role_basis, recognizer='english' if strict else 'primary', accepted_text=None, reason='keep-original', change_kind=None, decision_basis='complete-token-digit-deletion', evidence=[dict(variant=o.variant, text=o.text, score=o.score, signature=_signature(o.text), digit_min=o.date_digit_min_score, eligible=False, rejections=['complete-token-digit-deletion']) for o in supporters]))
                        continue
                    active[index] = replace(linked[index], text=best.text, score=min((obs.score for obs in supporters)), role_basis='recovery_anchor' if linked[index].role else None, character_scores=best.character_scores, date_digit_score=min((obs.date_digit_score for obs in supporters)) if all((obs.date_digit_score is not None for obs in supporters)) else None, date_digit_min_score=min((obs.date_digit_min_score for obs in supporters)) if all((obs.date_digit_min_score is not None for obs in supporters)) else None)
                    accepted = best.text
                    change_kind = 'date-token'
                    decision_basis = 'stable-date-change'
                elif count >= 2 and len(votes) == 1:
                    supporters = [obs for obs in evidence if eligible(obs) and _signature(obs.text) == signature]
                    roles = {_inline_role(obs.text) for obs in supporters} - {None}
                    grounded = [obs for obs in supporters if _supported_role(obs)]
                    if not strict and linked[index].role is None and (len(roles) == 1) and (len(grounded) >= 2):
                        best = max(grounded, key=lambda obs: obs.score)
                        active[index] = replace(linked[index], text=best.text, score=min(lines[index].score, *(obs.score for obs in grounded)), character_scores=best.character_scores, date_digit_score=best.date_digit_score, date_digit_min_score=best.date_digit_min_score)
                        accepted = best.text
                        change_kind = 'context-only'
                        decision_basis = 'stable-role-recovery'
                    elif len(supporters) >= 2:
                        grounded_digits = [obs for obs in supporters if obs.date_digit_score is not None and obs.date_digit_score >= 0.95 and (obs.date_digit_min_score is not None) and (obs.date_digit_min_score >= 0.9)]
                        if len(grounded_digits) >= 2:
                            best = max(grounded_digits, key=lambda obs: obs.date_digit_min_score)
                            active[index] = replace(linked[index], character_scores=best.character_scores if best.text == linked[index].text else linked[index].character_scores, date_digit_score=min((obs.date_digit_score for obs in grounded_digits)), date_digit_min_score=min((obs.date_digit_min_score for obs in grounded_digits)))
                            accepted = lines[index].text
                            change_kind = 'digit-evidence-only'
                            decision_basis = 'stable-aligned-date-digits'
            audit = []
            for obs in evidence:
                signature = _signature(obs.text)
                rejections = []
                if not signature:
                    rejections.append('no-unrepaired-date-signature')
                if obs.date_digit_min_score is not None and obs.date_digit_min_score < 0.5:
                    rejections.append('weak-digit')
                if not _stable(obs):
                    rejections.append('unstable-view')
                if strict and (not eligible(obs)):
                    rejections.append('secondary-evidence-gate')
                audit.append(dict(variant=obs.variant, text=obs.text, signature=signature, score=obs.score, digit_min=obs.date_digit_min_score, eligible=bool(signature and eligible(obs)), rejections=rejections))
            decisions.append(dict(line_index=index, original_text=lines[index].text, original_box=lines[index].original_box, role=linked[index].role, role_basis=linked[index].role_basis, recognizer='english' if strict else 'primary', accepted_text=accepted, reason='same-model-view-stability' if accepted else 'keep-original', change_kind=change_kind, decision_basis=decision_basis, evidence=audit))
        return decisions
    
    def recover_lines(image, lines, recognize_crops, fallback_recognize=None, *, on_progress=None):
        """Return active lines, raw view observations, and decisions for audit."""
        active = list(lines)
        linked = _link_roles(lines)
        (observations, decisions, jobs, crops) = ([], [], [], [])
        (height, width) = image.shape[:2]
        for index in recovery_targets(lines):
            line = lines[index]
            if not all((math.isfinite(v) for v in line.box)):
                continue
            pad = min(12, max(4, round(line.height * 0.08)))
            for (view, margin, enhance) in [('contrast', 0, True), ('padded', pad, False), ('padded-contrast', pad, True)]:
                (left, top, right, bottom) = line.box
                bounds = (max(0, int(left) - margin), max(0, int(top) - margin), min(width, math.ceil(right) + margin), min(height, math.ceil(bottom) + margin))
                (x1, y1, x2, y2) = bounds
                if x2 - x1 < 8 or y2 - y1 < 8:
                    continue
                crop = image[y1:y2, x1:x2]
                if enhance:
                    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
                    crop = cv2.cvtColor(cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4)).apply(gray), cv2.COLOR_GRAY2BGR)
                crops.append(crop)
                jobs.append((index, view, bounds))
        if not crops:
            return (active, observations, decisions, 0.0)
        started = time.perf_counter()
        results = recognize_crops(crops)
        seconds = time.perf_counter() - started
        if len(results) != len(jobs):
            raise ValueError('Recognition-only result count does not match requested crops')
        for ((index, view, bounds), result) in zip(jobs, results):
            (text, score) = result[:2]
            character_scores = tuple(result[2]) if len(result) > 2 else ()
            (mean_digit, min_digit) = _digit_evidence(text, character_scores)
            observations.append(replace(lines[index], text=text, score=score, box=bounds, polygon=(), variant=f'line-{index}-{view}', members=(), character_scores=character_scores, date_digit_score=mean_digit, date_digit_min_score=min_digit))
        decisions = _apply_views(active, lines, linked, jobs, observations, defer_complete_conflicts=True)
        if on_progress:
            on_progress(active, observations, decisions, seconds)
        pending = {d['line_index'] for d in decisions if not d['accepted_text'] and (not _signature(lines[d['line_index']].text))}
        if fallback_recognize is not None and pending:
            positions = [i for (i, job) in enumerate(jobs) if job[0] in pending]
            started = time.perf_counter()
            try:
                results = fallback_recognize([crops[i] for i in positions])
                if len(results) != len(positions):
                    raise ValueError('Secondary recognition count does not match requested crops')
                extra = []
                for (position, result) in zip(positions, results):
                    chars = tuple(result[2]) if len(result) > 2 else ()
                    (mean, minimum) = _digit_evidence(result[0], chars)
                    extra.append(replace(observations[position], text=result[0], score=result[1], variant=observations[position].variant + '-english', character_scores=chars, date_digit_score=mean, date_digit_min_score=minimum))
                decisions.extend(_apply_views(active, lines, linked, [jobs[i] for i in positions], extra, strict=True))
                observations.extend(extra)
            except Exception as exc:
                decisions.append(dict(line_index=-1, original_text='', original_box=None, accepted_text=None, reason='secondary-recognition-error', error=f'{type(exc).__name__}: {exc}'))
            seconds += time.perf_counter() - started
            if on_progress:
                on_progress(active, observations, decisions, seconds)
        conflicts = {job[0] for (job, obs) in zip(jobs, observations[:len(jobs)]) if _signature(lines[job[0]].text) and _signature(obs.text) and (_signature(obs.text) != _signature(lines[job[0]].text)) and (obs.score >= 0.65)}
        accepted = {d['line_index'] for d in decisions if d.get('accepted_text')}
        (rect_jobs, rect_crops) = ([], [])
        if fallback_recognize is not None:
            for index in sorted((conflicts | pending) - accepted)[:2]:
                views = [(f'rectified-{padding}', rectified_crop(image, lines[index].polygon, padding)) for padding in (0.0, 0.16)] if index in conflicts else recognition_views(image, lines[index].polygon)
                for (view, crop) in views:
                    if crop is not None:
                        rect_crops.append(crop)
                        rect_jobs.append((index, view, lines[index].box))
        if rect_crops:
            started = time.perf_counter()
            try:
                results = fallback_recognize(rect_crops)
                if len(results) != len(rect_jobs):
                    raise ValueError('Rectified recognition count does not match requested crops')
                extra = []
                for ((index, view, bounds), result) in zip(rect_jobs, results):
                    chars = tuple(result[2]) if len(result) > 2 else ()
                    (mean, minimum) = _digit_evidence(result[0], chars)
                    extra.append(replace(lines[index], text=result[0], score=result[1], variant=f'line-{index}-{view}-english', members=(), character_scores=chars, date_digit_score=mean, date_digit_min_score=minimum))
                rect_decisions = _apply_views(active, lines, linked, rect_jobs, extra, strict=True)
                for decision in rect_decisions:
                    decision['stage'] = 'rectified-conflict-recheck' if decision['line_index'] in conflicts else 'rectified-unparsed-recheck'
                decisions.extend(rect_decisions)
                observations.extend(extra)
            except Exception as exc:
                decisions.append(dict(line_index=-1, original_text='', original_box=None, accepted_text=None, reason='rectified-recognition-error', error=f'{type(exc).__name__}: {exc}'))
            seconds += time.perf_counter() - started
            if on_progress:
                on_progress(active, observations, decisions, seconds)
        accepted = {d['line_index'] for d in decisions if d.get('accepted_text')}
        (extra_jobs, extra_crops) = ([], [])
        for index in sorted(pending - accepted):
            for padding in (0.3, 0.6):
                crop = rectified_crop(image, lines[index].polygon, padding)
                if crop is None:
                    continue
                h = crop.shape[0]
                extra_crops.append(cv2.resize(crop, (round(h * 4.8), h), interpolation=cv2.INTER_AREA))
                extra_jobs.append((index, f'primary-rectified-{padding}', lines[index].box))
        if extra_crops:
            started = time.perf_counter()
            results = recognize_crops(extra_crops)
            if len(results) != len(extra_jobs):
                raise ValueError('Primary rectified result count does not match requested crops')
            extra = []
            for ((index, view, bounds), result) in zip(extra_jobs, results):
                chars = tuple(result[2]) if len(result) > 2 else ()
                (mean, minimum) = _digit_evidence(result[0], chars)
                extra.append(replace(lines[index], text=result[0], score=result[1], variant=view, character_scores=chars, date_digit_score=mean, date_digit_min_score=minimum))
            audit = _apply_views(active, lines, linked, extra_jobs, extra, strict=True)
            for decision in audit:
                decision.update(stage='primary-rectified-unparsed', recognizer='primary')
            decisions.extend(audit)
            observations.extend(extra)
            seconds += time.perf_counter() - started
            if on_progress:
                on_progress(active, observations, decisions, seconds)
        return (active, observations, decisions, seconds)
    
    def _flat_label_rows(image, label, *, masked=False):
        """Pixel rows in a bright panel left of one expiry label, in original coordinates.
    
        This is a narrow missing-detector fallback, not a general text detector.
        Long, shallow dark rows are eligible; more than two rows is ambiguous.
        """
        (height, width) = image.shape[:2]
        (x1, y1, x2, y2) = label.box
        h = y2 - y1
        (left, top) = (max(0, int(x1 - 12 * h)), max(0, int(y1 - 2 * h)))
        (right, bottom) = (min(width, int(x1)), min(height, int(y2 + h)))
        if h < 8 or right - left < 8 or bottom - top < 8:
            return []
        gray = cv2.cvtColor(image[top:bottom, left:right], cv2.COLOR_BGR2GRAY)
        (contours, _) = cv2.findContours(cv2.inRange(gray, 170, 255), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        panels = [cv2.boundingRect(c) for c in contours if cv2.contourArea(c) > 1.5 * h * h]
        if not panels:
            return []
        (px, py, pw, ph) = max(panels, key=lambda b: b[2] * b[3])
        inset = max(2, round(h * 0.14))
        if pw <= 2 * inset or ph <= 2 * inset:
            return []
        panel_mask = None
        if masked:
            import numpy as np
            contour = max((c for c in contours if cv2.contourArea(c) > 1.5 * h * h), key=lambda c: math.prod(cv2.boundingRect(c)[2:]))
            panel_mask = np.zeros_like(gray)
            cv2.drawContours(panel_mask, [contour], -1, 255, -1)
            panel_mask = cv2.erode(panel_mask, cv2.getStructuringElement(cv2.MORPH_RECT, (2 * inset + 1, 2 * inset + 1)))
            panel_mask = panel_mask[py + inset:py + ph - inset, px + inset:px + pw - inset]
        gray = gray[py + inset:py + ph - inset, px + inset:px + pw - inset]
        left += px + inset
        top += py + inset
        binary = cv2.inRange(gray, 0, 120)
        if masked:
            binary = cv2.bitwise_and(binary, panel_mask)
            joined = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (max(3, round(h * 1.5)), 1)))
            ys = np.flatnonzero(np.count_nonzero(joined, axis=1) >= 2 * h)
            if not len(ys):
                return []
            rows = []
            gap_limit = max(2, round(h * 0.06))
            for run in np.split(ys, np.flatnonzero(np.diff(ys) > gap_limit + 1) + 1):
                (a, b) = (int(run[0]), int(run[-1]) + 1)
                xs = np.flatnonzero(np.any(binary[a:b], axis=0))
                if not len(xs):
                    continue
                (x, w, rh) = (int(xs[0]), int(xs[-1] - xs[0] + 1), b - a)
                if w >= 2 * h and 8 <= rh <= 1.4 * h and (w / rh >= 4):
                    rows.append((left + x, top + a, left + x + w, top + b))
            return rows if 1 <= len(rows) <= 2 else []
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(3, round(h * 1.5)), 3))
        joined = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        (contours, _) = cv2.findContours(joined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        rows = []
        for contour in contours:
            (x, y, w, rh) = cv2.boundingRect(contour)
            if w >= 2 * h and 8 <= rh <= 1.4 * h and (w / rh >= 8):
                rows.append((left + x, top + y, left + x + w, top + y + rh))
        return sorted(rows, key=lambda b: b[1]) if 1 <= len(rows) <= 2 else []
    
    def recover_missing_rows(image, lines, recognize_crops, fallback_recognize=None):
        """At most two batches of four crops, two stable views per new row.
    
        Date digits must be supported by aligned character probabilities. No digit
        rewriting, date-order override, or model-agreement confidence bonus.
        """
        labels = [line for line in lines if line.geometry_valid and line.score >= 0.9 and (line.box[2] - line.box[0] >= line.height) and re.fullmatch('까지|EXP|BBD|소비기한|유통기한', line.text.strip(), re.I)]
        if len(labels) != 1 or any((parse_dates(line.text) for line in lines)):
            return ([], [], [], 0.0)
        label = labels[0]
        boxes = _flat_label_rows(image, label)
        primary = _recover_missing_boxes(image, lines, label, boxes, recognize_crops)
        if primary[0] or fallback_recognize is None:
            return primary
        boxes = _flat_label_rows(image, label, masked=True)
        secondary = _recover_missing_boxes(image, lines, label, boxes, fallback_recognize, secondary=True, index_offset=len(primary[2]))
        return (secondary[0], primary[1] + secondary[1], primary[2] + secondary[2], primary[3] + secondary[3])
    
    def _recover_missing_boxes(image, lines, label, boxes, recognize_crops, *, secondary=False, index_offset=0):
        (jobs, crops) = ([], [])
        (height, width) = image.shape[:2]
        for (index, (x1, y1, x2, y2)) in enumerate(boxes):
            pad = max(3, round((y2 - y1) * 0.3))
            bounds = (max(0, x1 - pad), max(0, y1 - pad), min(width, x2 + pad), min(height, y2 + pad))
            (a, b, c, d) = bounds
            ratio = (x2 - x1) / (y2 - y1)
            scales = (min(1.0, 4.8 / ratio), min(1.08, 5.2 / ratio)) if secondary else (0.4, 0.45)
            for scale in scales:
                crops.append(cv2.resize(image[b:d, a:c], None, fx=scale, fy=1, interpolation=cv2.INTER_AREA))
                jobs.append((index, scale, bounds))
        if not crops:
            return ([], [], [], 0.0)
        started = time.perf_counter()
        results = recognize_crops(crops)
        seconds = time.perf_counter() - started
        if len(results) != len(jobs):
            raise ValueError('Missing-row recognition count does not match requested crops')
        observations = []
        for ((index, scale, bounds), result) in zip(jobs, results):
            (text, score) = result[:2]
            chars = tuple(result[2]) if len(result) > 2 else ()
            (mean, minimum) = _digit_evidence(text, chars)
            observations.append(OCRLine(text, score, bounds, source=label.source, variant=f'line-{len(lines) + index_offset + index}-flat-{scale}' + ('-english' if secondary else ''), original_box=bounds, character_scores=chars, date_digit_score=mean, date_digit_min_score=minimum))
        (accepted, decisions) = ([], [])
        for (index, box) in enumerate(boxes):
            views = observations[2 * index:2 * index + 2]
            signatures = [_signature(view.text) for view in views]
            stable = all(signatures) and signatures[0] == signatures[1] and all((view.score >= 0.85 and view.date_digit_score is not None and (view.date_digit_score >= (0.95 if secondary else 0.9)) and (view.date_digit_min_score >= (0.8 if secondary else 0.7)) for view in views))
            best = max(views, key=lambda view: view.score)
            if stable:
                accepted.append(replace(best, score=min((view.score for view in views)), box=box, original_box=box, variant=label.variant, date_digit_score=min((view.date_digit_score for view in views)), date_digit_min_score=min((view.date_digit_min_score for view in views))))
            decisions.append(dict(line_index=len(lines) + index_offset + index, original_text='', original_box=box, recognizer='english' if secondary else 'primary', accepted_text=best.text if stable else None, reason='flat-row-view-stability' if stable else 'unstable-flat-row'))
        if len(accepted) != len(boxes):
            for decision in decisions:
                decision.update(accepted_text=None, reason='incomplete-flat-row-block')
            accepted = []
        return (accepted, observations, decisions, seconds)
    return _ITDA_Namespace(**locals())
_ITDA_NS['line_recovery'] = _itda_define_line_recovery()
del _itda_define_line_recovery


In [ ]:
from __future__ import annotations
# Embedded from src/ocr_trace.py; generated, do not edit separately.
def _itda_define_ocr_trace():
    """Diagnostic region identity and evidence, deliberately not a date selector."""
    from dataclasses import asdict, dataclass
    import math
    import re
    POSITIVE_CONTEXT = _ITDA_NS['date_extraction'].POSITIVE_CONTEXT
    NEGATIVE_CONTEXT = _ITDA_NS['date_extraction'].NEGATIVE_CONTEXT
    parse_dates = _ITDA_NS['date_extraction'].parse_dates
    _partial_dates = _ITDA_NS['date_extraction']._partial_dates
    
    @dataclass(frozen=True)
    class ImageFrame:
        """Affine map from continuous image-edge coordinates to EXIF-oriented input.
    
        Width/height are edge extents, not last pixel indices. Crop scale uses
        actual resized dimensions, not the requested floating-point resize factor.
        """
        width: int
        height: int
        to_original: tuple[float, ...] = (1.0, 0.0, 0.0, 0.0, 1.0, 0.0)
    
        def map_polygon(self, polygon):
            (a, b, c, d, e, f) = self.to_original
            return tuple(((a * x + b * y + c, d * x + e * y + f) for (x, y) in polygon))
    
        @classmethod
        def crop(cls, bounds, shape):
            (left, top, right, bottom) = bounds
            (height, width) = shape[:2]
            return cls(width, height, ((right - left) / width, 0.0, left, 0.0, (bottom - top) / height, top))
    
        @classmethod
        def rotated(cls, width, height, angle):
            transforms = {90: (height, width, (0.0, 1.0, 0.0, -1.0, 0.0, height)), 180: (width, height, (-1.0, 0.0, width, 0.0, -1.0, height)), 270: (height, width, (0.0, -1.0, width, 1.0, 0.0, 0.0))}
            return cls(*transforms[angle])
    
    def box_polygon(box):
        (left, top, right, bottom) = box
        return ((left, top), (right, top), (right, bottom), (left, bottom))
    
    def _bounds(polygon):
        (xs, ys) = zip(*polygon)
        return (min(xs), min(ys), max(xs), max(ys))
    
    def _overlap(first, second):
        (left, top) = (max(first[0], second[0]), max(first[1], second[1]))
        (right, bottom) = (min(first[2], second[2]), min(first[3], second[3]))
        intersection = max(0, right - left) * max(0, bottom - top)
        area1 = (first[2] - first[0]) * (first[3] - first[1])
        area2 = (second[2] - second[0]) * (second[3] - second[1])
        return intersection / (area1 + area2 - intersection) if area1 + area2 else 0.0
    
    def _evidence(text):
        hints = []
        for (role, pattern) in (('expiry', POSITIVE_CONTEXT), ('manufacturing', NEGATIVE_CONTEXT), ('quality', re.compile('\\b(?:BBD|TETT)\\b', re.I))):
            hints.extend((dict(role=role, text=m.group(), span=(m.start(), m.end()), basis='lexical_only') for m in pattern.finditer(text)))
        full = [dict(value=p.value.isoformat(), raw=p.raw, span=(p.start, p.end), order=p.order, order_reason=p.order_reason, repaired=p.repaired) for p in parse_dates(text)]
        partial = list(dict.fromkeys(_partial_dates(text))) if not full else []
        digits = sum((c.isdigit() for c in text))
        separators = sum((text.count(s) for s in './-'))
        date_like = bool(hints or full or partial or (len(text) <= 40 and (digits >= 4 and separators >= 1 or (digits >= 2 and separators >= 2))))
        return dict(role_hints=hints, full_parses=full, partial_parses=partial, date_like=date_like, parse_status='full' if full else 'partial' if partial else 'unparsed' if text.strip() else 'no_text')
    
    class ImageTrace:
        """Per-image trace. IDs are local, geometric hypotheses are not semantic merges."""
    
        def __init__(self, image_id, emit=None):
            self.image_id = image_id
            self.emit = emit
            self.frames = []
            self.observations = []
            self.regions = []
            self.outcomes = []
            self.original_size = None
            self.recovery_decisions = []
    
        def start(self, width, height):
            self.original_size = (width, height)
            self._emit(dict(kind='image_start', original_size=self.original_size, coordinate_space='exif_oriented_image_edges', schema_version=1))
    
        def _emit(self, event):
            if self.emit is not None:
                self.emit(dict(image_id=self.image_id, **event))
    
        def start_pass(self, frame, detector, variant):
            self._emit(dict(kind='ocr_start', pass_id=f'p{len(self.frames) + 1:03d}', detector=detector, variant=variant, geometry=asdict(frame) if frame else None))
    
        def record_pass(self, lines, unrecognized, frame, detector, variant, seconds, selection=None, error=None):
            pass_id = f'p{len(self.frames) + 1:03d}'
            frame_record = dict(pass_id=pass_id, detector=detector, variant=variant, geometry=asdict(frame) if frame else None)
            self.frames.append(frame_record)
            added = []
            used = set()
            for (index, line) in enumerate([*lines, *unrecognized]):
                polygon = line.polygon or box_polygon(line.box)
                valid = frame is not None and line.geometry_valid and (len(polygon) >= 3) and all((math.isfinite(v) for point in polygon for v in point))
                mapped = frame.map_polygon(polygon) if valid else None
                bounds = _bounds(mapped) if mapped else None
                if bounds and (bounds[2] <= bounds[0] or bounds[3] <= bounds[1]):
                    (mapped, bounds) = (None, None)
                matches = []
                if bounds:
                    matches = [region for region in self.regions if region['anchor_box'] and _overlap(bounds, region['anchor_box']) >= 0.75]
                match = matches[0] if len(matches) == 1 and matches[0]['region_id'] not in used else None
                if match is None:
                    match = dict(region_id=f'r{len(self.regions) + 1:04d}', anchor_box=bounds, observation_ids=[])
                    self.regions.append(match)
                    association = 'ambiguous_new' if matches else 'new'
                else:
                    association = 'geometric_overlap_hypothesis'
                region_id = match['region_id']
                used.add(region_id)
                observation_id = f'{pass_id}:o{index + 1:04d}'
                match['observation_ids'].append(observation_id)
                record = dict(observation_id=observation_id, region_id=region_id, pass_id=pass_id, source=line.source, variant=line.variant, text=line.text, character_scores=line.character_scores, date_digit_score=line.date_digit_score, date_digit_min_score=line.date_digit_min_score, score=None if line.geometry_source == 'detector_polygon' else line.score, local_box=line.box, local_polygon=polygon, original_polygon=mapped, original_box=bounds, geometry_source=line.geometry_source, association=association, possible_region_ids=[r['region_id'] for r in matches], recognition_state='recognized' if index < len(lines) else 'detected_without_text', **_evidence(line.text))
                added.append(record)
                self.observations.append(record)
            outcome = dict(pass_id=pass_id, ocr_seconds=seconds, error=error, selection=self._selection(selection) if selection else None)
            self.outcomes.append(outcome)
            self._emit(dict(kind='ocr_pass', frame=frame_record, observations=added, outcome=outcome))
    
        def _selection(self, selection):
            candidates = []
            for candidate in selection.candidates:
                ids = [o['observation_id'] for o in self.observations if o['source'] == candidate.source and o['variant'] == candidate.variant and (tuple(o['local_box']) == tuple(candidate.box)) and (candidate.raw in o['text'])]
                for decision in self.recovery_decisions:
                    if decision['accepted_text'] and candidate.raw in decision['accepted_text'] and (tuple(decision.get('original_box') or ()) == tuple(candidate.box)):
                        ids.extend((o['observation_id'] for o in self.observations if o['variant'].startswith(f"line-{decision['line_index']}-") and any((p['value'] == candidate.iso for p in o['full_parses']))))
                candidates.append(dict(value=candidate.iso, raw=candidate.raw, box=candidate.box, source=candidate.source, variant=candidate.variant, observation_ids=ids, order_reason=candidate.order_reason, explicit_expiry=candidate.explicit_positive, explicit_manufacturing=candidate.explicit_negative))
            return dict(final_date=selection.final_date, reason=selection.reason, stop_ocr=selection.stop_ocr, digits_confident=selection.digits_confident, order_resolved=selection.order_resolved, candidates=candidates, policy_details=getattr(selection, 'policy_details', None))
    
        def record_recovery(self, decisions):
            self.recovery_decisions.extend(decisions)
            self._emit(dict(kind='line_recovery', decisions=decisions))
    
        def record_selector_input(self, lines, *, stage, final=False):
            self._emit(dict(kind='selector_input', stage=stage, final=final, lines=[asdict(line) for line in lines]))
    
        def finish(self, selection, error=None):
            self._emit(dict(kind='image_end', selection=self._selection(selection), error=error, summary=self.summary()))
    
        def summary(self):
            return dict(passes=len(self.frames), regions=len(self.regions), observations=len(self.observations), unparsed_date_like=sum((o['date_like'] and o['parse_status'] == 'unparsed' for o in self.observations)), detected_without_text=sum((o['recognition_state'] == 'detected_without_text' for o in self.observations)), unknown_geometry=sum((o['original_box'] is None for o in self.observations)), geometric_associations=sum((o['association'] == 'geometric_overlap_hypothesis' for o in self.observations)), failed_passes=sum((o['error'] is not None for o in self.outcomes)), line_recovery_changes=sum((d['accepted_text'] is not None for d in self.recovery_decisions)), ocr_seconds=sum((o['ocr_seconds'] for o in self.outcomes)))
    return _ITDA_Namespace(**locals())
_ITDA_NS['ocr_trace'] = _itda_define_ocr_trace()
del _itda_define_ocr_trace


In [ ]:
from __future__ import annotations
# Embedded from src/recovery_policy.py; generated, do not edit separately.
def _itda_define_recovery_policy():
    """Validated, model-bound gain-per-second scheduling; no labels at inference."""
    import hashlib
    import json
    import math
    from pathlib import Path
    submission_decision = _ITDA_NS['date_extraction'].submission_decision
    submission_fields = _ITDA_NS['date_extraction'].submission_fields
    VERSION = 'paper-recovery-v1'
    FIELDS = ('year', 'month', 'day')
    
    def binding(config):
        models = {p.relative_to(config.weights_dir).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(Path(config.weights_dir).rglob('inference.*')) if p.is_file()}
        knobs = {k: getattr(config, k) for k in ('mobile_side_limit', 'recovery_side_limit', 'recognition_minimum_width', 'recognition_batch_size', 'enable_width_batches', 'collect_ctc_candidates')}
        return hashlib.sha256(json.dumps(dict(version=VERSION, models=models, knobs=knobs), sort_keys=True).encode()).hexdigest()
    
    def features(selection, lines, value=None):
        fields = submission_fields(value if value is not None else selection.final_date)
        valid = [c for c in selection.candidates if not c.explicit_negative and (not c.repaired)]
        confidences = {}
        for field in FIELDS:
            supporting = [c.ocr_score for c in valid if submission_fields(c.iso)[field] == fields[field]]
            confidences[field] = max(supporting, default=0.0) if fields[field] != 'NONE' else 0.0
        decision = submission_decision(selection, base_partial=selection.is_partial)
        reason = decision.recovery_reason or ('partial' if selection.is_partial else 'selected')
        missing = sum((fields[k] == 'NONE' for k in FIELDS))
        score = min(confidences.values())
        bucket = min(4, max(0, int(score * 5)))
        key = '|'.join(map(str, (reason, missing, min(len(lines) // 20, 3), bucket, int(selection.role_resolved), int(selection.order_resolved))))
        return dict(key=key, confidence=confidences, missing_fields=missing, reason=reason)
    
    class RecoveryPolicy:
    
        def __init__(self, path, expected_binding):
            self.path = Path(path)
            self.artifact = json.loads(self.path.read_text(encoding='utf-8'))
            a = self.artifact
            if a.get('version') != VERSION or a.get('binding') != expected_binding or a.get('deployment_approved') is not True:
                raise ValueError('Recovery policy must be validated, approved and match the exact model/config')
            proof = a.get('validation', {})
            if proof.get('group_disjoint') is not True or proof.get('passed') is not True or (not proof.get('report_sha256')) or (not proof.get('source_reference')):
                raise ValueError('Independent validation evidence is required; fitted tables alone are not deployment proof')
            self.sha256 = hashlib.sha256(self.path.read_bytes()).hexdigest()
    
        def estimate(self, state, action):
            table = self.artifact['actions'].get(state['key'] + '|' + action)
            if not table or table.get('independent_images', 0) < 20:
                return None
            (gain, seconds) = (table['mean_net_fields'], table['p80_seconds'] * 1.25)
            if not all((math.isfinite(v) for v in (gain, seconds))) or not -3 <= gain <= 3 or seconds <= 0:
                raise ValueError('Invalid recovery utility table')
            return dict(expected_net_fields=gain, estimated_seconds=seconds, utility=gain / seconds)
    
        def field_probabilities(self, state):
            result = {}
            for (field, score) in state['confidence'].items():
                table = self.artifact['fields'].get(field + '|' + str(min(4, max(0, int(score * 5)))))
                result[field] = table['correct_probability'] if table and table.get('independent_images', 0) >= 20 else None
            return result
    
        def pop(self, queue):
            self.turn = getattr(self, 'turn', 0) + 1
            if self.turn % 4 == 0:
                return queue.pop(0)
            scored = []
            for (index, item) in enumerate(queue):
                state = features(item['selection'], item['lines'], item['retained_output'].final_date)
                estimate = self.estimate(state, item['actions'][0])
                if estimate and estimate['expected_net_fields'] > 0:
                    scored.append((estimate['utility'], -index, index))
            return queue.pop(max(scored)[2] if scored else 0)
    return _ITDA_Namespace(**locals())
_ITDA_NS['recovery_policy'] = _itda_define_recovery_policy()
del _itda_define_recovery_policy


In [ ]:
from __future__ import annotations
# Embedded from src/verified_product_rules.py; generated, do not edit separately.
def _itda_define_verified_product_rules():
    """User-confirmed package formats; identify printed products, never image IDs."""
    ProductDateRule = _ITDA_NS['date_extraction'].ProductDateRule
    VERIFIED_PRODUCT_RULES = (ProductDateRule(name='mocha_gold_mix_sticks', required_text=('모카골드', '믹스커피', 'STICKS'), order='ymd', evidence='User confirmed YMD for the reviewed mocha-gold mix product on 2026-09-13.'), ProductDateRule(name='barilla_multilingual_parma_package', required_text=('Barilla', 'Fratelli', 'Parma', 'HOUDBAAR TOT'), order='dmy', evidence='User confirmed DMY for the reviewed Barilla multilingual package on 2026-09-13.'))
    return _ITDA_Namespace(**locals())
_ITDA_NS['verified_product_rules'] = _itda_define_verified_product_rules()
del _itda_define_verified_product_rules


In [ ]:
from __future__ import annotations
# Embedded from src/printed_context_recovery.py; generated, do not edit separately.
def _itda_define_printed_context_recovery():
    """Bounded dot-stroke recovery on automatic full-image regions only."""
    from dataclasses import replace
    import time
    import cv2
    import numpy as np
    OCRLine = _ITDA_NS['date_extraction'].OCRLine
    _inline_role = _ITDA_NS['date_extraction']._inline_role
    numeric_region_proposals = _ITDA_NS['date_region_recovery'].numeric_region_proposals
    rectified_crop = _ITDA_NS['date_region_recovery'].rectified_crop
    _apply_views = _ITDA_NS['line_recovery']._apply_views
    _digit_evidence = _ITDA_NS['line_recovery']._digit_evidence
    _supported_role = _ITDA_NS['line_recovery']._supported_role
    
    def stroke_views(image, polygon):
        crop = rectified_crop(image, polygon, 0.1)
        if crop is None:
            return []
        (h, w) = crop.shape[:2]
        crop = cv2.resize(crop, (round(w * 96 / h), 96))
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        joined = cv2.morphologyEx(gray, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
        return [(f'stroke-{ratio}', cv2.cvtColor(cv2.resize(joined, (round(96 * ratio), 96)), cv2.COLOR_GRAY2BGR)) for ratio in (4.8, 6.5)]
    
    def recover_printed_context(image, lines, recognize_crops):
        started = time.perf_counter()
        eligible = [line for line in lines if line.variant in ('original', 'clahe', 'geometric-rows')]
        polygons = numeric_region_proposals(image, eligible)
        (anchors, crops, jobs, seen) = ([], [], [], set())
        for polygon in polygons:
            key = tuple((round(v) for p in polygon for v in p))
            if key in seen:
                continue
            seen.add(key)
            views = stroke_views(image, polygon)
            if len(views) != 2 or np.array_equal(views[0][1], views[1][1]):
                continue
            (xs, ys) = zip(*polygon)
            box = (min(xs), min(ys), max(xs), max(ys))
            index = len(anchors)
            anchors.append(OCRLine('', 0.0, box, source='paddle-stroke', variant='stroke-rows', original_box=box, polygon=polygon))
            for (name, crop) in views:
                crops.append(crop)
                jobs.append((index, name, box))
        if not crops:
            return ([], [], [], time.perf_counter() - started)
        results = recognize_crops(crops)
        if len(results) != len(jobs):
            raise ValueError('Stroke recognition result count mismatch')
        observations = []
        for ((index, name, _), result) in zip(jobs, results):
            chars = tuple(result[2]) if len(result) > 2 else ()
            (mean, minimum) = _digit_evidence(result[0], chars)
            observations.append(replace(anchors[index], text=result[0], score=result[1], variant=name, character_scores=chars, date_digit_score=mean, date_digit_min_score=minimum))
        active = list(anchors)
        decisions = _apply_views(active, anchors, anchors, jobs, observations, strict=True)
        added = []
        for decision in decisions:
            index = decision['line_index']
            views = observations[2 * index:2 * index + 2]
            roles = [_inline_role(view.text) for view in views]
            if any(roles) and (len(set(roles)) != 1 or any((_supported_role(v) != roles[0] for v in views))):
                decision.update(accepted_text=None, decision_basis='unsupported-role-glyphs')
            if decision['accepted_text']:
                added.append(active[index])
            decision.update(stage='printed-stroke-context', recognizer='primary', line_index=len(lines) + index)
        return (added, observations, decisions, time.perf_counter() - started)
    return _ITDA_Namespace(**locals())
_ITDA_NS['printed_context_recovery'] = _itda_define_printed_context_recovery()
del _itda_define_printed_context_recovery


In [ ]:
from __future__ import annotations
# Embedded from src/thin_dot_recovery.py; generated, do not edit separately.
def _itda_define_thin_dot_recovery():
    """Recover disconnected print without merging the neighboring clock/lot row."""
    from dataclasses import replace
    import time
    import cv2
    import numpy as np
    OCRLine = _ITDA_NS['date_extraction'].OCRLine
    _inline_role = _ITDA_NS['date_extraction']._inline_role
    dot_row_proposals = _ITDA_NS['date_region_recovery'].dot_row_proposals
    rectified_crop = _ITDA_NS['date_region_recovery'].rectified_crop
    _apply_views = _ITDA_NS['line_recovery']._apply_views
    _digit_evidence = _ITDA_NS['line_recovery']._digit_evidence
    _signature = _ITDA_NS['line_recovery']._signature
    _supported_role = _ITDA_NS['line_recovery']._supported_role
    
    def thin_views(image, polygon, *, natural=False):
        crop = rectified_crop(image, polygon, 0.15 if natural else 0.35)
        if crop is None:
            return []
        (h, w) = crop.shape[:2]
        crop = cv2.resize(crop, (round(w * 96 / h), 96))
        plain = crop if natural else cv2.resize(crop, (624, 96))
        joined = cv2.morphologyEx(plain, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
        if np.array_equal(plain, joined):
            return []
        return [('thin-plain', plain), ('thin-joined', joined)]
    
    def recover_thin_dots(image, lines, primary, english=None, *, natural=False):
        started = time.perf_counter()
        (anchors, jobs, crops) = ([], [], [])
        options = dict(limit=6, thin=True)
        if natural:
            options['contrast'] = 8
        stage = 'natural-dot-rows' if natural else 'thin-dot-rows'
        for polygon in dot_row_proposals(image, **options):
            views = thin_views(image, polygon, natural=natural)
            if len(views) != 2:
                continue
            (xs, ys) = zip(*polygon)
            box = (min(xs), min(ys), max(xs), max(ys))
            index = len(anchors)
            anchors.append(OCRLine('', 0.0, box, source='paddle-thin-dot', variant=stage, original_box=box, polygon=polygon))
            for (name, crop) in views:
                jobs.append((index, name, box))
                crops.append(crop)
        (observations, decisions, choices) = ([], [], {})
        for (name, recognize) in [('primary', primary), ('english', english)]:
            if recognize is None or not crops:
                continue
            results = recognize(crops)
            if len(results) != len(jobs):
                raise ValueError('Thin-dot recognition result count mismatch')
            extra = []
            for ((index, variant, _), result) in zip(jobs, results):
                chars = tuple(result[2]) if len(result) > 2 else ()
                (mean, minimum) = _digit_evidence(result[0], chars)
                extra.append(replace(anchors[index], text=result[0], score=result[1], variant=variant + '-' + name, character_scores=chars, date_digit_score=mean, date_digit_min_score=minimum))
            active = list(anchors)
            audit = _apply_views(active, anchors, anchors, jobs, extra, strict=True)
            for decision in audit:
                index = decision['line_index']
                views = extra[2 * index:2 * index + 2]
                roles = [_inline_role(v.text) for v in views]
                if any(roles) and (len(set(roles)) != 1 or any((_supported_role(v) != roles[0] for v in views))):
                    decision.update(accepted_text=None, decision_basis='unsupported-role-glyphs')
                if decision['accepted_text']:
                    choices.setdefault(index, []).append(active[index])
                decision.update(stage=stage, recognizer=name, line_index=len(lines) + index)
            observations.extend(extra)
            decisions.extend(audit)
        accepted = []
        for (index, values) in choices.items():
            if len({_signature(v.text) for v in values}) > 1:
                for d in decisions:
                    if d['line_index'] == len(lines) + index:
                        d.update(accepted_text=None, decision_basis='cross-recognizer-conflict')
                continue
            accepted.append(max(values, key=lambda v: v.score))
        discarded = set()
        for (i, first) in enumerate(accepted):
            for (j, second) in enumerate(accepted[:i]):
                (a, b) = (first.box, second.box)
                intersection = max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
                if intersection / min(first.width * first.height, second.width * second.height) < 0.7:
                    continue
                if _signature(first.text) != _signature(second.text):
                    discarded.update((i, j))
                else:
                    discarded.add(i if first.score <= second.score else j)
        kept = [v for (i, v) in enumerate(accepted) if i not in discarded]
        for d in decisions:
            if d.get('accepted_text') and (not any((v.box == d['original_box'] and v.text == d['accepted_text'] for v in kept))):
                d.update(accepted_text=None, decision_basis='overlapping-row-conflict-or-duplicate')
        return (kept, observations, decisions, time.perf_counter() - started)
    return _ITDA_Namespace(**locals())
_ITDA_NS['thin_dot_recovery'] = _itda_define_thin_dot_recovery()
del _itda_define_thin_dot_recovery


In [ ]:
from __future__ import annotations
# Embedded from src/pipeline.py; generated, do not edit separately.
def _itda_define_pipeline():
    import csv
    import hashlib
    import json
    import math
    import os
    import re
    import time
    from collections import Counter
    from collections.abc import Callable, Sequence
    from dataclasses import dataclass, field, replace
    from pathlib import Path
    from typing import Any
    os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = '1'
    import cv2
    import numpy as np
    from PIL import Image, ImageOps
    DateContext = _ITDA_NS['date_extraction'].DateContext
    DateSelection = _ITDA_NS['date_extraction'].DateSelection
    OCRLine = _ITDA_NS['date_extraction'].OCRLine
    ProductDateRule = _ITDA_NS['date_extraction'].ProductDateRule
    parse_dates = _ITDA_NS['date_extraction'].parse_dates
    select_date = _ITDA_NS['date_extraction'].select_date
    submission_fields = _ITDA_NS['date_extraction'].submission_fields
    ImageFrame = _ITDA_NS['ocr_trace'].ImageFrame
    ImageTrace = _ITDA_NS['ocr_trace'].ImageTrace
    recover_lines = _ITDA_NS['line_recovery'].recover_lines
    recover_missing_rows = _ITDA_NS['line_recovery'].recover_missing_rows
    recovery_targets = _ITDA_NS['line_recovery'].recovery_targets
    recover_geometric_rows = _ITDA_NS['date_region_recovery'].recover_geometric_rows
    recover_printed_context = _ITDA_NS['printed_context_recovery'].recover_printed_context
    recover_thin_dots = _ITDA_NS['thin_dot_recovery'].recover_thin_dots
    CTCEvidence = _ITDA_NS['recognition_evidence'].CTCEvidence
    VERIFIED_PRODUCT_RULES = _ITDA_NS['verified_product_rules'].VERIFIED_PRODUCT_RULES
    inner_ocr_pipeline = _ITDA_NS['shared_detector'].inner_ocr_pipeline
    SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
    OUTPUT_COLUMNS = ['image_id', 'year', 'month', 'day', 'final_date']
    MODEL_FILES = ('inference.json', 'inference.pdiparams', 'inference.yml')
    
    @dataclass(frozen=True)
    class PipelineConfig:
        weights_dir: Path = field(default_factory=lambda : _ITDA_BUNDLE_ROOT / 'weights' / 'paddle')
        mobile_side_limit: int = 1600
        recovery_detector_name: str = 'PP-OCRv6_small_det'
        recovery_side_limit: int = 1600
        cpu_threads: int = 4
        recognition_batch_size: int = 8
        recognition_minimum_width: int = 160
        enable_clahe: bool = True
        enable_recovery_fallback: bool = True
        enable_rotation_fallback: bool = False
        enable_tile_fallback: bool = True
        tile_fraction: float = 0.62
        tile_max_original_lines: int = 32
        tile_sparse_line_limit: int = 5
        progress_every: int = 25
        product_date_rules: tuple[ProductDateRule, ...] = VERIFIED_PRODUCT_RULES
        date_context: DateContext | None = field(default_factory=DateContext)
        collect_trace: bool = True
        enable_line_recovery: bool = True
        enable_geometric_recovery: bool = True
        enable_width_batches: bool = True
        collect_ctc_candidates: bool = False
        recovery_policy_path: Path | None = None
    
    @dataclass(frozen=True)
    class ImagePrediction:
        image_id: str
        final_date: str | None
        selection: DateSelection
        elapsed_seconds: float
        passes: tuple[str, ...]
        error: str | None = None
        trace: dict[str, Any] | None = None
    
    def configure_recognizer_width(model, width):
        """Trim only short-line zero padding; retain height and longer-line pixels."""
        if width not in (160, 192, 320):
            raise ValueError('Unsupported verified padding experiment width')
        resize = model.pre_tfs['ReisizeNorm']
        if list(resize.rec_image_shape) != [3, 48, 320]:
            raise ValueError('Recognizer resize contract changed')
        resize.rec_image_shape = [3, 48, width]
    
    class PaddleOCRBackend:
        """Two-detector PaddleOCR backend with fully local model paths."""
    
        def __init__(self, config: PipelineConfig):
            self.config = config
            Profile = _ITDA_NS['performance_profile'].Profile
            self.profile = Profile(os.environ.get('ITDA_PROFILE_PATH'))
            self._validate_model('PP-OCRv5_mobile_det')
            self._validate_model('korean_PP-OCRv5_mobile_rec')
            self._mobile = self._build('PP-OCRv5_mobile_det', config.mobile_side_limit)
            self._recovery = None
            self._date_recognizer = None
    
        def _model_dir(self, name: str) -> Path:
            return self.config.weights_dir / name
    
        def _validate_model(self, name: str) -> None:
            model_dir = self._model_dir(name)
            missing = [filename for filename in MODEL_FILES if not (model_dir / filename).is_file()]
            if missing:
                raise FileNotFoundError(f"Missing offline model files for {name}: {', '.join(missing)}. Run download_weights.sh before predict.ipynb.")
    
        def _build(self, detector_name: str, side_limit: int):
            from paddleocr import PaddleOCR
            model = PaddleOCR(text_detection_model_name=detector_name, text_detection_model_dir=str(self._model_dir(detector_name)), text_recognition_model_name='korean_PP-OCRv5_mobile_rec', text_recognition_model_dir=str(self._model_dir('korean_PP-OCRv5_mobile_rec')), use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False, device='cpu', enable_mkldnn=True, cpu_threads=self.config.cpu_threads, text_recognition_batch_size=self.config.recognition_batch_size, text_det_limit_type='max', text_det_limit_side_len=side_limit, text_det_thresh=0.25, text_det_box_thresh=0.4, text_det_unclip_ratio=1.8, text_rec_score_thresh=0.15)
            pipeline = inner_ocr_pipeline(model)
            configure_recognizer_width(pipeline.text_rec_model, self.config.recognition_minimum_width)
            install_padding_guard = _ITDA_NS['recognizer_padding'].install_padding_guard
            install_padding_guard(pipeline.text_rec_model)
            pipeline.text_det_model = self.profile.wrap(pipeline.text_det_model, detector_name + ':detector')
            pipeline.text_rec_model = self.profile.wrap(pipeline.text_rec_model, detector_name + ':recognizer')
            return model
    
        def _recovery_model(self):
            if self._recovery is None:
                name = self.config.recovery_detector_name
                self._validate_model(name)
                if os.environ.get('ITDA_SHARE_RECOGNIZER', '1') == '1':
                    from paddlex import create_model
                    SharedDetectorView = _ITDA_NS['shared_detector'].SharedDetectorView
                    wrapper = create_model(name, model_dir=str(self._model_dir(name)), device='cpu', cpu_threads=self.config.cpu_threads, enable_mkldnn=True)
                    detector = self.profile.wrap(wrapper._predictor, name + ':detector')
                    self._recovery = SharedDetectorView(self._mobile, detector, self.config.recovery_side_limit)
                else:
                    self._recovery = self._build(name, self.config.recovery_side_limit)
            return self._recovery
    
        def recognize(self, image: np.ndarray, *, detector: str, variant: str) -> list[OCRLine]:
            self.last_unrecognized_regions = []
            model = self._mobile if detector == 'mobile' else self._recovery_model()
            results = list(model.predict(image))
            if not results:
                return []
            payload = results[0].json
            data = payload.get('res', payload)
            texts = data.get('rec_texts', [])
            scores = data.get('rec_scores', [])
            polygons = data.get('rec_polys', [])
            boxes = data.get('rec_boxes', [])
            lines: list[OCRLine] = []
            recognized_polygons = set()
            empty_regions = []
            for (index, (text, score)) in enumerate(zip(texts, scores)):
                polygon = ()
                geometry_valid = True
                geometry_source = 'polygon'
                if index < len(polygons):
                    points = np.asarray(polygons[index], dtype=float)
                    polygon = tuple((tuple(point) for point in points.tolist()))
                    box = (float(points[:, 0].min()), float(points[:, 1].min()), float(points[:, 0].max()), float(points[:, 1].max()))
                elif index < len(boxes):
                    values = np.asarray(boxes[index], dtype=float).tolist()
                    box = tuple(values[:4])
                    geometry_source = 'box'
                else:
                    box = (0.0, float(index), 1.0, float(index + 1))
                    geometry_valid = False
                    geometry_source = 'synthetic_index'
                line = OCRLine(text=str(text), score=float(score), box=box, source=f'paddle-{detector}', variant=variant, polygon=polygon, geometry_valid=geometry_valid, geometry_source=geometry_source)
                if str(text).strip():
                    lines.append(line)
                    if polygon:
                        recognized_polygons.add(polygon)
                else:
                    empty_regions.append(line)
            unmatched = {}
            for points in data.get('dt_polys', []):
                polygon = tuple((tuple((float(v) for v in point)) for point in points))
                if polygon and polygon not in recognized_polygons:
                    (xs, ys) = zip(*polygon)
                    unmatched[polygon] = OCRLine('', 0.0, (min(xs), min(ys), max(xs), max(ys)), f'paddle-{detector}', variant, polygon=polygon, geometry_source='detector_polygon')
            for line in empty_regions:
                if line.polygon not in unmatched:
                    self.last_unrecognized_regions.append(line)
            self.last_unrecognized_regions.extend(unmatched.values())
            return lines
    
        def recognize_crops(self, crops):
            """Use the already-loaded recognizer without rerunning detection."""
            pipeline = inner_ocr_pipeline(self._mobile)
            model = pipeline.text_rec_model
            resize = getattr(model, 'pre_tfs', {}).get('ReisizeNorm')
            if resize is None:
                return self._recognize_cached(model, crops)
            original_shape = resize.rec_image_shape
            try:
                resize.rec_image_shape = [3, 48, 320]
                return self._recognize_cached(model, crops)
            finally:
                resize.rec_image_shape = original_shape
    
        def recognize_date_crops(self, crops):
            """Lazy offline English/numeric recovery; no additional detector."""
            if self._date_recognizer is None:
                from paddlex import create_model
                name = 'en_PP-OCRv5_mobile_rec'
                self._validate_model(name)
                wrapper = create_model(name, model_dir=str(self._model_dir(name)), device='cpu', cpu_threads=self.config.cpu_threads, enable_mkldnn=True)
                self._date_recognizer = self.profile.wrap(wrapper._predictor, 'numeric:recognizer')
            return self._recognize_cached(self._date_recognizer, crops)
    
        def begin_image(self):
            self._crop_cache = {}
            self.last_ctc_candidates = []
    
        def _recognize_cached(self, model, crops):
            cache = getattr(self, '_crop_cache', None)
            if cache is None:
                return self._recognize_with_evidence(model, crops)
            resize = getattr(model, 'pre_tfs', {}).get('ReisizeNorm')
            preprocess = (tuple(getattr(resize, 'rec_image_shape', ())), tuple(getattr(resize, 'input_shape', ()) or ()), self.config.collect_ctc_candidates)
            keys = [(id(model), preprocess, crop.shape, str(crop.dtype), hashlib.sha256(crop.tobytes()).digest()) for crop in crops]
            missing = list(dict.fromkeys((key for key in keys if key not in cache)))
            results = self._recognize_with_evidence(model, [crops[keys.index(key)] for key in missing]) if missing else []
            if len(results) != len(missing):
                raise ValueError('Cached recognition result count mismatch')
            found = dict(zip(missing, results))
            answer = [cache[key] if key in cache else found[key] for key in keys]
            if len(cache) + len(found) > 128:
                cache.clear()
            cache.update(list(found.items())[:128])
            return answer
    
        def _recognize_with_evidence(self, model, crops):
            width_batches = _ITDA_NS['crop_batching'].width_batches
            if not crops:
                return []
            batches = width_batches(crops, self.config.recognition_batch_size) if self.config.enable_width_batches else [list(range(len(crops)))]
            answer = [None] * len(crops)
            for indices in batches:
                results = self._recognize_batch_with_evidence(model, [crops[i] for i in indices])
                if len(results) != len(indices):
                    raise ValueError('Width batch changed crop result count')
                for (index, result) in zip(indices, results):
                    answer[index] = result
            return answer
    
        def _recognize_batch_with_evidence(self, model, crops):
            original = model.post_op
            evidence = CTCEvidence(original, collect_candidates=self.config.collect_ctc_candidates)
            try:
                model.post_op = evidence
                results = list(model(crops))
            finally:
                model.post_op = original
            aligned = len(evidence.rows) == len(results) and all((text == result['rec_text'] for ((text, _), result) in zip(evidence.rows, results)))
            if aligned and self.config.collect_ctc_candidates:
                diagnostics = getattr(self, 'last_ctc_candidates', [])
                for (crop, result, alternatives) in zip(crops, results, evidence.candidate_rows):
                    if alternatives:
                        diagnostics.append(dict(crop_sha256=hashlib.sha256(crop.tobytes()).hexdigest(), crop_shape=crop.shape, greedy=result['rec_text'], candidates=alternatives, admissible_selector_input=False))
                self.last_ctc_candidates = diagnostics[-32:]
            return [(str(result['rec_text']), float(result['rec_score']), evidence.rows[i][1] if aligned else ()) for (i, result) in enumerate(results)]
    
    def discover_images(input_dir: str | os.PathLike[str]) -> list[Path]:
        root = Path(input_dir)
        if not root.is_dir():
            raise FileNotFoundError(f'ITDA_INPUT_DIR is not a directory: {root}')
        images = sorted((path for path in root.iterdir() if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS), key=lambda path: path.name.casefold())
        if not images:
            raise ValueError(f'No supported images found in ITDA_INPUT_DIR: {root}')
        stems = [path.stem for path in images]
        duplicates = sorted((stem for (stem, count) in Counter(stems).items() if count > 1))
        if duplicates:
            raise ValueError(f'Duplicate image_id values after removing extensions: {duplicates[:5]}')
        return images
    
    def _load_bgr(path: Path) -> np.ndarray:
        with Image.open(path) as source:
            rgb = np.asarray(ImageOps.exif_transpose(source).convert('RGB'))
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    
    def _clahe(image: np.ndarray) -> np.ndarray:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        enhanced = cv2.createCLAHE(clipLimit=2.4, tileGridSize=(8, 8)).apply(gray)
        return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    
    def _date_fragment_lines(lines: Sequence[OCRLine]) -> list[OCRLine]:
        """Choose at most two recovery anchors, not two most legible numbers.
    
        Damaged date-like text is only a crop hint: no glyphs/digits are repaired
        here. Keep legacy numeric anchors as a last resort, including for tiles.
        """
        selected = []
        for line in lines:
            text = line.text.strip()
            digits = sum((character.isdigit() for character in text))
            separated = any((separator in text for separator in ('.', '/', '-', ':')))
            legacy = digits >= 4 and separated
            non_date = re.search('%|kcal|\\d\\s*(?:mg|ml|g|brix)\\b|영양|내용량|지방|당류|고객|상담|전화|품목|보고번호|\\b(?:tel|fax|lot)\\b|(?<!\\d)0\\d{1,2}[- )]\\d{2,4}-\\d{3,4}|(?:로|길)\\s*\\d+.*\\d+-\\d+', text, re.IGNORECASE)
            fragment = separated and len(text) <= 32 and (digits >= 4 or (digits >= 3 and sum((text.count(s) for s in './-')) >= 2)) and (digits / max(1, len(text)) >= 0.25) and (not non_date) and (not re.fullmatch('\\d{1,2}:\\d{2}(?::\\d{2})?', text))
            if legacy or fragment:
                priority = 2 if fragment and parse_dates(text) else 1 if fragment else 0
                selected.append((priority, line))
        return [line for (_, line) in sorted(selected, key=lambda item: (item[0], item[1].score), reverse=True)[:2]]
    
    def _fragment_bounds(image: np.ndarray, line: OCRLine) -> tuple[int, int, int, int]:
        (height, width) = image.shape[:2]
        (left, top, right, bottom) = line.box
        box_width = max(1.0, right - left)
        box_height = max(1.0, bottom - top)
        x1 = max(0, int(left - max(64.0, box_width * 0.8)))
        x2 = min(width, int(right + max(64.0, box_width * 0.8)))
        y1 = max(0, int(top - max(64.0, box_height * 2.5)))
        y2 = min(height, int(bottom + max(64.0, box_height * 2.5)))
        return (x1, y1, x2, y2)
    
    def _crop_fragment(image: np.ndarray, line: OCRLine) -> np.ndarray | None:
        (x1, y1, x2, y2) = _fragment_bounds(image, line)
        if x2 - x1 < 24 or y2 - y1 < 16:
            return None
        crop = image[y1:y2, x1:x2]
        longest = max(crop.shape[:2])
        if longest < 900:
            scale = min(3.0, 900.0 / longest)
            crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        return crop
    
    def _label_crop_bounds(image, lines):
        """At most one bounded search beside a clear, standalone expiry label."""
        (height, width) = image.shape[:2]
        labels = [line for line in lines if line.geometry_valid and line.score >= 0.9 and (line.width >= line.height) and re.fullmatch('까지|소비\\s*기한|유통\\s*기한|EXP|BBD', line.text, re.I)]
        if not labels:
            return None
        if len(labels) != 1:
            return None
        label = labels[0]
        h = label.height
        if re.fullmatch('소비\\s*기한|유통\\s*기한|EXP|BBD', label.text, re.I):
            return (max(0, int(label.box[0] - 6 * h)), max(0, int(label.box[1] - h)), min(width, int(label.box[2] + 12 * h)), min(height, int(label.box[3] + 8 * h)))
        return (max(0, int(label.box[0] - 12 * h)), max(0, int(label.box[1] - 2 * h)), min(width, int(label.box[2] + h)), min(height, int(label.box[3] + h)))
    
    def _tile_bounds(image: np.ndarray, fraction: float) -> list[tuple[int, int, int, int]]:
        if not 0.5 < fraction < 1.0:
            raise ValueError('tile_fraction must be between 0.5 and 1.0')
        (height, width) = image.shape[:2]
        tile_height = max(1, int(height * fraction))
        tile_width = max(1, int(width * fraction))
        origins = ((0, 0), (0, width - tile_width), (height - tile_height, 0), (height - tile_height, width - tile_width))
        return [(left, top, left + tile_width, top + tile_height) for (top, left) in origins]
    
    def _overlapping_tiles(image: np.ndarray, fraction: float) -> list[np.ndarray]:
        return [image[top:bottom, left:right] for (left, top, right, bottom) in _tile_bounds(image, fraction)]
    
    def _append_pass(all_lines: list[OCRLine], passes: list[str], backend: Any, image: np.ndarray, *, detector: str, variant: str, product_rules: Sequence[ProductDateRule]=(), trace: ImageTrace | None=None, frame: ImageFrame | None=None, context: DateContext | None=None) -> DateSelection:
        if trace is not None:
            trace.start_pass(frame, detector, variant)
        started = time.perf_counter()
        try:
            lines = backend.recognize(image, detector=detector, variant=variant)
        except Exception as exc:
            if trace is not None:
                trace.record_pass([], [], frame, detector, variant, time.perf_counter() - started, error=f'{type(exc).__name__}: {exc}')
            raise
        ocr_seconds = time.perf_counter() - started
        if frame is not None:
            mapped_lines = []
            for line in lines:
                if line.geometry_valid:
                    (left, top, right, bottom) = line.box
                    points = frame.map_polygon(((left, top), (right, top), (right, bottom), (left, bottom)))
                    (xs, ys) = zip(*points)
                    line = replace(line, original_box=(min(xs), min(ys), max(xs), max(ys)))
                mapped_lines.append(line)
            lines = mapped_lines
        all_lines.extend(lines)
        passes.append(f'{detector}:{variant}')
        if trace is not None:
            trace.record_selector_input(all_lines, stage=f'{detector}:{variant}')
        try:
            selection = select_date(all_lines, product_rules=product_rules, context=context)
        except Exception as exc:
            if trace is not None:
                trace.record_pass(lines, getattr(backend, 'last_unrecognized_regions', ()), frame, detector, variant, ocr_seconds, error=f'selection: {type(exc).__name__}: {exc}')
            raise
        if trace is not None:
            trace.record_pass(lines, getattr(backend, 'last_unrecognized_regions', ()), frame, detector, variant, ocr_seconds, selection)
        return selection
    
    def predict_image(path: Path, backend: Any, config: PipelineConfig, *, trace: ImageTrace | None=None) -> ImagePrediction:
        started = time.perf_counter()
        if hasattr(backend, 'profile'):
            backend.profile.image_id = path.stem
        if hasattr(backend, 'begin_image'):
            backend.begin_image()
        image = _load_bgr(path)
        (height, width) = image.shape[:2]
        if trace is None and config.collect_trace:
            trace = ImageTrace(path.stem)
        if trace is not None:
            trace.start(width, height)
        original_frame = ImageFrame(width, height)
        all_lines: list[OCRLine] = []
        passes: list[str] = []
        original_lines: list[OCRLine] = []
        geometric_result = None
    
        def finish(selection):
            if config.enable_geometric_recovery and selection.final_date is None and (not selection.stop_ocr or selection.reason.startswith('review_order:')) and hasattr(backend, 'recognize_crops'):
                recovery_started = time.perf_counter()
                if trace is not None:
                    trace.start_pass(original_frame, 'recognition-only', 'geometric-rows')
                try:
                    (added, observations, decisions, seconds) = geometric_result if geometric_result is not None else recover_geometric_rows(image, original_lines, backend.recognize_crops, getattr(backend, 'recognize_date_crops', None))
                    if geometric_result is not None:
                        (observations, decisions, seconds) = ([], [], 0.0)
                    candidate_lines = [*all_lines, *added]
                    if trace is not None:
                        trace.record_selector_input(candidate_lines, stage='geometric-final', final=True)
                    candidate = select_date(candidate_lines, final=True, product_rules=config.product_date_rules, context=config.date_context)
                    selection = candidate
                    all_lines[:] = candidate_lines
                    if observations:
                        passes.append('recognition-only:geometric-rows')
                    if trace is not None:
                        trace.record_recovery(decisions)
                        trace.record_pass([*observations, *added], [], original_frame, 'recognition-only', 'geometric-rows', seconds, selection)
                except Exception as exc:
                    if trace is not None:
                        trace.record_pass([], [], original_frame, 'recognition-only', 'geometric-rows', time.perf_counter() - recovery_started, error=f'{type(exc).__name__}: {exc}')
            if config.enable_line_recovery and selection.final_date is None and (not selection.stop_ocr or selection.reason.startswith('review_order:')) and hasattr(backend, 'recognize_crops'):
                recovery_started = time.perf_counter()
                if trace is not None:
                    trace.start_pass(original_frame, 'recognition-only', 'stroke-rows')
                try:
                    (added, observations, decisions, seconds) = recover_printed_context(image, all_lines, backend.recognize_crops)
                    all_lines.extend(added)
                    if trace is not None:
                        trace.record_selector_input(all_lines, stage='stroke-final', final=True)
                    selection = select_date(all_lines, final=True, product_rules=config.product_date_rules, context=config.date_context)
                    if observations:
                        passes.append('recognition-only:stroke-rows')
                    if trace is not None:
                        trace.record_recovery(decisions)
                        trace.record_pass([*observations, *added], [], original_frame, 'recognition-only', 'stroke-rows', seconds, selection)
                except Exception as exc:
                    if trace is not None:
                        trace.record_pass([], [], original_frame, 'recognition-only', 'stroke-rows', time.perf_counter() - recovery_started, error=f'{type(exc).__name__}: {exc}')
            for natural in (False, True):
                if not (config.enable_geometric_recovery and selection.final_date is None and (not selection.stop_ocr or selection.reason.startswith('review_order:')) and hasattr(backend, 'recognize_crops')):
                    continue
                if natural and (not selection.reason.startswith('review_order:')):
                    continue
                stage = 'natural-dot-rows' if natural else 'thin-dot-rows'
                recovery_started = time.perf_counter()
                if trace is not None:
                    trace.start_pass(original_frame, 'recognition-only', stage)
                try:
                    (added, observations, decisions, seconds) = recover_thin_dots(image, all_lines, backend.recognize_crops, getattr(backend, 'recognize_date_crops', None), natural=natural)
                    all_lines.extend(added)
                    if trace is not None:
                        trace.record_selector_input(all_lines, stage='natural-dot-final' if natural else 'thin-dot-final', final=True)
                    selection = select_date(all_lines, final=True, product_rules=config.product_date_rules, context=config.date_context)
                    if observations:
                        passes.append('recognition-only:' + stage)
                    if trace is not None:
                        trace.record_recovery(decisions)
                        trace.record_pass([*observations, *added], [], original_frame, 'recognition-only', stage, seconds, selection)
                except Exception as exc:
                    if trace is not None:
                        trace.record_pass([], [], original_frame, 'recognition-only', stage, time.perf_counter() - recovery_started, error=f'{type(exc).__name__}: {exc}')
            if trace is not None:
                trace.finish(selection)
            details = dict(schema_version=1, original_size=trace.original_size, coordinate_space='exif_oriented_image_edges', frames=trace.frames, observations=trace.observations, regions=trace.regions, outcomes=trace.outcomes, recovery_decisions=trace.recovery_decisions, summary=trace.summary()) if trace is not None else None
            return ImagePrediction(path.stem, selection.final_date, selection, time.perf_counter() - started, tuple(passes), trace=details)
        selection = _append_pass(all_lines, passes, backend, image, detector='mobile', variant='original', product_rules=config.product_date_rules, trace=trace, frame=original_frame, context=config.date_context)
        original_lines = list(all_lines)
        if config.enable_line_recovery and hasattr(backend, 'recognize_crops') and recovery_targets(all_lines):
            recovery_started = time.perf_counter()
            if trace is not None:
                trace.start_pass(original_frame, 'recognition-only', 'date-lines')
            try:
                (active, observations, decisions, seconds) = recover_lines(image, all_lines, backend.recognize_crops, getattr(backend, 'recognize_date_crops', None))
                if trace is not None:
                    trace.record_selector_input(active, stage='recognition-only:date-lines')
                recovered_selection = select_date(active, product_rules=config.product_date_rules, context=config.date_context)
                all_lines[:] = active
                selection = recovered_selection
                if observations:
                    passes.append('recognition-only:date-lines')
                if trace is not None:
                    trace.record_recovery(decisions)
                    trace.record_pass(observations, [], original_frame, 'recognition-only', 'date-lines', seconds, selection)
            except Exception as exc:
                if trace is not None:
                    trace.record_pass([], [], original_frame, 'recognition-only', 'date-lines', time.perf_counter() - recovery_started, error=f'{type(exc).__name__}: {exc}')
        if config.enable_geometric_recovery and selection.final_date is None and (not selection.candidates) and (not selection.stop_ocr) and hasattr(backend, 'recognize_crops'):
            recovery_started = time.perf_counter()
            if trace is not None:
                trace.start_pass(original_frame, 'recognition-only', 'geometric-early')
            try:
                geometric_result = recover_geometric_rows(image, original_lines, backend.recognize_crops, getattr(backend, 'recognize_date_crops', None))
                (added, observations, decisions, seconds) = geometric_result
                candidate_lines = [*all_lines, *added]
                if trace is not None:
                    trace.record_selector_input(candidate_lines, stage='geometric-early')
                candidate = select_date(candidate_lines, product_rules=config.product_date_rules, context=config.date_context)
                if observations:
                    passes.append('recognition-only:geometric-early')
                if trace is not None:
                    trace.record_recovery(decisions)
                    trace.record_pass([*observations, *added], [], original_frame, 'recognition-only', 'geometric-early', seconds, candidate)
                if candidate.stop_ocr and candidate.final_date is not None and (not candidate.is_partial):
                    return finish(candidate)
            except Exception as exc:
                if trace is not None:
                    trace.record_pass([], [], original_frame, 'recognition-only', 'geometric-early', time.perf_counter() - recovery_started, error=f'{type(exc).__name__}: {exc}')
        original_line_count = len(all_lines)
        original_fragments = _date_fragment_lines(all_lines)
        if selection.stop_ocr:
            return finish(selection)
        if selection.digits_confident and selection.reason.startswith('review_order:') and (_label_crop_bounds(image, all_lines) is None):
            if config.enable_recovery_fallback:
                _append_pass(all_lines, passes, backend, image, detector='recovery', variant='original', product_rules=config.product_date_rules, trace=trace, frame=original_frame, context=config.date_context)
            if trace is not None:
                trace.record_selector_input(all_lines, stage='order-final', final=True)
            return finish(select_date(all_lines, final=True, product_rules=config.product_date_rules, context=config.date_context))
        if selection.final_date is None and (not any((c.explicit_positive for c in selection.candidates))):
            label_lines = list(all_lines)
            bounds = _label_crop_bounds(image, all_lines)
            if bounds is not None:
                (left, top, right, bottom) = bounds
                crop = cv2.resize(image[top:bottom, left:right], None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
                selection = _append_pass(all_lines, passes, backend, crop, detector='mobile', variant='label-roi', product_rules=config.product_date_rules, trace=trace, frame=ImageFrame.crop(bounds, crop.shape), context=config.date_context)
                if selection.stop_ocr:
                    return finish(selection)
                if config.enable_line_recovery and hasattr(backend, 'recognize_crops') and (selection.final_date is None) and (not selection.candidates):
                    recovery_started = time.perf_counter()
                    if trace is not None:
                        trace.start_pass(original_frame, 'recognition-only', 'label-rows')
                    try:
                        (added, observations, decisions, seconds) = recover_missing_rows(image, label_lines, backend.recognize_crops, getattr(backend, 'recognize_date_crops', None))
                        all_lines.extend(added)
                        if trace is not None:
                            trace.record_selector_input(all_lines, stage='recognition-only:label-rows')
                        selection = select_date(all_lines, product_rules=config.product_date_rules, context=config.date_context)
                        if observations:
                            passes.append('recognition-only:label-rows')
                        if trace is not None:
                            trace.record_recovery(decisions)
                            trace.record_pass([*observations, *added], [], original_frame, 'recognition-only', 'label-rows', seconds, selection)
                    except Exception as exc:
                        if trace is not None:
                            trace.record_pass([], [], original_frame, 'recognition-only', 'label-rows', time.perf_counter() - recovery_started, error=f'{type(exc).__name__}: {exc}')
                    if selection.stop_ocr:
                        return finish(selection)
        for (index, fragment) in enumerate(original_fragments, start=1):
            crop = _crop_fragment(image, fragment)
            if crop is None:
                continue
            selection = _append_pass(all_lines, passes, backend, crop, detector='mobile', variant=f'roi-{index}', product_rules=config.product_date_rules, trace=trace, frame=ImageFrame.crop(_fragment_bounds(image, fragment), crop.shape), context=config.date_context)
            if selection.stop_ocr:
                return finish(selection)
        if config.enable_clahe:
            selection = _append_pass(all_lines, passes, backend, _clahe(image), detector='mobile', variant='clahe', product_rules=config.product_date_rules, trace=trace, frame=original_frame, context=config.date_context)
            if selection.stop_ocr:
                return finish(selection)
        if config.enable_recovery_fallback:
            selection = _append_pass(all_lines, passes, backend, image, detector='recovery', variant='original', product_rules=config.product_date_rules, trace=trace, frame=original_frame, context=config.date_context)
            if selection.stop_ocr:
                return finish(selection)
        if config.enable_rotation_fallback and (selection.final_date is None or selection.is_partial or (selection.candidates and selection.candidates[0].repaired)):
            rotations = ((180, cv2.ROTATE_180), (90, cv2.ROTATE_90_CLOCKWISE), (270, cv2.ROTATE_90_COUNTERCLOCKWISE))
            for (angle, rotation) in rotations:
                rotated = cv2.rotate(image, rotation)
                selection = _append_pass(all_lines, passes, backend, rotated, detector='mobile', variant=f'rot{angle}', product_rules=config.product_date_rules, trace=trace, frame=ImageFrame.rotated(width, height, angle), context=config.date_context)
                if selection.stop_ocr:
                    return finish(selection)
        if config.enable_tile_fallback and (selection.final_date is None or selection.is_partial) and (original_line_count <= config.tile_max_original_lines) and (original_line_count <= config.tile_sparse_line_limit or bool(original_fragments)):
            for (index, tile) in enumerate(_overlapping_tiles(image, config.tile_fraction), start=1):
                selection = _append_pass(all_lines, passes, backend, tile, detector='mobile', variant=f'tile-{index}', product_rules=config.product_date_rules, trace=trace, frame=ImageFrame.crop(_tile_bounds(image, config.tile_fraction)[index - 1], tile.shape), context=config.date_context)
                if selection.stop_ocr:
                    return finish(selection)
        if trace is not None:
            trace.record_selector_input(all_lines, stage='final', final=True)
        selection = select_date(all_lines, final=True, product_rules=config.product_date_rules, context=config.date_context)
        return finish(selection)
    
    def _write_submission(path: Path, rows: Sequence[dict[str, str]]) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open('w', encoding='utf-8', newline='') as output:
            writer = csv.DictWriter(output, fieldnames=OUTPUT_COLUMNS, lineterminator='\n')
            writer.writeheader()
            serialize_fields = _ITDA_NS['date_fields'].serialize_fields
            writer.writerows(({'image_id': row['image_id'], **serialize_fields(row)} for row in rows))
    
    def _percentile(values: Sequence[float], fraction: float) -> float:
        ordered = sorted(values)
        index = min(len(ordered) - 1, max(0, math.ceil(fraction * len(ordered)) - 1))
        return ordered[index]
    
    def run_pipeline(input_dir: str | os.PathLike[str], output_path: str | os.PathLike[str], *, config: PipelineConfig | None=None, backend: Any | None=None, max_images: int | None=None, on_image: Callable[[dict[str, Any]], None] | None=None, notebook_started: float | None=None) -> dict[str, Any]:
        if os.environ.get('ITDA_EXECUTION_POLICY', 'base-first-v2') in {'base-first-v1', 'base-first-v2'}:
            run_budget_pipeline = _ITDA_NS['budget_pipeline'].run_budget_pipeline
            return run_budget_pipeline(input_dir, output_path, config=config, backend=backend, max_images=max_images, on_image=on_image, budget_seconds=float(os.environ.get('ITDA_BUDGET_SECONDS', '1470')), notebook_started=notebook_started)
        total_started = time.perf_counter()
        config = config or PipelineConfig()
        images = discover_images(input_dir)
        if max_images is not None:
            if max_images <= 0:
                raise ValueError('max_images must be positive')
            images = images[:max_images]
        trace_path = Path(str(output_path) + '.trace.jsonl') if config.collect_trace else None
        if trace_path is not None:
            trace_path.parent.mkdir(parents=True, exist_ok=True)
            with trace_path.open('w', encoding='utf-8'):
                pass
    
        def emit_trace(event):
            with trace_path.open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(event, ensure_ascii=False) + '\n')
        init_started = time.perf_counter()
        if backend is None:
            backend = PaddleOCRBackend(config)
        init_seconds = time.perf_counter() - init_started
        started = time.perf_counter()
        rows: list[dict[str, str]] = []
        timings: list[float] = []
        pass_counts: Counter[str] = Counter()
        reason_counts: Counter[str] = Counter()
        failures: list[dict[str, str]] = []
        none_count = 0
        trace_totals: Counter[str] = Counter()
        for (index, image_path) in enumerate(images, start=1):
            image_started = time.perf_counter()
            trace = ImageTrace(image_path.stem, emit_trace) if config.collect_trace else None
            try:
                prediction = predict_image(image_path, backend, config, trace=trace) if trace is not None else predict_image(image_path, backend, config)
                fields = submission_fields(prediction.final_date)
            except Exception as exc:
                selection = DateSelection(None, float('-inf'), float('inf'), False, 'exception', ())
                prediction = ImagePrediction(image_path.stem, None, selection, time.perf_counter() - image_started, (), f'{type(exc).__name__}: {exc}')
                failures.append({'image_id': image_path.stem, 'error': prediction.error})
                fields = submission_fields(None)
                if trace is not None:
                    trace.finish(selection, error=prediction.error)
            rows.append({'image_id': image_path.stem, **fields})
            timings.append(prediction.elapsed_seconds)
            pass_counts.update(prediction.passes)
            reason_counts.update([prediction.selection.reason])
            none_count += fields['final_date'] == 'NONE'
            trace_summary = trace.summary() if trace is not None else {}
            trace_totals.update(trace_summary)
            if on_image is not None:
                on_image({'row': rows[-1], 'seconds': prediction.elapsed_seconds, 'error': prediction.error, 'passes': list(prediction.passes), 'trace_summary': trace_summary})
            if config.progress_every > 0 and index % config.progress_every == 0 or index == len(images):
                print(f'Processed {index}/{len(images)} images in {time.perf_counter() - started:.1f}s', flush=True)
        output = Path(output_path)
        _write_submission(output, rows)
        elapsed = time.perf_counter() - started
        summary = {'images': len(images), 'elapsed_seconds': round(elapsed, 3), 'total_elapsed_seconds': round(time.perf_counter() - total_started, 3), 'backend_init_seconds': round(init_seconds, 3), 'timing_scope': 'total starts at run_pipeline entry; caller/import startup excluded', 'seconds_per_image': round(elapsed / len(images), 3), 'p50_image_seconds': round(_percentile(timings, 0.5), 3), 'p95_image_seconds': round(_percentile(timings, 0.95), 3), 'predicted_none': none_count, 'failures': failures, 'passes': dict(pass_counts), 'selection_reasons': dict(reason_counts), 'ocr_calls': sum(pass_counts.values()), 'trace_path': str(trace_path.resolve()) if trace_path is not None else None, 'trace_summary': dict(trace_totals), 'ocr_backend_seconds_per_image': trace_totals['ocr_seconds'] / len(images) if trace_path is not None else None, 'output_path': str(output.resolve())}
        print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)
        return summary
    return _ITDA_Namespace(**locals())
_ITDA_NS['pipeline'] = _itda_define_pipeline()
del _itda_define_pipeline


In [ ]:
from __future__ import annotations
# Embedded from src/selective_recovery.py; generated, do not edit separately.
def _itda_define_selective_recovery():
    """Small evidence-directed stages; the caller commits each completed stage."""
    import cv2
    from dataclasses import replace
    _link_roles = _ITDA_NS['date_extraction']._link_roles
    parse_dates = _ITDA_NS['date_extraction'].parse_dates
    submission_decision = _ITDA_NS['date_extraction'].submission_decision
    POSITIVE_CONTEXT = _ITDA_NS['date_extraction'].POSITIVE_CONTEXT
    NEGATIVE_CONTEXT = _ITDA_NS['date_extraction'].NEGATIVE_CONTEXT
    recovery_targets = _ITDA_NS['line_recovery'].recovery_targets
    recover_lines = _ITDA_NS['line_recovery'].recover_lines
    recover_missing_rows = _ITDA_NS['line_recovery'].recover_missing_rows
    recover_geometric_rows = _ITDA_NS['date_region_recovery'].recover_geometric_rows
    recover_printed_context = _ITDA_NS['printed_context_recovery'].recover_printed_context
    recover_thin_dots = _ITDA_NS['thin_dot_recovery'].recover_thin_dots
    ImageFrame = _ITDA_NS['ocr_trace'].ImageFrame
    
    def needs_recovery(selection, lines):
        if selection.reason == 'explicit-partial-date':
            return False
        decision = submission_decision(selection, base_partial=selection.is_partial)
        if decision.status == 'NOT_FOUND':
            return False
        if decision.status == 'SELECTED' and (not selection.is_partial):
            if selection.role_resolved and (not selection.stop_ocr):
                return True
            return bool(recovery_targets(lines)) and any((c.iso == decision.output_date and c.ocr_score < 0.98 for c in selection.candidates))
        if selection.final_date is None or selection.is_partial:
            return not selection.stop_ocr or selection.reason.startswith('review_')
        if not selection.stop_ocr:
            return True
        return bool(recovery_targets(lines)) and any((c.iso == selection.final_date and c.ocr_score < 0.92 for c in selection.candidates))
    
    def actions_for(selection, lines):
        decision = submission_decision(selection, base_partial=selection.is_partial)
        if decision.recovery_reason in {'role', 'order'}:
            return ['context-roi', 'secondary']
        if recovery_targets(lines):
            if decision.status != 'SELECTED':
                return ['date-lines', 'fragment-roi', 'geometric', 'full-contrast', 'secondary', 'stroke', 'thin-dot']
            return ['date-lines', 'fragment-roi', 'contrast-roi', 'secondary', 'geometric', 'stroke', 'thin-dot']
        if not selection.candidates and (not any((POSITIVE_CONTEXT.search(line.text) or NEGATIVE_CONTEXT.search(line.text) for line in lines))):
            return ['secondary', 'geometric', 'stroke', 'thin-dot']
        return ['context-roi', 'geometric', 'secondary', 'stroke', 'thin-dot']
    
    def run_stage(action, image, lines, backend, guard, config, trace, commit):
        _append_pass = _ITDA_NS['pipeline']._append_pass
        _clahe = _ITDA_NS['pipeline']._clahe
        _label_crop_bounds = _ITDA_NS['pipeline']._label_crop_bounds
        _date_fragment_lines = _ITDA_NS['pipeline']._date_fragment_lines
        _fragment_bounds = _ITDA_NS['pipeline']._fragment_bounds
        (height, width) = image.shape[:2]
        frame = ImageFrame(width, height)
        active = [replace(line, box=line.original_box, polygon=()) if line.original_box and tuple(line.box) != tuple(line.original_box) else line for line in lines]
        numeric = guard.recognize_date_crops if hasattr(backend, 'recognize_date_crops') else None
        (observations, decisions, seconds) = ([], [], 0.0)
        if action in {'date-lines', 'geometric', 'stroke', 'thin-dot'}:
            if not hasattr(backend, 'recognize_crops'):
                return (active, [], [], 0.0)
            if trace:
                trace.start_pass(frame, 'recognition-only', action)
            if action == 'date-lines':
                return recover_lines(image, active, guard.recognize_crops, numeric, on_progress=commit)
            function = {'geometric': recover_geometric_rows, 'stroke': recover_printed_context, 'thin-dot': recover_thin_dots}[action]
            if action == 'stroke':
                (added, observations, decisions, seconds) = function(image, active, guard.recognize_crops)
            else:
                (added, observations, decisions, seconds) = function(image, active, guard.recognize_crops, numeric)
            return ([*active, *added], [*observations, *added], decisions, seconds)
        bounds = None
        if action == 'context-roi':
            bounds = _label_crop_bounds(image, active)
            if bounds is None:
                return (active, [], [], 0.0)
        elif action in {'fragment-roi', 'contrast-roi'}:
            fragments = _date_fragment_lines(active)
            if not fragments:
                return (active, [], [], 0.0)
            bounds = _fragment_bounds(image, fragments[0])
        detector = 'recovery' if action == 'secondary' else 'mobile'
        if bounds:
            (x1, y1, x2, y2) = bounds
            image = image[y1:y2, x1:x2]
            if action == 'context-roi':
                image = cv2.resize(image, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
            frame = ImageFrame.crop(bounds, image.shape)
        if action in {'contrast-roi', 'full-contrast'}:
            image = _clahe(image)
        guard.check()
        guard.calls += 1
        _append_pass(active, [], backend, image, detector=detector, variant='selective-' + action, product_rules=config.product_date_rules, context=config.date_context, frame=frame, trace=trace)
        return (active, [], [], seconds)
    return _ITDA_Namespace(**locals())
_ITDA_NS['selective_recovery'] = _itda_define_selective_recovery()
del _itda_define_selective_recovery


In [ ]:
from __future__ import annotations
# Embedded from src/budget_pipeline.py; generated, do not edit separately.
def _itda_define_budget_pipeline():
    """Base coverage first, then incremental evidence-directed recovery within a deadline."""
    import json
    import math
    import os
    import time
    from contextlib import nullcontext
    from dataclasses import replace
    from itertools import zip_longest
    from pathlib import Path
    select_date = _ITDA_NS['date_extraction'].select_date
    submission_fields = _ITDA_NS['date_extraction'].submission_fields
    submission_decision = _ITDA_NS['date_extraction'].submission_decision
    ImageFrame = _ITDA_NS['ocr_trace'].ImageFrame
    ImageTrace = _ITDA_NS['ocr_trace'].ImageTrace
    PipelineConfig = _ITDA_NS['pipeline'].PipelineConfig
    PaddleOCRBackend = _ITDA_NS['pipeline'].PaddleOCRBackend
    discover_images = _ITDA_NS['pipeline'].discover_images
    _load_bgr = _ITDA_NS['pipeline']._load_bgr
    _append_pass = _ITDA_NS['pipeline']._append_pass
    _write_submission = _ITDA_NS['pipeline']._write_submission
    _percentile = _ITDA_NS['pipeline']._percentile
    recovery_targets = _ITDA_NS['line_recovery'].recovery_targets
    recover_lines = _ITDA_NS['line_recovery'].recover_lines
    recover_geometric_rows = _ITDA_NS['date_region_recovery'].recover_geometric_rows
    needs_recovery = _ITDA_NS['selective_recovery'].needs_recovery
    actions_for = _ITDA_NS['selective_recovery'].actions_for
    run_stage = _ITDA_NS['selective_recovery'].run_stage
    RecoveryPolicy = _ITDA_NS['recovery_policy'].RecoveryPolicy
    binding = _ITDA_NS['recovery_policy'].binding
    features = _ITDA_NS['recovery_policy'].features
    
    class BudgetExhausted(RuntimeError):
        pass
    
    class RecoveryGuard:
        """Cooperative call boundary only; does not claim to interrupt native inference."""
    
        def __init__(self, backend, deadline, clock, max_calls=None, max_crops=32):
            (self.backend, self.deadline, self.clock) = (backend, deadline, clock)
            (self.calls, self.max_calls, self.max_crops) = (0, max_calls, max_crops)
    
        def check(self):
            if self.clock() >= self.deadline:
                raise BudgetExhausted('deadline_exhausted')
            if self.max_calls is not None and self.calls >= self.max_calls:
                raise BudgetExhausted('call_allowance_exhausted')
    
        def crops(self, name, crops):
            self.check()
            if len(crops) > self.max_crops:
                raise BudgetExhausted('crop_allowance_exhausted')
            self.calls += 1
            return getattr(self.backend, name)(crops)
    
        def recognize_crops(self, crops):
            return self.crops('recognize_crops', crops)
    
        def recognize_date_crops(self, crops):
            return self.crops('recognize_date_crops', crops)
    
    def atomic_csv(path, rows):
        temp = Path(str(path) + '.tmp')
        _write_submission(temp, rows)
        os.replace(temp, path)
    
    def output_selection(selection, *, base_partial=False, previous=None):
        """Use the same evidence decision for CSV and diagnostics, never elapsed time."""
        field_evidence = _ITDA_NS['date_extraction'].field_evidence
        serialize_fields = _ITDA_NS['date_fields'].serialize_fields
        fields = field_evidence(selection)
        retained_fields = []
        if previous is not None and previous.final_date:
            old = submission_fields(previous.final_date)
            names = ('year', 'month', 'day')
            compatible = all((fields[k] == 'NONE' or old[k] == 'NONE' or fields[k] == old[k] for k in names))
            anchored = all((fields[k] == 'NONE' for k in names)) or any((fields[k] != 'NONE' and fields[k] == old[k] for k in names))
            retracted = selection.reason in {'negative-context', 'expiry-not-printed'} or any((c.explicit_negative and (not c.explicit_positive) and (c.ocr_score >= 0.65) and all((old[k] == 'NONE' or old[k] == submission_fields(c.iso)[k] for k in names)) for c in selection.candidates))
            if compatible and anchored and (not retracted):
                combined = {k: fields[k] if fields[k] != 'NONE' else old[k] for k in names}
                try:
                    combined = serialize_fields(combined)
                except ValueError:
                    pass
                else:
                    retained_fields = [k for k in names if fields[k] == 'NONE' and combined[k] != 'NONE']
                    fields = combined
        value = fields['final_date'] if fields['final_date'] != 'NONE' else None
        decision = submission_decision(selection, base_partial=base_partial)
        details = dict(selection.policy_details or {})
        details['retained_fields'] = retained_fields
        details.update(candidate_date=decision.candidate_date, expiration_date=value, output_fields=fields, status=decision.status, recovery_reason=decision.recovery_reason, confidence_level=details.get('confidence_level', 'REVIEW') if decision.status == 'SELECTED' else 'REVIEW')
        details['status'] = 'SELECTED' if value and 'NONE' not in value else 'PARTIAL' if value else decision.status
        return replace(selection, final_date=value, output_fields=fields, policy_details=details)
    
    def recovery_queue(pending):
        """Give each evidence class a turn before taking another from one class."""
        groups = [sorted((item for item in pending if item['priority'] == priority), key=lambda item: item['index']) for priority in sorted({item['priority'] for item in pending})]
        return [item for group in zip_longest(*groups) for item in group if item is not None]
    
    def run_budget_pipeline(input_dir, output_path, *, config=None, backend=None, max_images=None, on_image=None, budget_seconds=1470.0, reserve_seconds=30.0, recovery_image_seconds=12.0, clock=time.perf_counter, notebook_started=None):
        if not math.isfinite(budget_seconds) or not 0 <= reserve_seconds < budget_seconds:
            raise ValueError('A finite positive execution budget with output reserve is required')
        if not math.isfinite(recovery_image_seconds) or recovery_image_seconds <= 0:
            raise ValueError('Recovery allowance must be finite and positive')
        started = clock()
        deadline = (started if notebook_started is None else notebook_started) + budget_seconds
        config = config or PipelineConfig()
        policy = RecoveryPolicy(config.recovery_policy_path, binding(config)) if config.recovery_policy_path else None
        images = discover_images(input_dir)
        if max_images is not None:
            if max_images <= 0:
                raise ValueError('max_images must be positive')
            images = images[:max_images]
        output = Path(output_path)
        partial = Path(str(output) + '.partial.csv')
        journal = Path(str(output) + '.progress.jsonl')
        status_path = Path(str(output) + '.status.json')
        trace_path = Path(str(output) + '.trace.jsonl')
        if any((p.exists() for p in (output, partial, journal, status_path, trace_path, Path(str(output) + '.tmp'), Path(str(partial) + '.tmp')))):
            raise FileExistsError('Use a new development output path; do not overwrite prior results')
        output.parent.mkdir(parents=True, exist_ok=True)
    
        def event(value):
            with journal.open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(value, ensure_ascii=False) + '\n')
        phase = 'base'
    
        def emit_trace(value):
            with trace_path.open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(dict(phase=phase, **value), ensure_ascii=False) + '\n')
    
        def stage(name):
            return backend.profile.measure(name) if hasattr(backend, 'profile') else nullcontext()
        event(dict(kind='run_start', expected_images=len(images), budget_seconds=budget_seconds, recovery_policy_sha256=policy.sha256 if policy else None, policy='base-first-v2', timing_scope='notebook non-CONFIG entry including imports' if notebook_started is not None else 'pipeline entry; notebook/import excluded'))
        init = clock()
        if backend is None:
            backend = PaddleOCRBackend(config)
        init_seconds = clock() - init
        (rows, pending, timings, failures, recovery_errors) = ([], [], [], [], [])
        completed = 0
        for path in images:
            if clock() >= deadline - reserve_seconds:
                event(dict(kind='base_budget_exhausted', next_image=path.stem))
                break
            image_started = clock()
            event(dict(kind='base_start', image_id=path.stem))
            if hasattr(backend, 'profile'):
                backend.profile.image_id = path.stem
            if hasattr(backend, 'begin_image'):
                backend.begin_image()
            try:
                with stage('image_load'):
                    image = _load_bgr(path)
                (height, width) = image.shape[:2]
                trace = ImageTrace(path.stem, emit_trace) if config.collect_trace else None
                if trace:
                    trace.start(width, height)
                (lines, passes) = ([], [])
                with stage('base_full_ocr_and_selection'):
                    _append_pass(lines, passes, backend, image, detector='mobile', variant='original', product_rules=config.product_date_rules, context=config.date_context, frame=ImageFrame(width, height), trace=trace)
                    selection = select_date(lines, final=False, product_rules=config.product_date_rules, context=config.date_context)
                if trace:
                    trace.finish(selection)
                provisional = output_selection(selection, base_partial=selection.is_partial)
                rows.append(dict(image_id=path.stem, **submission_fields(provisional.final_date)))
                completed += 1
                if needs_recovery(selection, lines):
                    priority = 0 if recovery_targets(lines) else 1 if selection.candidates else 2
                    pending.append(dict(priority=priority, index=len(rows) - 1, path=path, lines=lines, selection=selection, retained_output=provisional, base_partial=selection.is_partial, recovery_seconds=0.0, actions=actions_for(selection, lines)))
                event(dict(kind='base_end', image_id=path.stem, row=rows[-1], error=None, reason=selection.reason, seconds=clock() - image_started))
            except Exception as exc:
                error = dict(image_id=path.stem, error=repr(exc))
                failures.append(error)
                rows.append(dict(image_id=path.stem, **submission_fields(None)))
                event(dict(kind='base_end', **error, row=rows[-1], seconds=clock() - image_started))
            timings.append(clock() - image_started)
            with stage('checkpoint_write'):
                atomic_csv(partial, rows)
            if on_image:
                on_image(dict(row=rows[-1], seconds=timings[-1], phase='base', error=failures[-1]['error'] if failures and failures[-1]['image_id'] == path.stem else None))
            if config.progress_every and len(rows) % config.progress_every == 0:
                print(f'Base processed {completed}/{len(images)}; attempted {len(rows)}', flush=True)
        base_complete = completed == len(images) and (not failures)
        base_seconds = clock() - started
        if base_complete:
            atomic_csv(output, rows)
            event(dict(kind='base_complete', processed_images=completed, seconds=base_seconds))
        queue = recovery_queue(pending) if base_complete else []
        costs = {}
        while queue:
            if clock() >= deadline - reserve_seconds:
                break
            item = policy.pop(queue) if policy else queue.pop(0)
            (index, path, lines, selection) = (item[k] for k in ('index', 'path', 'lines', 'selection'))
            action = item['actions'].pop(0)
            state_before = features(selection, lines, item['retained_output'].final_date)
            learned = policy.estimate(state_before, action) if policy else None
            remaining = recovery_image_seconds - item['recovery_seconds']
            if remaining <= 0:
                event(dict(kind='recovery_skipped', image_id=path.stem, action=action, reason='image_allowance_exhausted', remaining_actions=item['actions'], recovery_seconds=item['recovery_seconds']))
                continue
            cost_key = (action, min(len(lines) // 20, 3))
            samples = sorted(costs.get(cost_key, []))
            estimate = samples[min(len(samples) - 1, int(0.8 * len(samples)))] * 1.25 if samples else 0.8 if action in {'date-lines', 'fragment-roi', 'geometric', 'stroke', 'thin-dot'} else 2.5
            if learned:
                estimate = max(estimate, learned['estimated_seconds'])
            if clock() + estimate >= deadline - reserve_seconds:
                event(dict(kind='recovery_skipped', image_id=path.stem, action=action, reason='insufficient_estimated_stage_time', estimate_seconds=estimate))
                if item['actions']:
                    queue.append(item)
                continue
            if hasattr(backend, 'profile'):
                backend.profile.image_id = path.stem
            if hasattr(backend, 'begin_image'):
                backend.begin_image()
            guard = RecoveryGuard(backend, min(deadline - reserve_seconds, clock() + remaining), clock)
            phase = 'recovery'
            event(dict(kind='recovery_start', image_id=path.stem, reason=selection.reason, action=action, state=state_before, row_before=rows[index], expected=learned, calibrated_fields=policy.field_probabilities(state_before) if policy else None))
            action_started = clock()
            trace = None
            decisions = []
    
            def commit(active, observations, decisions, seconds):
                recovered = select_date(active, final=False, product_rules=config.product_date_rules, context=config.date_context)
                permitted = output_selection(recovered, base_partial=item['base_partial'], previous=item.get('retained_output'))
                (item['lines'], item['selection'], item['retained_output']) = (list(active), recovered, permitted)
                previous_row = rows[index]
                rows[index] = dict(image_id=path.stem, **submission_fields(permitted.final_date))
                atomic_csv(output, rows)
                atomic_csv(partial, rows)
                if trace:
                    trace.record_recovery(decisions)
                    trace.record_selector_input(active, stage='incremental-' + action, final=False)
                    trace.record_pass(observations, [], frame, 'recognition-only', action, seconds, recovered)
                event(dict(kind='recovery_checkpoint', image_id=path.stem, action=action, row=rows[index], previous_row=previous_row, output_changed=previous_row != rows[index], field_changes={k: dict(before=previous_row[k], after=rows[index][k], selection_reason=recovered.reason, supporting_raw=[c.raw for c in recovered.candidates if not c.explicit_negative and submission_fields(c.iso)[k] == rows[index][k]]) for k in ('year', 'month', 'day') if previous_row[k] != rows[index][k]}, submission_decision=permitted.policy_details, reason=recovered.reason, native_calls=guard.calls, decisions=decisions))
            try:
                with stage('recovery_image_load'):
                    image = _load_bgr(path)
                guard.check()
                (height, width) = image.shape[:2]
                trace = ImageTrace(path.stem, emit_trace) if config.collect_trace else None
                frame = ImageFrame(width, height)
                if trace:
                    trace.start(width, height)
                with stage('selected_recovery'):
                    (active, observations, decisions, seconds) = run_stage(action, image, lines, backend, guard, config, trace, commit)
                    commit(active, observations, decisions, seconds)
                if trace:
                    trace.finish(item['selection'])
                event(dict(kind='recovery_end', image_id=path.stem, action=action, row=rows[index], reason=item['selection'].reason, native_calls=guard.calls, seconds=clock() - action_started, state_after=features(item['selection'], item['lines'], item['retained_output'].final_date)))
                if getattr(backend, 'last_ctc_candidates', None):
                    event(dict(kind='ctc_diagnostics', image_id=path.stem, action=action, candidates=backend.last_ctc_candidates, used_for_selection=False))
            except BudgetExhausted as exc:
                event(dict(kind='recovery_skipped', image_id=path.stem, reason=str(exc), native_calls=guard.calls))
            except Exception as exc:
                recovery_errors.append(dict(image_id=path.stem, error=repr(exc)))
                event(dict(kind='recovery_error', **recovery_errors[-1]))
            costs.setdefault(cost_key, []).append(clock() - action_started)
            item['recovery_seconds'] += clock() - action_started
            costs[cost_key] = costs[cost_key][-20:]
            if action == 'secondary' and recovery_targets(item['lines']) and needs_recovery(item['selection'], item['lines']):
                item['actions'].insert(0, 'date-lines')
            verified_selection = submission_decision(item['selection'], base_partial=item['base_partial'])
            checked = any((d.get('decision_basis') in {'stable-aligned-date-digits', 'stable-role-recovery'} for d in decisions))
            if action == 'date-lines' and checked and (verified_selection.status == 'SELECTED') and (not item['selection'].is_partial) and item['selection'].digits_confident:
                event(dict(kind='recovery_stopped', image_id=path.stem, reason='selected_after_digit_check', row=rows[index]))
            elif item['actions'] and needs_recovery(item['selection'], item['lines']):
                if item['recovery_seconds'] < 2.0 and item['actions'][0] in {'date-lines', 'fragment-roi', 'geometric'}:
                    queue.insert(0, item)
                else:
                    queue.append(item)
        for item in queue:
            event(dict(kind='recovery_skipped', image_id=item['path'].stem, reason='global_deadline', remaining_actions=item['actions']))
        if base_complete:
            for item in pending:
                final = output_selection(item['selection'], base_partial=item['base_partial'], previous=item.get('retained_output'))
                rows[item['index']] = dict(image_id=item['path'].stem, **submission_fields(final.final_date))
                event(dict(kind='final_selection', image_id=item['path'].stem, row=rows[item['index']], reason=final.reason))
            atomic_csv(output, rows)
            atomic_csv(partial, rows)
        elapsed = clock() - (started if notebook_started is None else notebook_started)
        summary = dict(status='completed' if base_complete and (not recovery_errors) and (elapsed <= budget_seconds) else 'failed' if failures or recovery_errors else 'timeout', images=len(images), processed_images=completed, attempted_images=len(rows), unprocessed_ids=[p.stem for p in images[len(rows):]], output_complete=base_complete, output_path=str(output.resolve()) if base_complete else None, partial_path=str(partial.resolve()), progress_path=str(journal.resolve()), execution_policy='base-first-v2', budget_seconds=budget_seconds, recovery_image_seconds=recovery_image_seconds, total_elapsed_seconds=elapsed, elapsed_seconds=elapsed - init_seconds, backend_init_seconds=init_seconds, base_seconds=base_seconds, failures=[*failures, *recovery_errors], predicted_none=sum((r['final_date'] == 'NONE' for r in rows)), p50_image_seconds=_percentile(timings, 0.5) if timings else None, p95_image_seconds=_percentile(timings, 0.95) if timings else None, timing_scope='notebook non-CONFIG entry including imports' if notebook_started is not None else 'pipeline entry including initialization; caller/import startup excluded', deadline_scope='cooperative native-call boundaries; no native interruption guarantee')
        status_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
        event(dict(kind='run_end', **summary))
        print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)
        return summary
    return _ITDA_Namespace(**locals())
_ITDA_NS['budget_pipeline'] = _itda_define_budget_pipeline()
del _itda_define_budget_pipeline


In [ ]:
summary = _ITDA_NS["pipeline"].run_pipeline(
    INPUT_DIR, OUTPUT_PATH, notebook_started=NOTEBOOK_STARTED,
    config=_ITDA_NS["pipeline"].PipelineConfig(weights_dir=_ITDA_BUNDLE_ROOT / "weights" / "paddle"))
